In [ ]:
# from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
import os, psutil 
def show_ram():
    proc = psutil.Process(os.getpid())
    rss = proc.memory_info().rss
    print(f"Notebook RAM usage: {rss/1e9:.2f} GB")

show_ram()


In [ ]:
import os
import gc
import glob
import numpy as np
import healpy as hp
import matplotlib
import hera_pspec as hp
from scipy import stats
import hera_cal as hc
from astropy import constants
from pyuvdata import utils as uvutils
from pyuvdata import UVData
import matplotlib.pyplot as plt
import itertools
from pyuvdata import UVData, UVCal

import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
from scipy import constants, interpolate
import copy
import glob
import re
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 1000)
from uvtools.plot import plot_antpos, plot_antclass
from hera_qm import ant_metrics, ant_class, xrfi
from hera_cal import io, utils, redcal, apply_cal, datacontainer, abscal
from hera_filters import dspec
from IPython.display import display, HTML
import linsolve
# display(HTML("<style>.container { width:100% !important; }</style>"))
# _ = np.seterr(all='ignore')  # get rid of red warnings
# %config InlineBackend.figure_format = 'retina'

# this enables better memory management on linux
import ctypes
def malloc_trim():
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0) 
    except OSError:
        pass
    
import sys

from copy import deepcopy
from datetime import datetime
import time

matplotlib.rcParams["mathtext.fontset"] = "cm"
matplotlib.rcParams["font.family"] = "STIXGeneral"
matplotlib.rcParams["font.size"] = "18"
from matplotlib.ticker import MultipleLocator


####################################################################################################################################################################################################################################################################################

for repo in ['numpy', 'scipy', 'astropy', 'hera_cal', 'hera_qm', 'hera_filters', 'pyuvdata', 'hera_pspec']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')
    
# Old hera_filter version : ValueError: ridge_alpha is not a valid argument!valid arguments include ['suppression_factors', 'eigenval_cutoff', 'max_contiguous_edge_flags'] : Version hera_filters: 0.1.3
# Latest version working 24 Nov 24 : hera_filters: 0.1.6.dev1+g297dcce
# How to update :
# conda activate <hera>
# pip install --upgrade hera-filters
# python -c "import hera_filters; print(hera_filters.__version__)"

# pip install --upgrade hera-filters --user
# pip install git+https://github.com/HERA-Team/hera_filters.git

# pld hera_pspec: 0.4.2.dev4+gc34b82e

In [ ]:
# All H4C dependencies

# import numpy as np
import matplotlib.pyplot as plt
# To run pspecdata.pspec_run
from hera_pspec import PSpecContainer
from hera_pspec import utils as pspec_utils
from hera_pspec import pspecdata, pstokes
from hera_pspec import uvpspec

# To run io.HERAdata etc
from hera_cal import frf, delay_filter, io, smooth_cal

# import glob, tqdm, os, copy
import glob, os, copy
import functools
import time

from hera_qm import utils
from astropy import units

from matplotlib.colors import LogNorm
from hera_cal import apply_cal
from pyuvdata import UVBeam, UVData
from hera_pspec import grouping
import warnings
# To run datetime.now() function
from datetime import datetime
from hera_pspec.conversions import Cosmo_Conversions as cc
from hera_cal import lstbin
from hera_cal import frf
import scipy.interpolate as interp
from hera_cal.vis_clean import VisClean

from pathlib import Path

SDAY_KSEC = units.sday.to("ks")

In [ ]:
# import numpy as np
# np.set_printoptions(linewidth=370)  # or any large number that suits your screen


In [ ]:
pol_root=['xx']#, 'yy']
proc1 = ["cutbl"]#, "allbl"] "bwcut" "cutbl"
proc2 = ["cutlst"]#, "alllst"]

run_batch = ['250925']

chunk_min, chunk_max = 0, 288
fch_min, fch_max = 271, 276 

# sky_type = "ptsrc"
# sky_type = "eor"

In [ ]:
import h5py

def dat_globber_multi(bl_len, bl_ang, pol_read):
    
    print("pol_read ", pol_read)
    
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1'
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_pI/' 
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_xx/' 
    OUTPUT_DIR = f'output_vis_sigloss_check_out/Sig_Loss_{run_batch[0]}_{pol_read}/' 
    BASE_OUTDIR= Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/outputs")
    print("OUTPUT_DIR ", OUTPUT_DIR)
    
#     pol='pI'
#     pol='xx'
#     pol='yy'
    
#     proc1 = ["cutbl", "allbl"]
    
    # PTSRC SKY

    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # ideal ENU, diameter Airy var (deltaD ~ <.2m)        airyprb, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # real ENU, diameter Airy var (deltaD ~ <.2m)         airyprb, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)      airytilt, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # EOR SKY =============================================

    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # Gaussian###################
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    
    # EOR SKY PLAYGROUND =============================================

    # Gaussian###################
    # MODEL_DIR  = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix_freqclone.03001951._beammapperant_airyred-nonred_airyred")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_spatmean_pwlw/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")

        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")

    # Airy ######################
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_tilt_skysd111_eoroffsetfix.06127f13._beammapperant_airytilt-nonred_airytilt/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")    
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/fftvis_xcheck/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy, CV rlzn challenge
    MODEL_DIR  = Path("eor-grf-256/seed700_freqslic_middle_fch0273ref/fftvis_xcheck/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # HERA Stripe Zenith Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_bright_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # HERA Stripe Zenith 5 Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_5_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # Isotropic ######################  
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    
    # EOR Noisy 

    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-2x/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-correct-beam/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
         # Rlzn 556
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # PTSRC Noisy =============================================

    # PURE Noise =============================================

    # MODEL_DIR   = Path("noise-only-300k/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")

    # sky_type = "ptsrc"    
    sky_type = "eor_ns"
    
    DATA_PATH = BASE_OUTDIR / MODEL_DIR

    # Get just the last path component
    base = MODEL_DIR.name
    # Extract ideal tag: after "subset_" and before the next "_"
    m_ideal = re.search(r"subset_([^_]+)", base)
    ideal_tag = m_ideal.group(1) if m_ideal else "ideal"
    # Extract airy tag: after "nonred_" to the end (no more "_")
    m_airy = re.search(r"nonred_([^_]+)$", base)
    airy_tag = m_airy.group(1) if m_airy else "airy"
    print("ideal_tag:", ideal_tag)  
    print("airy_tag :", airy_tag) 

    batchnum = 0 
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Coh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck{chunk_min:05d}-{chunk_max:05d}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Coh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_fch{fch_min:04d}-{fch_max:04d}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Coh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))

    print(input_datfile_list)
    print("input_datfile_list ", input_datfile_list[0:10])
    
    print("bl_len, bl_ang ", bl_len, bl_ang)
    uvps_dat_list = []
    start = time.time()
    for fname in input_datfile_list: 
#         print("fname ", fname)
        uvpsi = {}
        psci = PSpecContainer(filename=fname, mode='r', keep_open=False)
#         print(psci.groups() )
        uvpsi = psci.get_pspec('dset0', 'dset0_x_dset0' )
#         print("uvpsi ", uvpsi)
#         print( psci.groups )
        uvps_dat_list.append(uvpsi)
    end = time.time()
    print(end - start)
    
#     for fname in input_datfile_list: 
#         with h5py.File(fname, 'r') as f:
#             # List the top-level groups/datasets in the file
#             print("Top-level keys:", list(f.keys()))
#             print("f ", f)
            
#             uvps_dat_list.append(f)

            # Suppose there is a group called 'mygroup', you can access it like this:
#             if 'data_spw0' in f:
#                 mygroup = f['data_spw0']
#                 print(mygroup)
#                 # print("Keys in 'mygroup':", list(mygroup.keys()))


    print("uvps_dat_list ", uvps_dat_list[:] )
    # global uvp 
    print("Combining ___________________________________")
    # Indices to remove
    indices_to_remove = {} 
    new_list = [value for idx, value in enumerate(uvps_dat_list) if idx not in indices_to_remove]    
    uvp  = uvpspec.combine_uvpspec(new_list, merge_history=True, verbose=False)
    print( uvp.get_blpairs() )
    return uvp

#     print("uvps_dat_tot ", uvps_dat_tot)
    
    

In [ ]:
# bl_len = [25, 29, 44] #, 38]
# bl_ang = [90, 0, 0] #, 161]

# bl_len = [ 15.0, 25.0 ]
# bl_ang = [ 0.0, 150.0 ]

# bl_len = [25, 25, 29, 44]
# bl_ang = [90, 150, 0, 120]

# bl_len = [15, 25, 25, 29, 29, 29, 39, 39, 39, 39, 44, 44, 44, 51, 51, 53, 53, 53, 53, 58, 58, 64, 64, 64, 67, 67, 67, 67, 73, 73, 76, 77, 77, 81, 88]
# bl_ang = [0, 150, 30, 0, 60, 120, 19, 161, 139, 41, 0, 60, 120, 30, 150, 166, 14, 46, 134, 0, 120, 144, 157, 24, 169, 11, 109, 131, 0, 120, 150, 161, 139, 171, 0]

# bl_len = [39, 64, 64, 64, 67, 67, 67,  88]
# bl_ang = [41, 144, 157, 24, 169, 109, 131, 0]

bl_len = [14.0]#, 25, 25, 29, 29, 29, 39, 39, 39, 39, 44, 44, 44, 51, 51, 53, 53, 53, 53, 58, 58 ] 
bl_ang = [0.0]#, 150, 30, 0, 60, 120, 19, 161, 139, 41, 0, 60, 120, 30, 150, 166, 14, 46, 134, 0, 120 ] 

# Loop over the N values and store each combined uvp in a dictionary.
combined_uvp_dict = {}
for pol_in in (pol_root):
    print("pol_in ", pol_in)
    for i in range(len(bl_len)):
        print("i ", i, bl_len[i],bl_ang[i], pol_in)
        uvp = dat_globber_multi(bl_len[i],bl_ang[i], pol_in)
        if uvp is not None:
            combined_uvp_dict[(bl_len[i],bl_ang[i],pol_in)] = uvp

In [ ]:
print(combined_uvp_dict)
for pol_in in (pol_root):
    for i in range(len(bl_len)):
        print(combined_uvp_dict[(bl_len[i],bl_ang[i],pol_in)])
    
# print(uvp)
# access each combined uvp by its N value.
# for bl_len, uvp_obj in combined_uvp_dict.items():
#     print(f"N = {bl_len}: combined uvp keys: {list(uvp_obj.keys())}")

In [ ]:
import h5py

def dat_globber_multi(bl_len, bl_ang, pol_read):
    
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1'
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_pI/' 
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_xx/' 
#     OUTPUT_DIR = f'/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_{pol_read}/' 
    OUTPUT_DIR = f'output_vis_sigloss_check_out/Sig_Loss_{run_batch[0]}_{pol_read}/'
    BASE_OUTDIR= Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/outputs") 

    
#     pol='pI'
#     pol='xx'
#     pol='yy'
    
#     proc1 = ["cutbl", "allbl"]

    
    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # ideal ENU, diameter Airy var (deltaD ~ <.2m)        airyprb, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # real ENU, diameter Airy var (deltaD ~ <.2m)         airyprb, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)      airytilt, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # EOR SKY

    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # Gaussian###################
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    
    # EOR SKY PLAYGROUND =============================================

    # Gaussian###################
    # MODEL_DIR  = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix_freqclone.03001951._beammapperant_airyred-nonred_airyred")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_spatmean_pwlw/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")

        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")

    # Airy ######################
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_tilt_skysd111_eoroffsetfix.06127f13._beammapperant_airytilt-nonred_airytilt/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")    
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/fftvis_xcheck/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy, CV rlzn challenge
    MODEL_DIR  = Path("eor-grf-256/seed700_freqslic_middle_fch0273ref/fftvis_xcheck/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # HERA Stripe Zenith Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_bright_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # HERA Stripe Zenith 5 Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_5_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # Isotropic ######################  
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    
    # EOR Noisy 

    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-2x/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-correct-beam/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
         # Rlzn 556
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # PTSRC Noisy =============================================

    # PURE Noise =============================================

    # MODEL_DIR   = Path("noise-only-300k/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    
    # sky_type = "ptsrc"    
    sky_type = "eor_ns"
    
    DATA_PATH = BASE_OUTDIR / MODEL_DIR

    # Get just the last path component
    base = MODEL_DIR.name
    # Extract ideal tag: after "subset_" and before the next "_"
    m_ideal = re.search(r"subset_([^_]+)", base)
    ideal_tag = m_ideal.group(1) if m_ideal else "ideal"
    # Extract airy tag: after "nonred_" to the end (no more "_")
    m_airy = re.search(r"nonred_([^_]+)$", base)
    airy_tag = m_airy.group(1) if m_airy else "airy"
    print("ideal_tag:", ideal_tag)  
    print("airy_tag :", airy_tag) 

    batchnum = 0 
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck{chunk_min:05d}-{chunk_max:05d}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_fch{fch_min:04d}-{fch_max:04d}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))

    print(f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5")

    print("input_datfile_list ", input_datfile_list[0:10])

    print("bl_len, bl_ang ", bl_len, bl_ang)
    
    uvps_dat_list = []
    start = time.time()
    for fname in input_datfile_list: 
        uvpsi = {}
        psci = PSpecContainer(filename=fname, mode='r', keep_open=False)
#         print(psci.groups() )
        uvpsi = psci.get_pspec('dset0', 'dset0_x_dset0' )
#         print("uvpsi ", uvpsi)
#         print( psci.groups )
        uvps_dat_list.append(uvpsi)
    end = time.time()
    print(end - start)
    
#     for fname in input_datfile_list: 
#         with h5py.File(fname, 'r') as f:
#             # List the top-level groups/datasets in the file
#             print("Top-level keys:", list(f.keys()))
#             print("f ", f)
            
#             uvps_dat_list.append(f)

            # Suppose there is a group called 'mygroup', you can access it like this:
#             if 'data_spw0' in f:
#                 mygroup = f['data_spw0']
#                 print(mygroup)
#                 # print("Keys in 'mygroup':", list(mygroup.keys()))


#     print("uvps_dat_list ", uvps_dat_list)
    # global uvpspec_averaged 
    print("Combining ___________________________________")
    uvpspec_averaged  = uvpspec.combine_uvpspec(uvps_dat_list, merge_history=True, verbose=False)
    print( uvpspec_averaged.get_blpairs() )
    return uvpspec_averaged
    
#     print("uvps_dat_tot ", uvps_dat_tot)
    
    

In [ ]:
# bl_len = [25, 29, 44] #, 38]
# bl_ang = [90, 0, 0] #, 161]

# bl_len = [ 15.0, 25.0 ]
# bl_ang = [ 0.0, 150.0 ]

# bl_len = [25, 25, 29, 44]
# bl_ang = [90, 150, 0, 120]

# bl_len = [15, 25, 25, 29, 29, 29, 39, 39, 39, 39, 44, 44, 44, 51, 51, 53, 53, 53, 53, 58, 58, 64, 64, 64, 67, 67, 67, 67, 73, 73, 76, 77, 77, 81, 88]
# bl_ang = [0, 150, 30, 0, 60, 120, 19, 161, 139, 41, 0, 60, 120, 30, 150, 166, 14, 46, 134, 0, 120, 144, 157, 24, 169, 11, 109, 131, 0, 120, 150, 161, 139, 171, 0]

# bl_len = [39, 64, 64, 64, 67, 67, 67,  88]
# bl_ang = [41, 144, 157, 24, 169, 109, 131, 0]

bl_len = [14.0]#, 25, 25, 29, 29, 29, 39, 39, 39, 39, 44, 44, 44, 51, 51, 53, 53, 53, 53, 58, 58 ] 
bl_ang = [0.0]#, 150, 30, 0, 60, 120, 19, 161, 139, 41, 0, 60, 120, 30, 150, 166, 14, 46, 134, 0, 120 ] 

# Loop over the N values and store each combined uvp in a dictionary.
combined_uvp_avg_dict = {}
for pol_in in (pol_root):
    print("pol_in ", pol_in)
    for i in range(len(bl_len)):
        uvp_avg = dat_globber_multi(bl_len[i], bl_ang[i], pol_in)
        if uvp_avg is not None:
            combined_uvp_avg_dict[(bl_len[i],bl_ang[i], pol_in)] = uvp_avg

In [ ]:
for pol_in in (pol_root):
    for i in range(len(bl_len)):
        print(combined_uvp_avg_dict[(bl_len[i],bl_ang[i],pol_in)])

In [ ]:
# pol='yy'

# Must be obtained from collect_num_baselines.sh

# nbls_counts_xx = [81 , 50 , 44 , 56 , 38 , 42 , 29 , 36 , 32 , 22 , 40 , 20 , 24 , 17 , 23 , 24 , 22 , 15 , 17 , 30 , 12 , 13 , 15 , 15 , 25 , 17 , 5 , 8 , 20 , 4 , 8 , 11 , 5 , 20 , 16]

# nbls_counts_xx = [ 22, 13 , 15 , 15 , 25 , 5 , 8, 16]

# nbls_counts_xx = [81 , 50 , 44 , 56 , 38 , 42 , 29 , 36 , 32 , 22 , 40 , 20 , 24 , 17 , 23 , 24 , 22 , 15 , 17 , 30 , 12 ]



# nbls_counts_yy = [81 , 50 , 44 , 56 , 38 , 42 , 29 , 36 , 32 , 22 , 40 , 20 , 24 , 17 , 23 , 24 , 22 , 15 , 17 , 30 , 12 , 13 , 15 , 15 , 25 , 17 , 5 , 8 , 20 , 4 , 8 , 11 , 5 , 20 , 16]

nbls_counts = [7]#1 , 50 , 44 , 56 , 38 , 42 , 29 , 36 , 32 , 22 , 40 , 20 , 24 , 17 , 23 , 24 , 22 , 15 , 17 , 30 , 12]

print(nbls_counts)

In [ ]:
# --- all 14 bands -------------------------------------------------
center_z_all = [
    24.6, 19.9, 16.8, 11.7, 10.8,  9.9,  8.9,
     8.2,  7.6,  7.1,  6.5,  6.0,  5.6,  5.2
]

avg_freq_all_MHz = [
   (50.2 +  62.2)/2,   #  56.20
   (63.3 +  73.5)/2,   #  68.40
   (74.6 +  85.4)/2,   #  80.00
  (108.0 + 116.1)/2,   # 112.05
  (117.3 + 124.4)/2,   # 120.85
  (125.4 + 136.2)/2,   # 130.80
  (138.3 + 148.2)/2,   # 143.25
  (150.1 + 159.2)/2,   # 154.65
  (159.3 + 169.9)/2,   # 164.60
  (171.9 + 181.1)/2,   # 176.50
  (181.4 + 196.4)/2,   # 188.90
  (198.5 + 208.4)/2,   # 203.45
  (212.3 + 220.6)/2,   # 216.45
  (224.3 + 231.1)/2    # 227.70
]

# --- only the 8 bands flagged “Used ✓” ----------------------------
center_z_used = [
    24.6, 19.9, 16.8, 10.8,  9.9,  7.6,  7.1,  5.6
]

avg_freq_used_MHz = [
   (50.2 +  62.2)/2,   #  56.20
   (63.3 +  73.5)/2,   #  68.40
   (74.6 +  85.4)/2,   #  80.00
  (117.3 + 124.4)/2,   # 120.85
  (125.4 + 136.2)/2,   # 130.80
  (159.3 + 169.9)/2,   # 164.60
  (171.9 + 181.1)/2,   # 176.50
  (212.3 + 220.6)/2    # 216.45
]


In [ ]:
# %matplotlib notebook
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

def get_spw_info(uvp):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    spw_indices = uvp.spw_array   # e.g., shape (14,)
    freq = uvp.freq_array         # e.g., shape (1114,)
    spw_freq = uvp.spw_freq_array # e.g., shape (1114,)
    
    spw_ranges = {}
    for spw in spw_indices:
        mask = (spw_freq == spw)
        if np.any(mask):
            freq_min = np.min(freq[mask])
            freq_max = np.max(freq[mask])
            spw_ranges[spw] = (freq_min, freq_max)
        else:
            spw_ranges[spw] = None
    return spw_ranges

def get_plot_data(uvp, uvpspec_averaged, pol, idx):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    blp = uvp.get_blpairs()[0]
    # key = (idx, blp, 'xx')
    key = (idx, blp, pol)

    # Retrieve LST array and convert to hours.
    tarr = uvp.lst_avg_array
    tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
    lst_array_rad = tarr
#         print("lst_array_rad ", lst_array_rad)
    lst_array = tarrq

    dlys = uvp.get_dlys(idx) * 1e9
    index_of_zero = np.where(np.isclose(dlys, 0))[0][0]

    uvp_power = np.abs(np.real(uvp.get_data(key)))[:, index_of_zero]
    uvpspec_averaged_power = np.abs(np.real(uvpspec_averaged.get_data(key)))[:, index_of_zero]

    percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
    median_val = np.nanmedian(percent_diff)
    median_vals.append(median_val)
    mean_val = np.nanmean(percent_diff)
    mean_vals.append(mean_val)

    lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        
    return dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, percent_diff, mean_vals


def get_plot_data_all_blpairs(uvp, uvp_avg, pol, spw):
    """
    Gather τ≈0 spectra from *all* baseline–pairs, concatenate by LST,
    and compute percent-difference between uvp and its averaged version.

    Parameters
    ----------
    uvp, uvp_avg : UVPSpec
        Original and incoherently-averaged spectra.
    pol          : str              (e.g. 'xx' or 'pI')
    spw          : int              (spectral-window index)

    Returns
    -------
    dlys               : (Ndlys,)  delay values [ns]
    zero_idx           : int       index of τ≈0 in `dlys`
    lst_rad_sorted     : (Ntimes,) LST in radians, sorted
    lst_hr_sorted      : (Ntimes,) LST in hours,  "
    uvp_pow_sorted     : (Ntimes,) |P|  from uvp          (τ≈0)
    uvp_avg_pow_sorted : (Ntimes,) |P|  from uvp_avg      (τ≈0)
    pct_diff_sorted    : (Ntimes,) 100*(uvp-avg)/avg
    mean_pct           : float      mean of pct_diff
    median_pct         : float      median of pct_diff
    """
    # ---------- fixed per-spw info ----------
    dlys = uvp.get_dlys(spw) * 1e9          # ns
    print("dlys ", dlys)
    print( (np.where(np.isclose(dlys, 558.54545455))) )
    zero_idx = int(np.where(np.isclose(dlys, 0))[0][0])

    # ---------- gather blocks ----------
    lst_list         = []
    uvp_pow_list     = []
    uvp_avg_pow_list = []

    for blp in uvp.get_blpairs():
        key = (spw, blp, pol)
        print("key ", key)

        # LST (radians) and convert now (same for both uvp and uvp_avg)
        lst_block = uvp.lst_avg_array[uvp.blpair_to_indices(blp)]
        lst_list.append(lst_block)

        # spectra, pick τ≈0 and |.| for power
        uvp_pow_block     = np.abs(np.real(uvp        .get_data(key)))[:, zero_idx]
        print("uvp_pow_block ", uvp_pow_block)
        uvp_avg_pow_block = np.abs(np.real(uvp_avg    .get_data(key)))[:, zero_idx]
        print("uvp_avg_pow_block ", uvp_avg_pow_block)

        uvp_pow_list    .append(uvp_pow_block)
        uvp_avg_pow_list.append(uvp_avg_pow_block)

    # ---------- concatenate and sort by LST ----------
    lst_all         = np.concatenate(lst_list)
    uvp_pow_all     = np.concatenate(uvp_pow_list)
    uvp_avg_pow_all = np.concatenate(uvp_avg_pow_list)

    order           = np.argsort(lst_all)
    lst_rad_sorted  = lst_all        [order]
    uvp_pow_sorted  = uvp_pow_all    [order]
    uvp_avg_sorted  = uvp_avg_pow_all[order]

    # ---------- compute percent difference ----------
    pct_diff_sorted = 100.0 * (uvp_pow_sorted - uvp_avg_sorted) / np.where(
                         uvp_avg_sorted != 0, uvp_avg_sorted, np.nan
                     )

    mean_pct   = np.nanmean(pct_diff_sorted)
    median_pct = np.nanmedian(pct_diff_sorted)

    # LST in hours
    lst_hr_sorted = lst_rad_sorted * (12 / np.pi)
    
    lst_array_roll = np.where(lst_hr_sorted > 20, lst_hr_sorted - 24, lst_hr_sorted)

    return (dlys, zero_idx,
            lst_rad_sorted, lst_hr_sorted, lst_array_roll,
            uvp_pow_sorted, uvp_avg_sorted,
            pct_diff_sorted, mean_pct, median_pct)


median_vals = []
mean_vals = []

def plot_percent_difference(combined_uvp_dict, combined_uvp_avg_dict, zero_delay_pspec_spw=None, avg_pspec=None):
    """
    Plot percent difference between uvp and uvpspec_averaged versus LST for each SPW.
    The title and legend include:
      - The baseline length (in m)
      - The baseline angle (forced to 0° here)
      - The SPW index and its full frequency range (in MHz)
      - The polarization used.
    """
    # for grp_key in combined_uvp_dict:
    #     print("grp_key ", grp_key, grp_key[2])
    #     pol_in=grp_key[2]
    #     spw_ranges = get_spw_info(combined_uvp_dict[grp_key])
    example_uvp = next(iter(combined_uvp_dict.values()))
    spw_ranges = get_spw_info(example_uvp)
    print("spw_ranges ", spw_ranges)
    num_spws = len(spw_ranges)
    fig, axes = plt.subplots(5, 3, figsize=(30, 18), sharex=False)
    axes = axes.flatten(order='C')
    
#     pol = 'pI'
    pol = 'xx'

    
    # Compute baseline information.
    # If you expect a horizontal baseline, you can force the angle to 0°.
    bl_vec = example_uvp.bl_vecs[0]  # using the first baseline vector as an example
    print("bl_vec ", bl_vec)
    baseline_length = np.linalg.norm(bl_vec)  # in meters
    baseline_angle = 0  # Force to 0° (if that's what you expect)
    # Otherwise, if you want to compute the angle:
    # baseline_angle = (np.arctan2(bl_vec[1], bl_vec[0]))
    
    # Iterate over SPWs using sorted items so we can unpack the (min, max) frequency tuple.
    for idx, (spw_key, spw_val) in enumerate(sorted(spw_ranges.items(), key=lambda x: x[0])):
        if spw_val is not None:
            freq_min, freq_max = spw_val
            # Create a string showing the full frequency range in MHz.
            freq_range_str = f"{freq_min/1e6:.2f}-{freq_max/1e6:.2f} MHz"
        else:
            freq_range_str = "N/A"
        
        # (Optional) If you have zero_delay_pspec_spw and avg_pspec available, use them:
        if zero_delay_pspec_spw is not None:
            bl = redgrp_unpol_comb[0]  # e.g., baseline pair (3,5)
            zero_delay_power = np.abs(zero_delay_pspec_spw[bl][pol][:, idx])
            avg_power = avg_pspec[idx][pol]
        
        if zero_delay_pspec_spw is not None:
            percent_diff_totpow = 100 * (zero_delay_power - avg_power) / np.where(avg_power != 0, avg_power, np.nan)
        
        blp = uvp.get_blpairs()[0]
        key = (idx, blp, 'xx')
        
#         for i in range(1):#len(bl_len)):
        for grp_key in combined_uvp_dict:
            print("grp_key ", grp_key)
            dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, percent_diff, mean_vals, med_vals = get_plot_data_all_blpairs(combined_uvp_dict[grp_key], combined_uvp_avg_dict[grp_key], pol, idx)
        print( lst_array_roll.shape, 
              percent_diff.shape
             )
        
        # Retrieve LST array and convert to hours.
#         tarr = uvp.lst_avg_array
#         tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
#         lst_array_rad = tarr
# #         print("lst_array_rad ", lst_array_rad)
#         lst_array = tarrq
        
#         dlys = uvp.get_dlys(idx) * 1e9
#         index_of_zero = np.where(np.isclose(dlys, 0))[0][0]
        
#         uvp_power = np.abs(np.real(uvp.get_data(key)))[:, index_of_zero]
#         uvpspec_averaged_power = np.abs(np.real(uvpspec_averaged.get_data(key)))[:, index_of_zero]
        
        percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
        print("percent_diff shape ", percent_diff.shape)
#         median_val = np.nanmedian(percent_diff)
#         median_vals.append(median_val)
#         mean_val = np.nanmean(percent_diff)
#         mean_vals.append(mean_val)
        
        lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        ax = axes[idx]
        ax.axhline(10, color='blue', alpha=0.5)
        ax.axhline(-10, color='blue', alpha=0.5)
        ax.axhline(5, color='green', alpha=0.5)
        ax.axhline(-5, color='green', alpha=0.5)
        ax.axhline(0, color='black', alpha=0.5)
        
        # Scatter plot: percent difference vs. LST (rolled), using a softer "skyblue" color.
        ax.scatter(lst_array_roll, percent_diff, s=1, marker="o", linestyle="-", color="royalblue",
                   label=f"PSPEC SPW {freq_range_str}")
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.set_ylim(-20, 20)
        ax.set_xlim(-4, 8)
        # ax.set_ylim(-5, 5)
        ax.set_xlim(-4,20)
        ax.grid(False)
        ax.set_ylabel("Percent \n Difference [%]", fontsize=12)
        ax.set_xlabel("LST (Hours)", fontsize=12)
        
        print("np.max(percent_diff) ", np.nanmax(np.abs(percent_diff)))
        print("np.max(percent_diff) ", np.nanmin(np.abs(percent_diff)))
#         print(">0% spw, LST, % ", idx, len(lst_array[percent_diff==np.nanmax(percent_diff)]), lst_array[percent_diff==np.nanmax(percent_diff)], '\n LST rad', lst_array_rad[percent_diff==np.nanmax(percent_diff)], '\n', percent_diff[percent_diff==np.nanmax(percent_diff)] )
#         print("in -5-6% spw, LST, % ", idx, len(lst_array[(percent_diff > -6) & (percent_diff < -5)]), lst_array[(percent_diff > -6) & (percent_diff < -5)], '\n LST rad', lst_array_rad[(percent_diff > -6) & (percent_diff < -5)], '\n', percent_diff[(percent_diff > -6) & (percent_diff < -5)] )
        
        if pol == 'nn':
            pol1 = 'xx'
        else:
            pol1 = pol
        
        # Set title with baseline length, forced angle, SPW index, frequency range, and polarization.
        ax.set_title(f"Len {int(baseline_length)}m, Ang {int(baseline_angle)}°, SPW {idx}, Freq: {freq_range_str}, Pol: {pol1}", fontsize=14)
        ax.legend(fontsize=10)
    
    # Hide any unused subplots.
    for idx in range(len(spw_ranges), len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.show()
    
#     print("Median percent difference for each SPW:")
#     for spw_key, med in zip(sorted(spw_ranges.keys()), median_vals):
#         print(f"SPW {spw_key}: median % difference = {med:.2f}%")
#     print("Mean percent difference for each SPW:")
#     for spw_key, mea in zip(sorted(spw_ranges.keys()), mean_vals):
#         print(f"SPW {spw_key}: mean % difference = {mea:.2f}%")

# Example usage:
plot_percent_difference(combined_uvp_dict, combined_uvp_avg_dict)


In [ ]:
# PLOT PSPEC 

# %matplotlib notebook
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

def bin_along_lst(lst_array, data, bin_width=0.5, stat="median"):
    """
    Bin a 1-D data array into uniform LST bins.

    Parameters
    ----------
    lst_array : array_like
        LST values in hours (same length as `data`).
    data : array_like
        The quantity to bin (e.g. direc_diff).
    bin_width : float
        Bin width in hours of LST (default 0.5 hr).
    stat : str
        Statistic per bin: 'mean', 'median', or 'both'.

    Returns
    -------
    bin_centers : ndarray
        Centre of each LST bin (hours).
    binned_vals : ndarray
        The chosen statistic in each bin (NaN where empty).
    binned_std  : ndarray
        Standard deviation in each bin (useful for errorbars).
    counts      : ndarray (int)
        Number of samples that fell in each bin.
    """
    lst_array = np.asarray(lst_array)
    data = np.asarray(data, dtype=float)

    # Build bin edges spanning the full LST range
    lst_min = np.nanmin(lst_array)
    lst_max = np.nanmax(lst_array)
    bin_edges = np.arange(lst_min, lst_max + bin_width, bin_width)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    nbins = len(bin_centers)
    binned_vals = np.full(nbins, np.nan)
    binned_std  = np.full(nbins, np.nan)
    counts      = np.zeros(nbins, dtype=int)

    # Digitize: assigns each LST sample to a bin index (1-based)
    indices = np.digitize(lst_array, bin_edges)  # 1..len(bin_edges)

    for i in range(nbins):
        mask = indices == (i + 1)          # digitize is 1-based
        n = np.count_nonzero(mask)
        counts[i] = n
        if n == 0:
            continue
        chunk = data[mask]
        if stat == "median":
            binned_vals[i] = np.nanmedian(chunk)
        elif stat == "mean":
            binned_vals[i] = np.nanmean(chunk)
        elif stat == "both":
            binned_vals[i] = np.nanmedian(chunk)   # store median; mean available via separate call
        binned_std[i] = np.nanstd(chunk)

    return bin_centers, binned_vals, binned_std, counts

def get_spw_info(uvp):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    spw_indices = uvp.spw_array   # e.g., shape (14,)
    freq = uvp.freq_array         # e.g., shape (1114,)
    spw_freq = uvp.spw_freq_array # e.g., shape (1114,)
    
    spw_ranges = {}
    for spw in spw_indices:
        mask = (spw_freq == spw)
        if np.any(mask):
            freq_min = np.min(freq[mask])
            freq_max = np.max(freq[mask])
            spw_ranges[spw] = (freq_min, freq_max)
        else:
            spw_ranges[spw] = None
    return spw_ranges

def get_plot_data(uvp, uvpspec_averaged, pol, idx):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    blp = uvp.get_blpairs()[0]
    # key = (idx, blp, 'xx')
    key = (idx, blp, pol)

    # Retrieve LST array and convert to hours.
    tarr = uvp.lst_avg_array
    tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
    lst_array_rad = tarr
#         print("lst_array_rad ", lst_array_rad)
    lst_array = tarrq

    dlys = uvp.get_dlys(idx) * 1e9
    index_of_zero = np.where(np.isclose(dlys, 0))[0][0]

    uvp_power = np.abs(np.real(uvp.get_data(key)))[:, index_of_zero]
    uvpspec_averaged_power = np.abs(np.real(uvpspec_averaged.get_data(key)))[:, index_of_zero]

    percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
    median_val = np.nanmedian(percent_diff)
    median_vals.append(median_val)
    mean_val = np.nanmean(percent_diff)
    mean_vals.append(mean_val)

    lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        
    return dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, percent_diff, mean_vals


def get_plot_data_all_blpairs(uvp, uvp_avg, pol, spw):
    """
    Gather τ≈0 spectra from *all* baseline–pairs, concatenate by LST,
    and compute percent-difference between uvp and its averaged version.

    Parameters
    ----------
    uvp, uvp_avg : UVPSpec
        Original and incoherently-averaged spectra.
    pol          : str              (e.g. 'xx' or 'pI')
    spw          : int              (spectral-window index)

    Returns
    -------
    dlys               : (Ndlys,)  delay values [ns]
    zero_idx           : int       index of τ≈0 in `dlys`
    lst_rad_sorted     : (Ntimes,) LST in radians, sorted
    lst_hr_sorted      : (Ntimes,) LST in hours,  "
    uvp_pow_sorted     : (Ntimes,) |P|  from uvp          (τ≈0)
    uvp_avg_pow_sorted : (Ntimes,) |P|  from uvp_avg      (τ≈0)
    pct_diff_sorted    : (Ntimes,) 100*(uvp-avg)/avg
    mean_pct           : float      mean of pct_diff
    median_pct         : float      median of pct_diff
    """
    # ---------- fixed per-spw info ----------
    dlys = uvp.get_dlys(spw) * 1e9          # ns
    print("dlys ", dlys)
    print( (np.where(np.isclose(dlys, 558.54545455))) )
    zero_idx = int(np.where(np.isclose(dlys, 0))[0][0])     # 1024

    # ---------- gather blocks ----------
    lst_list         = []
    uvp_pow_list     = []
    uvp_avg_pow_list = []

    for blp in uvp.get_blpairs():
        key = (spw, blp, pol)
        print("key ", key)

        # LST (radians) and convert now (same for both uvp and uvp_avg)
        lst_block = uvp.lst_avg_array[uvp.blpair_to_indices(blp)]
        lst_list.append(lst_block)

        # spectra, pick τ≈0 and |.| for power
        uvp_pow_block     = np.abs(np.real(uvp        .get_data(key)))[:, zero_idx]
        print("uvp_pow_block ", uvp_pow_block)
        uvp_avg_pow_block = np.abs(np.real(uvp_avg    .get_data(key)))[:, zero_idx]
        print("uvp_avg_pow_block ", uvp_avg_pow_block)

        uvp_pow_list    .append(uvp_pow_block)
        uvp_avg_pow_list.append(uvp_avg_pow_block)

    # ---------- concatenate and sort by LST ----------
    lst_all         = np.concatenate(lst_list)
    uvp_pow_all     = np.concatenate(uvp_pow_list)
    uvp_avg_pow_all = np.concatenate(uvp_avg_pow_list)

    order           = np.argsort(lst_all)
    lst_rad_sorted  = lst_all        [order]
    uvp_pow_sorted  = uvp_pow_all    [order]
    uvp_avg_sorted  = uvp_avg_pow_all[order]

    # ---------- compute percent difference ----------
    pct_diff_sorted = 100.0 * (uvp_pow_sorted - uvp_avg_sorted) / np.where(
                         uvp_avg_sorted != 0, uvp_avg_sorted, np.nan
                     )

    mean_pct   = np.nanmean(pct_diff_sorted)
    median_pct = np.nanmedian(pct_diff_sorted)

    # LST in hours
    lst_hr_sorted = lst_rad_sorted * (12 / np.pi)
    
    lst_array_roll = np.where(lst_hr_sorted > 20, lst_hr_sorted - 24, lst_hr_sorted)

    return (dlys, zero_idx,
            lst_rad_sorted, lst_hr_sorted, lst_array_roll,
            uvp_pow_sorted, uvp_avg_sorted,
            pct_diff_sorted, mean_pct, median_pct)


median_vals = []
mean_vals = []

def plot_percent_difference(combined_uvp_dict, combined_uvp_avg_dict, zero_delay_pspec_spw=None, avg_pspec=None):
    """
    Plot percent difference between uvp and uvpspec_averaged versus LST for each SPW.
    The title and legend include:
      - The baseline length (in m)
      - The baseline angle (forced to 0° here)
      - The SPW index and its full frequency range (in MHz)
      - The polarization used.
    """
    # for grp_key in combined_uvp_dict:
    #     print("grp_key ", grp_key, grp_key[2])
    #     pol_in=grp_key[2]
    #     spw_ranges = get_spw_info(combined_uvp_dict[grp_key])
    example_uvp = next(iter(combined_uvp_dict.values()))
    spw_ranges = get_spw_info(example_uvp)
    print("spw_ranges ", spw_ranges)
    num_spws = len(spw_ranges)
    fig, axes = plt.subplots(5, 3, figsize=(30, 18), sharex=False)
    axes = axes.flatten(order='C')
    
#     pol = 'pI'
    pol = 'xx'

    
    # Compute baseline information.
    # If you expect a horizontal baseline, you can force the angle to 0°.
    bl_vec = example_uvp.bl_vecs[0]  # using the first baseline vector as an example
    print("bl_vec ", bl_vec)
    baseline_length = np.linalg.norm(bl_vec)  # in meters
    baseline_angle = 0  # Force to 0° (if that's what you expect)
    # Otherwise, if you want to compute the angle:
    # baseline_angle = (np.arctan2(bl_vec[1], bl_vec[0]))
    
    # Iterate over SPWs using sorted items so we can unpack the (min, max) frequency tuple.
    for idx, (spw_key, spw_val) in enumerate(sorted(spw_ranges.items(), key=lambda x: x[0])):
        if spw_val is not None:
            freq_min, freq_max = spw_val
            # Create a string showing the full frequency range in MHz.
            freq_range_str = f"{freq_min/1e6:.2f}-{freq_max/1e6:.2f} MHz"
        else:
            freq_range_str = "N/A"
        
        # (Optional) If you have zero_delay_pspec_spw and avg_pspec available, use them:
        if zero_delay_pspec_spw is not None:
            bl = redgrp_unpol_comb[0]  # e.g., baseline pair (3,5)
            zero_delay_power = np.abs(zero_delay_pspec_spw[bl][pol][:, idx])
            avg_power = avg_pspec[idx][pol]
        
        if zero_delay_pspec_spw is not None:
            percent_diff_totpow = 100 * (zero_delay_power - avg_power) / np.where(avg_power != 0, avg_power, np.nan)
        
        blp = example_uvp.get_blpairs()[0]
        key = (idx, blp, 'xx')
        
#         for i in range(1):#len(bl_len)):
        for grp_key in combined_uvp_dict:
            print("grp_key ", grp_key)
            dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, percent_diff, mean_vals, med_vals = get_plot_data_all_blpairs(combined_uvp_dict[grp_key], combined_uvp_avg_dict[grp_key], pol, idx)
        print( lst_array_roll.shape, 
              percent_diff.shape
             )
        
        # Retrieve LST array and convert to hours.
#         tarr = uvp.lst_avg_array
#         tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
#         lst_array_rad = tarr
# #         print("lst_array_rad ", lst_array_rad)
#         lst_array = tarrq
        
#         dlys = uvp.get_dlys(idx) * 1e9
#         index_of_zero = np.where(np.isclose(dlys, 0))[0][0]
        
#         uvp_power = np.abs(np.real(uvp.get_data(key)))[:, index_of_zero]
#         uvpspec_averaged_power = np.abs(np.real(uvpspec_averaged.get_data(key)))[:, index_of_zero]
        
        # percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
        direc_diff = uvp_power - uvpspec_averaged_power
        print("percent_diff shape ", percent_diff.shape)
#         median_val = np.nanmedian(percent_diff)
#         median_vals.append(median_val)
#         mean_val = np.nanmean(percent_diff)
#         mean_vals.append(mean_val)

        # sine^2 with period = 1 hour LST
        sin2_lst = np.sin(np.pi * lst_array_roll) ** 2
        amplitude = 5000.0  # adjust as needed
        sin2_signal = (amplitude * np.sin(np.pi * lst_array_roll) ** 2) + 100
        
        lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        ax = axes[idx]
        ax.axhline(10, color='blue', alpha=0.5)
        ax.axhline(-10, color='blue', alpha=0.5)
        ax.axhline(5, color='green', alpha=0.5)
        ax.axhline(-5, color='green', alpha=0.5)
        ax.axhline(0, color='black', alpha=0.5)
        
        # Scatter plot: percent difference vs. LST (rolled), using a softer "skyblue" color.
        # ax.plot(lst_array_roll, sin2_signal, color="orange", linewidth=1.5, label=r"$\sin^2(\pi \cdot \mathrm{LST})$, T=1hr")
        # ax.scatter(lst_array_roll, direc_diff, s=1, marker="o", linestyle="-", color="royalblue",
        #            label=f"PSPEC SPW {freq_range_str}")
        ax.scatter(lst_array_roll, uvpspec_averaged_power, s=1, marker="o", linestyle="-", color="royalblue",
                   label=f"PSPEC SPW {freq_range_str}")
        ax.scatter(lst_array_roll, uvp_power, s=1, marker="o", linestyle="-", color="red",
                   label=f"PSPEC SPW {freq_range_str}", alpha = 0.2)
        # Bin with 0.25-hour LST bins, using median
        # bin_cen, bin_val, bin_err, bin_n = bin_along_lst(lst_array_roll, direc_diff, bin_width=0.25, stat="median")
        # # Overlay on the existing axis
        # ax.errorbar(bin_cen, bin_val, yerr=bin_err, fmt="o-", color="orange",
        #             markersize=4, linewidth=1.2, capsize=2,
        #             label=f"Binned (Δt={0.25} hr, median)")
        
        ax.xaxis.set_major_locator(MultipleLocator(1))
        # ax.set_ylim(-20, 20)
        ax.set_xlim(-4, 8)
        # ax.set_ylim(1e6, 2e9)
        ax.set_xlim(-4,20)
        ax.set_yscale('log')
        ax.grid(False)
        ax.set_ylabel(r"PSPEC [mK$^2$]", fontsize=12)
        ax.set_xlabel("LST (Hours)", fontsize=12)
        
        print("np.max(direc_diff) ", np.nanmax(np.abs(direc_diff)))
        print("np.max(direc_diff) ", np.nanmin(np.abs(direc_diff)))
#         print(">0% spw, LST, % ", idx, len(lst_array[direc_diff==np.nanmax(direc_diff)]), lst_array[direc_diff==np.nanmax(direc_diff)], '\n LST rad', lst_array_rad[direc_diff==np.nanmax(direc_diff)], '\n', direc_diff[direc_diff==np.nanmax(direc_diff)] )
#         print("in -5-6% spw, LST, % ", idx, len(lst_array[(direc_diff > -6) & (direc_diff < -5)]), lst_array[(direc_diff > -6) & (direc_diff < -5)], '\n LST rad', lst_array_rad[(direc_diff > -6) & (direc_diff < -5)], '\n', direc_diff[(direc_diff > -6) & (direc_diff < -5)] )
        
        if pol == 'nn':
            pol1 = 'xx'
        else:
            pol1 = pol
        
        # Set title with baseline length, forced angle, SPW index, frequency range, and polarization.
        ax.set_title(f"Len {int(baseline_length)}m, Ang {int(baseline_angle)}°, SPW {idx}, Freq: {freq_range_str}, Pol: {pol1}", fontsize=14)
        ax.legend(fontsize=10)
    
    # Hide any unused subplots.
    for idx in range(len(spw_ranges), len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.show()
    
#     print("Median percent difference for each SPW:")
#     for spw_key, med in zip(sorted(spw_ranges.keys()), median_vals):
#         print(f"SPW {spw_key}: median % difference = {med:.2f}%")
#     print("Mean percent difference for each SPW:")
#     for spw_key, mea in zip(sorted(spw_ranges.keys()), mean_vals):
#         print(f"SPW {spw_key}: mean % difference = {mea:.2f}%")

# Example usage:
plot_percent_difference(combined_uvp_dict, combined_uvp_avg_dict)


In [ ]:
# Fourier spectrum along the LST axis for the PSPEC curves plotted above

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows as signal_windows


def get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw):
    """
    Recompute the same tau ~= 0 LST tracks plotted in the previous PSPEC cell,
    but without the diagnostic print statements.
    """
    dlys = uvp.get_dlys(spw) * 1e9
    zero_candidates = np.where(np.isclose(dlys, 0.0))[0]
    zero_idx = int(zero_candidates[0]) if zero_candidates.size else int(np.nanargmin(np.abs(dlys)))

    lst_blocks = []
    uvp_power_blocks = []
    uvp_avg_power_blocks = []

    for blp in uvp.get_blpairs():
        key = (spw, blp, pol)
        lst_blocks.append(uvp.lst_avg_array[uvp.blpair_to_indices(blp)])
        uvp_power_blocks.append(np.abs(np.real(uvp.get_data(key)))[:, zero_idx])
        uvp_avg_power_blocks.append(np.abs(np.real(uvp_avg.get_data(key)))[:, zero_idx])

    lst_rad = np.concatenate(lst_blocks)
    lst_hr = lst_rad * (12.0 / np.pi)
    lst_array_roll = np.where(lst_hr > 20.0, lst_hr - 24.0, lst_hr)
    uvp_power = np.concatenate(uvp_power_blocks)
    uvpspec_averaged_power = np.concatenate(uvp_avg_power_blocks)

    order = np.argsort(lst_array_roll)
    return (
        dlys,
        zero_idx,
        lst_rad[order],
        lst_hr[order],
        lst_array_roll[order],
        uvp_power[order],
        uvpspec_averaged_power[order],
    )


def _uniform_lst_grid(lst_hours, values, dt_hours=None, statistic="median"):
    """Bin/interpolate an LST series onto a uniform grid for an FFT."""
    x = np.asarray(lst_hours, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]

    if x.size < 2:
        raise ValueError("Need at least two finite LST samples for an FFT.")

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    unique_x = np.unique(x)
    positive_dx = np.diff(unique_x)
    positive_dx = positive_dx[positive_dx > 0]
    if dt_hours is None:
        if positive_dx.size == 0:
            raise ValueError("Could not infer an LST sampling interval.")
        dt_hours = np.nanmedian(positive_dx)

    if not np.isfinite(dt_hours) or dt_hours <= 0:
        raise ValueError("dt_hours must be a positive finite number.")

    n_grid = int(np.floor((x.max() - x.min()) / dt_hours)) + 1
    lst_grid = x.min() + dt_hours * np.arange(n_grid)
    if lst_grid[-1] < x.max() - 0.25 * dt_hours:
        lst_grid = np.append(lst_grid, lst_grid[-1] + dt_hours)

    y_grid = np.full(lst_grid.size, np.nan)
    bin_index = np.floor((x - (lst_grid[0] - 0.5 * dt_hours)) / dt_hours).astype(int)
    valid = (bin_index >= 0) & (bin_index < lst_grid.size)

    for idx in np.unique(bin_index[valid]):
        chunk = y[valid & (bin_index == idx)]
        if statistic == "mean":
            y_grid[idx] = np.nanmean(chunk)
        else:
            y_grid[idx] = np.nanmedian(chunk)

    good = np.isfinite(y_grid)
    if np.count_nonzero(good) < 2:
        raise ValueError("Need at least two populated LST bins for an FFT.")

    if not np.all(good):
        y_grid = np.interp(lst_grid, lst_grid[good], y_grid[good])

    return lst_grid, y_grid, dt_hours


def _clean_sort_lst_samples(lst_hours, values):
    """Return finite LST/value samples sorted by LST."""
    x = np.asarray(lst_hours, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if x.size < 2:
        raise ValueError("Need at least two finite LST samples for a Fourier transform.")
    order = np.argsort(x)
    return x[order], y[order]


def _collapse_repeated_lst_samples(lst_hours, values, dt_hours=None, statistic="median", duplicate_tol=None):
    """
    Collapse repeated or nearly repeated LST samples before a Fourier transform.

    This is useful because concatenating many redundant baseline-pair PSPEC values can
    produce several values at the same LST. Treating those repeats as a time stream
    would overweight that LST, so by default we reduce each repeated-LST group to
    one robust representative value.
    """
    x, y = _clean_sort_lst_samples(lst_hours, values)

    unique_x = np.unique(x)
    positive_dx = np.diff(unique_x)
    positive_dx = positive_dx[positive_dx > 0]
    if duplicate_tol is None:
        if dt_hours is not None and np.isfinite(dt_hours) and dt_hours > 0:
            duplicate_tol = 0.05 * dt_hours
        elif positive_dx.size:
            duplicate_tol = 0.05 * np.nanmedian(positive_dx)
        else:
            duplicate_tol = 0.0

    groups_x = []
    groups_y = []
    start = 0
    for idx in range(1, x.size + 1):
        at_end = idx == x.size
        starts_new_group = False if at_end else (x[idx] - x[start] > duplicate_tol)
        if at_end or starts_new_group:
            x_chunk = x[start:idx]
            y_chunk = y[start:idx]
            groups_x.append(np.nanmedian(x_chunk))
            if statistic == "mean":
                groups_y.append(np.nanmean(y_chunk))
            else:
                groups_y.append(np.nanmedian(y_chunk))
            start = idx

    return np.asarray(groups_x, dtype=float), np.asarray(groups_y, dtype=float)


def _is_uniform_sampling(lst_hours, dt_hours=None, rtol=1e-5, atol=1e-8):
    """Check whether sorted LST samples are regularly spaced to tolerance."""
    x = np.asarray(lst_hours, dtype=float)
    if x.size < 2:
        return False, np.nan
    dx = np.diff(x)
    positive_dx = dx[dx > 0]
    if positive_dx.size == 0:
        return False, np.nan
    dt = np.nanmedian(positive_dx) if dt_hours is None else dt_hours
    if not np.isfinite(dt) or dt <= 0:
        return False, np.nan
    tol = max(float(atol), float(rtol) * abs(dt))
    return bool(np.all(np.abs(dx - dt) <= tol)), dt


def _lst_window_weights(n, window):
    """Construct a Fourier taper using NumPy/SciPy windows where available."""
    window_key = None if window is None else str(window).lower().replace("_", "-")
    if window_key == "hann" and n >= 3:
        return np.hanning(n)
    if window_key in ("blackmanharris", "blackman-harris", "blackman harris", "bh") and n >= 3:
        return signal_windows.blackmanharris(n, sym=False)
    if window_key in (None, "boxcar", "rect", "rectangular"):
        return np.ones(n)
    raise ValueError("window must be 'hann', 'blackmanharris', 'boxcar', or None.")


def _preprocess_lst_fft_values(values, normalize="fractional", remove_mean=True):
    """Apply optional fractional normalization and optional DC/mean removal."""
    y_fft = np.asarray(values, dtype=float).copy()
    normalize_key = "none" if normalize is None else str(normalize).lower()

    if normalize_key == "fractional":
        scale = np.nanmedian(np.abs(y_fft))
        if np.isfinite(scale) and scale > 0:
            y_fft = y_fft / scale - 1.0
    elif normalize_key in ("none", "raw"):
        pass
    else:
        raise ValueError("normalize must be 'fractional', 'none', 'raw', or None.")

    if remove_mean:
        y_fft = y_fft - np.nanmean(y_fft)
    return np.nan_to_num(y_fft, nan=0.0, posinf=0.0, neginf=0.0)


def _one_sided_amplitude(fft, n, coherent_gain):
    """Convert an rFFT-like complex spectrum to a one-sided amplitude spectrum."""
    amp = np.abs(fft) / (n * coherent_gain)
    if n % 2 == 0 and amp.size > 2:
        amp[1:-1] *= 2.0
    elif n % 2 == 1 and amp.size > 1:
        amp[1:] *= 2.0
    return amp


def _direct_nonuniform_fourier(lst_hours, values, dt_hours=None, freq_grid=None):
    """
    Directly evaluate sum_n values[n] exp(-2 pi i f (t_n - t_0)).

    This avoids interpolation/gridding for irregular LST samples. It is slower than
    an FFT, but the arrays here are small enough that the clarity is useful.
    """
    x = np.asarray(lst_hours, dtype=float)
    y = np.asarray(values, dtype=complex)
    n = x.size
    if freq_grid is None:
        _, dt_eff = _is_uniform_sampling(x, dt_hours=dt_hours)
        if not np.isfinite(dt_eff) or dt_eff <= 0:
            positive_dx = np.diff(np.unique(x))
            positive_dx = positive_dx[positive_dx > 0]
            if positive_dx.size == 0:
                raise ValueError("Could not infer a frequency grid for the direct FT.")
            dt_eff = np.nanmedian(positive_dx)
        freq = np.fft.rfftfreq(n, d=dt_eff)
    else:
        freq = np.asarray(freq_grid, dtype=float)

    phase = np.exp(-2j * np.pi * freq[:, None] * (x[None, :] - x[0]))
    return freq, phase @ y


def lst_fft_spectrum(lst_hours,
                     values,
                     dt_hours=None,
                     normalize="fractional",
                     window="hann",
                     method="auto",
                     statistic="median",
                     collapse_repeats=True,
                     duplicate_tol=None,
                     remove_mean=True,
                     uniform_rtol=1e-5,
                     uniform_atol=1e-8,
                     freq_grid=None,
                     return_info=False):
    """
    Fourier transform a PSPEC-vs-LST track.

    Frequencies are in cycles per LST hour.

    Parameters
    ----------
    method : {'auto', 'grid', 'fft', 'direct', 'nonuniform'}
        'grid' reproduces the original robust notebook behavior: bin/interpolate onto
        a uniform grid and FFT. 'fft' requires the LST samples to be uniform after the
        optional repeated-LST collapse. 'direct'/'nonuniform' evaluates the Fourier
        sum at the requested frequencies without gridding. 'auto' uses an FFT when
        samples are already uniform and otherwise falls back to the direct sum.
    normalize : {'fractional', 'none', 'raw', None}
        'fractional' transforms P/median(|P|) - 1. Use None/'none'/'raw' to transform
        the PSPEC values themselves.
    remove_mean : bool
        If True, subtract the arithmetic mean before transforming. Set False to keep
        the DC mode and make the transform as raw as possible.
    window : {None, 'boxcar', 'hann', 'blackman-harris', 'blackmanharris', 'bh'}
        Optional LST taper. Blackman-Harris uses scipy.signal.windows.blackmanharris.
    """
    method_key = "auto" if method is None else str(method).lower()
    method_aliases = {"nudft": "direct", "nonuniform": "direct", "non-uniform": "direct"}
    method_key = method_aliases.get(method_key, method_key)
    if method_key not in ("auto", "grid", "fft", "direct"):
        raise ValueError("method must be 'auto', 'grid', 'fft', 'direct', or 'nonuniform'.")

    if method_key == "grid":
        lst_grid, y_grid, dt_eff = _uniform_lst_grid(
            lst_hours,
            values,
            dt_hours=dt_hours,
            statistic=statistic,
        )
        transform_method = "grid-fft"
    else:
        if collapse_repeats:
            lst_grid, y_grid = _collapse_repeated_lst_samples(
                lst_hours,
                values,
                dt_hours=dt_hours,
                statistic=statistic,
                duplicate_tol=duplicate_tol,
            )
        else:
            lst_grid, y_grid = _clean_sort_lst_samples(lst_hours, values)

        is_uniform, dt_eff = _is_uniform_sampling(
            lst_grid,
            dt_hours=dt_hours,
            rtol=uniform_rtol,
            atol=uniform_atol,
        )

        if method_key == "fft" and not is_uniform:
            raise ValueError("method='fft' requires uniformly sampled LSTs. Use method='grid', 'direct', or 'auto'.")
        if method_key == "auto":
            transform_method = "direct" if not is_uniform else "fft"
        else:
            transform_method = method_key

    y_fft = _preprocess_lst_fft_values(y_grid, normalize=normalize, remove_mean=remove_mean)
    n = y_fft.size
    weights = _lst_window_weights(n, window)
    coherent_gain = np.mean(weights)
    if not np.isfinite(coherent_gain) or coherent_gain == 0:
        coherent_gain = 1.0

    if transform_method in ("fft", "grid-fft"):
        fft = np.fft.rfft(y_fft * weights)
        freq = np.fft.rfftfreq(n, d=dt_eff)
    else:
        freq, fft = _direct_nonuniform_fourier(
            lst_grid,
            y_fft * weights,
            dt_hours=dt_eff,
            freq_grid=freq_grid,
        )

    amp = _one_sided_amplitude(fft, n, coherent_gain)
    info = {
        "method": transform_method,
        "requested_method": method_key,
        "dt_hours": dt_eff,
        "n_samples": int(n),
        "collapse_repeats": bool(collapse_repeats),
        "normalize": normalize,
        "remove_mean": bool(remove_mean),
        "window": window,
        "coherent_gain": coherent_gain,
    }

    if return_info:
        return freq, amp, lst_grid, y_grid, info
    return freq, amp, lst_grid, y_grid


def _running_percentile(values, window, percentile=50):
    """Simple centered running percentile; keeps this cell independent of scipy."""
    y = np.asarray(values, dtype=float)
    n = y.size
    if n == 0:
        return y

    window = int(window)
    if window < 3:
        window = 3
    if window % 2 == 0:
        window += 1
    if window > n:
        window = n if n % 2 else max(1, n - 1)

    half = window // 2
    out = np.full(n, np.nan)
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        out[i] = np.nanpercentile(y[lo:hi], percentile)
    return out


def spectral_information_filter(freq,
                                amp,
                                n_features=5,
                                continuum_window=19,
                                continuum_percentile=45,
                                score_threshold=2.5,
                                min_frequency=0.0,
                                max_frequency=6.0,
                                min_separation_bins=2):
    """
    Pick Fourier feature regions by spectral whitening rather than by a fixed bandpass.

    Formalism:
      1. Work in log-amplitude, log10 A(f).
      2. Estimate a smooth continuum C(f) with a broad running percentile.
         This continuum captures the expected red/noisy fall from low to high cycles.
      3. Score each mode by its robust residual above the continuum:

             score(f) = [log10 A(f) - C(f)] / [1.4826 * MAD(residual)]

      4. Group contiguous above-threshold bins into feature regions and rank them
         by integrated positive excess. This makes the selection favor coherent
         bumps over isolated noisy high-frequency bins.
    """
    freq = np.asarray(freq, dtype=float)
    amp = np.asarray(amp, dtype=float)

    continuum = np.full_like(amp, np.nan, dtype=float)
    score = np.full_like(amp, np.nan, dtype=float)
    excess_ratio = np.full_like(amp, np.nan, dtype=float)

    valid = np.isfinite(freq) & np.isfinite(amp) & (freq > 0) & (amp > 0)
    if max_frequency is not None:
        valid &= freq <= max_frequency
    valid &= freq >= min_frequency

    valid_idx = np.flatnonzero(valid)
    if valid_idx.size < 5:
        return {
            "continuum": continuum,
            "score": score,
            "excess_ratio": excess_ratio,
            "features": [],
        }

    f_valid = freq[valid_idx]
    log_amp = np.log10(amp[valid_idx])
    log_continuum = _running_percentile(log_amp, continuum_window, percentile=continuum_percentile)
    residual = log_amp - log_continuum

    # A tiny median smooth suppresses one-bin spikes but preserves broad bumps.
    residual_for_peaks = _running_percentile(residual, 3, percentile=50)
    med = np.nanmedian(residual_for_peaks)
    mad = 1.4826 * np.nanmedian(np.abs(residual_for_peaks - med))
    if not np.isfinite(mad) or mad <= 0:
        mad = np.nanstd(residual_for_peaks)
    if not np.isfinite(mad) or mad <= 0:
        mad = 1.0

    local_score = (residual_for_peaks - med) / mad
    continuum[valid_idx] = 10.0 ** log_continuum
    score[valid_idx] = local_score
    excess_ratio[valid_idx] = 10.0 ** residual

    df = np.nanmedian(np.diff(f_valid)) if f_valid.size > 1 else 0.0
    half_df = 0.5 * df if np.isfinite(df) and df > 0 else 0.0

    def build_feature(region_local_idx):
        region_local_idx = np.asarray(region_local_idx, dtype=int)
        peak_loc = region_local_idx[np.nanargmax(local_score[region_local_idx])]
        peak_glob = valid_idx[peak_loc]
        region_glob = valid_idx[region_local_idx]

        weights = np.clip(residual[region_local_idx], 0.0, None)
        if np.nansum(weights) > 0:
            centroid = np.nansum(f_valid[region_local_idx] * weights) / np.nansum(weights)
        else:
            centroid = freq[peak_glob]

        region_score = np.nansum(np.clip(local_score[region_local_idx] - score_threshold, 0.0, None))
        if not np.isfinite(region_score) or region_score <= 0:
            region_score = np.nanmax(local_score[region_local_idx])

        f_min = max(0.0, freq[region_glob[0]] - half_df)
        f_max = freq[region_glob[-1]] + half_df

        return {
            "frequency_cyc_per_hr": freq[peak_glob],
            "frequency_centroid_cyc_per_hr": centroid,
            "frequency_min_cyc_per_hr": f_min,
            "frequency_max_cyc_per_hr": f_max,
            "period_hr": np.inf if freq[peak_glob] == 0 else 1.0 / freq[peak_glob],
            "amplitude": amp[peak_glob],
            "continuum": continuum[peak_glob],
            "excess_ratio": excess_ratio[peak_glob],
            "score": score[peak_glob],
            "region_score": region_score,
            "n_bins": int(region_local_idx.size),
        }

    above = np.isfinite(local_score) & (local_score >= score_threshold)
    regions = []
    start = None
    for i, is_above in enumerate(above):
        if is_above and start is None:
            start = i
        if start is not None and ((not is_above) or i == above.size - 1):
            stop = i if not is_above else i + 1
            regions.append(np.arange(start, stop))
            start = None

    if not regions:
        # Fallback: report the strongest positive local maxima if the threshold is too strict.
        local_max = np.zeros(valid_idx.size, dtype=bool)
        if valid_idx.size > 2:
            local_max[1:-1] = (
                (local_score[1:-1] >= local_score[:-2])
                & (local_score[1:-1] > local_score[2:])
            )
        candidate_local_idx = np.flatnonzero(local_max & (local_score > 0))
        order = candidate_local_idx[np.argsort(local_score[candidate_local_idx])[::-1]]
        chosen = []
        for loc in order:
            glob = valid_idx[loc]
            if any(abs(glob - prev) < min_separation_bins for prev in chosen):
                continue
            chosen.append(glob)
            regions.append(np.array([loc]))
            if len(regions) >= n_features:
                break

    features = [build_feature(region) for region in regions]
    features = sorted(features, key=lambda feat: feat["region_score"], reverse=True)[:n_features]

    return {
        "continuum": continuum,
        "score": score,
        "excess_ratio": excess_ratio,
        "features": features,
    }

def plot_lst_fourier_spectrum(combined_uvp_dict,
                              combined_uvp_avg_dict,
                              pol="xx",
                              group_key=None,
                              dominant_n=5,
                              normalize="fractional",
                              window="hann",
                              fourier_method="auto",
                              remove_mean=True,
                              collapse_repeats=True,
                              bin_statistic="median",
                              duplicate_tol=None,
                              uniform_rtol=1e-5,
                              uniform_atol=1e-8,
                              plot_max_frequency=6.0,
                              feature_score_threshold=2.5,
                              continuum_window=19,
                              continuum_percentile=45,
                              max_cols=3):
    """
    Plot LST-axis Fourier amplitude spectra and highlight continuum-excess features.

    Fourier controls are passed to lst_fft_spectrum(). Use fourier_method='grid'
    for the original gridded diagnostic FFT, fourier_method='auto' to skip gridding
    when samples are already uniform and otherwise use a direct nonuniform Fourier
    sum, or fourier_method='direct' to always avoid gridding.
    """
    if group_key is None:
        group_key = list(combined_uvp_dict.keys())[-1]

    uvp = combined_uvp_dict[group_key]
    uvp_avg = combined_uvp_avg_dict[group_key]
    spw_items = sorted(get_spw_info(uvp).items(), key=lambda item: item[0])

    n_spws = len(spw_items)
    ncols = min(max_cols, max(1, n_spws))
    nrows = int(np.ceil(n_spws / ncols))
    subplot_width = 7.2
    subplot_height = 3.8
    fig_width = subplot_width * ncols
    fig_height = subplot_height * nrows + 0.45

    rc = {
        "figure.figsize": (fig_width, fig_height),
        "figure.dpi": 110,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7,
        "figure.titlesize": 11,
    }

    mode_summary = {}
    fft_info_summary = {}
    bl_vec = uvp.bl_vecs[0]
    baseline_length = np.linalg.norm(bl_vec)

    with plt.rc_context(rc):
        fig, axes = plt.subplots(nrows, ncols, squeeze=False, constrained_layout=True)
        axes = axes.ravel()

        for ax_idx, (spw_key, spw_val) in enumerate(spw_items):
            ax = axes[ax_idx]
            if spw_val is None:
                freq_range_str = "N/A"
            else:
                freq_min, freq_max = spw_val
                freq_range_str = f"{freq_min / 1e6:.2f}-{freq_max / 1e6:.2f} MHz"

            try:
                (_, _, _, _, lst_array_roll,
                 uvp_power, uvpspec_averaged_power) = get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw_key)

                spectra = [
                    ("uvpspec_averaged_power", uvpspec_averaged_power, "royalblue", 1.0),
                    ("uvp_power", uvp_power, "crimson", 0.82),
                ]
                mode_summary[spw_key] = {}
                fft_info_by_label = {}

                for label, values, color, alpha in spectra:
                    freq, amp, _, _, fft_info = lst_fft_spectrum(
                        lst_array_roll,
                        values,
                        normalize=normalize,
                        window=window,
                        method=fourier_method,
                        statistic=bin_statistic,
                        collapse_repeats=collapse_repeats,
                        duplicate_tol=duplicate_tol,
                        remove_mean=remove_mean,
                        uniform_rtol=uniform_rtol,
                        uniform_atol=uniform_atol,
                        return_info=True,
                    )
                    fft_info_by_label[label] = fft_info
                    use = freq > 0
                    if plot_max_frequency is not None:
                        use &= freq <= plot_max_frequency

                    feature_result = spectral_information_filter(
                        freq,
                        amp,
                        n_features=dominant_n,
                        continuum_window=continuum_window,
                        continuum_percentile=continuum_percentile,
                        score_threshold=feature_score_threshold,
                        max_frequency=plot_max_frequency,
                    )
                    mode_summary[spw_key][label] = feature_result["features"]

                    ax.plot(freq[use], amp[use], color=color, alpha=alpha, linewidth=1.15, label=label)
                    continuum = feature_result["continuum"]
                    ax.plot(freq[use], continuum[use], color=color, linestyle="--", alpha=0.45, linewidth=0.9)

                    for feat in feature_result["features"]:
                        ax.axvspan(
                            feat["frequency_min_cyc_per_hr"],
                            feat["frequency_max_cyc_per_hr"],
                            color=color,
                            alpha=0.08,
                            linewidth=0,
                            zorder=0,
                        )

                    feature_freq = [feat["frequency_cyc_per_hr"] for feat in feature_result["features"]]
                    feature_amp = [feat["amplitude"] for feat in feature_result["features"]]
                    if feature_freq:
                        ax.scatter(
                            feature_freq,
                            feature_amp,
                            s=34,
                            marker="o",
                            facecolors="none",
                            edgecolors=color,
                            linewidths=1.4,
                            zorder=5,
                        )

                fft_info_summary[spw_key] = dict(fft_info_by_label)

            except Exception as exc:
                ax.text(0.5, 0.5, f"SPW {spw_key}\n{exc}", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            ax.set_yscale("log")
            # ax.set_xscale("log")
            ax.grid(alpha=0.22, linewidth=0.6)
            if plot_max_frequency is not None:
                ax.set_xlim(0, plot_max_frequency)
            ax.margins(x=0.02)
            ax.set_title(f"{baseline_length:.1f} m | SPW {spw_key} | {freq_range_str}", pad=4)
            if ax_idx // ncols == nrows - 1:
                ax.set_xlabel("LST Fourier frequency [cycles / hr]")
            if ax_idx % ncols == 0:
                ylabel = "FFT amplitude"
                if normalize == "fractional":
                    ylabel += "\n(fractional PSPEC)"
                ax.set_ylabel(ylabel)
            ax.legend(loc="best", frameon=True, borderpad=0.35, handlelength=1.8)

        for ax in axes[n_spws:]:
            fig.delaxes(ax)

        group_label = str(group_key)
        if len(group_label) > 52:
            group_label = group_label[:49] + "..."
        fig.suptitle(f"LST Fourier feature spectrum | group={group_label} | pol={pol}")
        plt.show()

    print("Feature-selected non-DC LST Fourier modes")
    print("score = robust log-amplitude excess above a smooth local continuum")
    for spw_key, label_modes in mode_summary.items():
        print(f"SPW {spw_key}:")
        for label, features in label_modes.items():
            if not features:
                print(f"  {label}: no feature above the current threshold")
                continue
            formatted = ", ".join(
                f"{feat['frequency_cyc_per_hr']:.4g} cyc/hr "
                f"[{feat['frequency_min_cyc_per_hr']:.4g}, {feat['frequency_max_cyc_per_hr']:.4g}] "
                f"(period {feat['period_hr']:.3g} hr, "
                f"excess x{feat['excess_ratio']:.2g}, "
                f"score {feat['score']:.2g}, region {feat['region_score']:.2g})"
                for feat in features
            )
            print(f"  {label}: {formatted}")
        for base_label, info in fft_info_summary.get(spw_key, {}).items():
            print(
                f"  {base_label} FFT: method={info['method']}, "
                f"N={info['n_samples']}, dt={info['dt_hours']:.5g} hr, "
                f"normalize={info['normalize']}, remove_mean={info['remove_mean']}, window={info['window']}"
            )

    return mode_summary


lst_fft_mode_summary = plot_lst_fourier_spectrum(
    combined_uvp_dict,
    combined_uvp_avg_dict,
    pol="xx",
    group_key=None,
    dominant_n=5,
    normalize="fractional",
    window="blackman-harris",
    fourier_method="direct",   # 'auto', 'grid', 'fft', or 'direct'/'nonuniform'
    remove_mean=True,         # set False to keep the DC mode in the transformed data
    collapse_repeats=True,    # collapse repeated baseline-pair samples at the same LST
    bin_statistic="median",   # repeated-LST/bin collapse statistic: 'median' or 'mean'
    plot_max_frequency=6.0,
    feature_score_threshold=2.5,
    continuum_window=15,
    continuum_percentile=40,
)


## Fringe spacing and FWHM from the RIME phase factor

Start with the geometric phase term in the scalar RIME for a baseline vector $\mathbf{b}$ at observing frequency $\nu$:

$$
V_{\mathbf{b}}(\nu)
= \int A(\hat{s},\nu) I(\hat{s},\nu)
\exp\left[-2\pi i\,\frac{\nu}{c}\,\mathbf{b}\cdot(\hat{s}-\hat{s}_0)\right] d\Omega .
$$

The exponential comes from the geometric delay between antennas. A plane wave from direction $\hat{s}$ reaches two antennas separated by $\mathbf{b}$ with delay

$$
\tau_g = \frac{\mathbf{b}\cdot(\hat{s}-\hat{s}_0)}{c},
$$

so the correlator sees the phase

$$
\phi = 2\pi \nu \tau_g
= 2\pi\frac{\mathbf{b}\cdot(\hat{s}-\hat{s}_0)}{\lambda},
\qquad
\lambda = \frac{c}{\nu}.
$$

Thus the complex fringe is

$$
\exp[-i\phi] = \cos\phi - i\sin\phi .
$$

The real cosine fringe is therefore not a separate physical assumption; it is the real part of the full complex phasor.

Now take an angular offset $\theta$ along the projected baseline direction. Then

$$
\mathbf{b}\cdot(\hat{s}-\hat{s}_0) \simeq b\sin\theta,
$$

and

$$
\phi(\theta) = 2\pi\frac{b}{\lambda}\sin\theta .
$$

### 1. Peak-to-peak fringe spacing

For the real fringe

$$
R(\theta)=\cos\phi(\theta),
$$

neighboring maxima occur when the phase changes by $2\pi$:

$$
\Delta\phi = 2\pi.
$$

Starting at the central maximum, $\theta=0$, the next maximum satisfies

$$
2\pi\frac{b}{\lambda}\sin\theta_{\rm fringe}=2\pi,
$$

so

$$
\boxed{\theta_{\rm fringe} = \arcsin\left(\frac{\lambda}{b}\right)}
$$

when $\lambda/b \le 1$. In the small-angle limit,

$$
\boxed{\theta_{\rm fringe} \simeq \frac{\lambda}{b}}
\qquad \mathrm{rad}.
$$

This is the usual interferometric fringe spacing or angular period near the phase center.

### 2. FWHM of the real amplitude fringe

For the central positive lobe of $R(\theta)=\cos\phi(\theta)$, the half-amplitude point satisfies

$$
\cos\phi_{1/2}=\frac{1}{2},
\qquad
\phi_{1/2}=\frac{\pi}{3}.
$$

Therefore

$$
2\pi\frac{b}{\lambda}\sin\theta_{1/2}=\frac{\pi}{3},
$$

which gives

$$
\sin\theta_{1/2}=\frac{\lambda}{6b}.
$$

The full width at half maximum is twice this one-sided angle:

$$
\boxed{\mathrm{FWHM}_{\rm amp}
=2\arcsin\left(\frac{\lambda}{6b}\right)}.
$$

For small angles,

$$
\boxed{\mathrm{FWHM}_{\rm amp} \simeq \frac{\lambda}{3b}}
\qquad \mathrm{rad}.
$$

### 3. FWHM of the power fringe

If the relevant quantity is power-like,

$$
P(\theta)=\cos^2\phi(\theta),
$$

then the half-power point satisfies

$$
\cos^2\phi_{1/2}=\frac{1}{2},
\qquad
\phi_{1/2}=\frac{\pi}{4}.
$$

Thus

$$
2\pi\frac{b}{\lambda}\sin\theta_{1/2}=\frac{\pi}{4},
$$

so

$$
\boxed{\mathrm{FWHM}_{\rm power}
=2\arcsin\left(\frac{\lambda}{8b}\right)}.
$$

For small angles,

$$
\boxed{\mathrm{FWHM}_{\rm power} \simeq \frac{\lambda}{4b}}
\qquad \mathrm{rad}.
$$

### 4. LST-hour conversion

Use

$$
24\ \mathrm{hr}_{\rm LST}=360^\circ=2\pi\ \mathrm{rad}.
$$

Therefore any angular width can be converted to LST hours by

$$
\boxed{\Delta t_{\rm LST}\,[\mathrm{hr}]
= \frac{\Delta\theta\,[\mathrm{deg}]}{15}
= \frac{12}{\pi}\,\Delta\theta\,[\mathrm{rad}]}.
$$

So the three useful widths are

$$
\boxed{\Delta t_{{\rm fringe},\,\rm LST}
= \frac{12}{\pi}\arcsin\left(\frac{\lambda}{b}\right)},
$$

$$
\boxed{\Delta t_{{\rm amp},\,\rm LST}
= \frac{24}{\pi}\arcsin\left(\frac{\lambda}{6b}\right)},
$$

and

$$
\boxed{\Delta t_{{\rm power},\,\rm LST}
= \frac{24}{\pi}\arcsin\left(\frac{\lambda}{8b}\right)}.
$$

Here $b$ is the projected baseline length in meters. A baseline length alone is not enough: the angular scale is set by $\lambda/b=c/(\nu b)$.


In [ ]:
# Fringe spacing, FWHM, LST conversion, and diagnostic plots

import numpy as np
import matplotlib.pyplot as plt

C_M_PER_S = 299_792_458.0
DEG_PER_LST_HOUR = 360.0 / 24.0
HERA_LATITUDE_DEG_FALLBACK = -30.7215271


def _default_hera_latitude_deg():
    """Return HERA latitude in degrees, preferring the packaged hera_sim value."""
    try:
        from hera_sim.io import HERA_LAT_LON_ALT
        return float(np.asarray(HERA_LAT_LON_ALT, dtype=float).ravel()[0]), "hera_sim.io.HERA_LAT_LON_ALT"
    except Exception:
        return HERA_LATITUDE_DEG_FALLBACK, "HERA_LATITUDE_DEG_FALLBACK"


def _latitude_deg_from_location_triplet(values, source_label):
    """Infer latitude from either lat/lon/alt degrees or ECEF xyz meters."""
    arr = np.asarray(values, dtype=float).ravel()
    if arr.size < 3 or not np.all(np.isfinite(arr[:3])):
        raise ValueError(f"Could not parse finite 3-vector from {source_label}.")

    x0, x1, x2 = arr[:3]
    if abs(x0) <= 90.0 and abs(x1) <= 360.0 and abs(x2) <= 1.0e5:
        return float(x0), f"{source_label} interpreted as lat/lon/alt deg"

    try:
        from astropy.coordinates import EarthLocation
        import astropy.units as u
        loc = EarthLocation.from_geocentric(x0 * u.m, x1 * u.m, x2 * u.m)
        return float(loc.lat.deg), f"{source_label} interpreted as ECEF xyz m"
    except Exception:
        radius_xy = np.hypot(x0, x1)
        if radius_xy <= 0:
            raise
        # Last-resort geocentric latitude. This is close, but not a full WGS84 geodetic conversion.
        return float(np.degrees(np.arctan2(x2, radius_xy))), f"{source_label} interpreted as geocentric xyz m"


def infer_instrument_latitude_deg(instrument=None):
    """Infer observatory latitude from a UVPSpec/UVData-like object, then fall back to HERA."""
    if instrument is not None:
        for attr in ("telescope_location_lat_lon_alt_degrees", "telescope_location_lat_lon_alt", "telescope_location"):
            if not hasattr(instrument, attr):
                continue
            values = getattr(instrument, attr)
            if values is None:
                continue
            try:
                if attr == "telescope_location_lat_lon_alt":
                    arr = np.asarray(values, dtype=float).ravel()
                    if arr.size >= 2 and abs(arr[0]) <= np.pi and abs(arr[1]) <= 2.0 * np.pi:
                        return float(np.degrees(arr[0])), f"instrument.{attr} interpreted as rad"
                return _latitude_deg_from_location_triplet(values, f"instrument.{attr}")
            except Exception:
                pass

    return _default_hera_latitude_deg()


def latitude_projection(latitude_deg=None, latitude_rad=None, instrument=None,
                        width_rad=None, width_deg=None, width_lst_hr=None):
    """
    Latitude projection for a constant-physical-width arc on a latitude circle.

    The circle at observatory latitude phi has circumference smaller than the
    equator by cos(phi). A fixed fringe spacing therefore maps to an effective
    24-hour sweep width larger by 1 / |cos(phi)|. Optional width inputs are
    returned after applying this scale.
    """
    if latitude_rad is not None:
        phi_rad = float(latitude_rad)
        latitude_deg = float(np.degrees(phi_rad))
        source = "latitude_rad input"
    elif latitude_deg is not None:
        latitude_deg = float(latitude_deg)
        phi_rad = float(np.radians(latitude_deg))
        source = "latitude_deg input"
    else:
        latitude_deg, source = infer_instrument_latitude_deg(instrument)
        phi_rad = float(np.radians(latitude_deg))

    cos_phi = abs(float(np.cos(phi_rad)))
    if not np.isfinite(cos_phi) or cos_phi <= 0:
        raise ValueError("Latitude projection is singular at the pole, where cos(phi)=0.")

    scale = 1.0 / cos_phi
    out = {
        "latitude_deg": latitude_deg,
        "latitude_rad": phi_rad,
        "cos_phi": cos_phi,
        "width_scale": scale,
        "source": source,
    }
    if width_rad is not None:
        out["width_phi_rad"] = np.asarray(width_rad, dtype=float) * scale
    if width_deg is not None:
        out["width_phi_deg"] = np.asarray(width_deg, dtype=float) * scale
    if width_lst_hr is not None:
        out["width_phi_lst_hr"] = np.asarray(width_lst_hr, dtype=float) * scale
    return out


def degrees_to_lst_hours(angle_deg):
    """Convert sky angle in degrees to equivalent LST hours using 24 hr = 360 deg."""
    return np.asarray(angle_deg, dtype=float) / DEG_PER_LST_HOUR


def radians_to_lst_hours(angle_rad):
    """Convert sky angle in radians to equivalent LST hours using 24 hr = 2 pi rad."""
    return np.asarray(angle_rad, dtype=float) * 12.0 / np.pi


def _wavelength_from_frequency_or_lambda(frequency_MHz=None, wavelength_m=None):
    if wavelength_m is not None:
        wavelength_m = float(wavelength_m)
        if wavelength_m <= 0:
            raise ValueError("wavelength_m must be positive.")
        frequency_MHz = C_M_PER_S / wavelength_m / 1e6
        return wavelength_m, frequency_MHz

    if frequency_MHz is None:
        raise ValueError("Provide frequency_MHz or wavelength_m; baseline length alone is insufficient.")

    frequency_MHz = float(frequency_MHz)
    if frequency_MHz <= 0:
        raise ValueError("frequency_MHz must be positive.")
    wavelength_m = C_M_PER_S / (frequency_MHz * 1e6)
    return wavelength_m, frequency_MHz


def _safe_arcsin_ratio(ratio):
    ratio = np.asarray(ratio, dtype=float)
    out = np.full_like(ratio, np.nan, dtype=float)
    ok = np.isfinite(ratio) & (np.abs(ratio) <= 1.0)
    out[ok] = np.arcsin(ratio[ok])
    if out.ndim == 0:
        return float(out)
    return out


def fringe_phase(theta_rad, b_m, frequency_MHz=None, wavelength_m=None):
    """RIME geometric fringe phase phi(theta) = 2 pi b sin(theta) / lambda."""
    wavelength_m, _ = _wavelength_from_frequency_or_lambda(
        frequency_MHz=frequency_MHz,
        wavelength_m=wavelength_m,
    )
    return 2.0 * np.pi * float(b_m) * np.sin(theta_rad) / wavelength_m


def fringe_widths(b_m, frequency_MHz=None, wavelength_m=None,
                  latitude_deg=None, latitude_rad=None, instrument=None):
    """
    Return peak-to-peak fringe spacing and central-lobe FWHM values.

    Widths are returned in radians, degrees, and equivalent LST hours.
    The exact expressions keep sin(theta); the small-angle values use sin(theta) ~= theta.

    The additional fringe_phi_spacing entry applies the latitude-circle projection:

        width_phi = width / |cos(phi)|

    where phi is the observatory latitude inferred from instrument coordinates
    when available, or HERA's latitude otherwise.
    """
    b_m = float(b_m)
    if b_m <= 0:
        raise ValueError("b_m must be positive.")

    wavelength_m, frequency_MHz = _wavelength_from_frequency_or_lambda(
        frequency_MHz=frequency_MHz,
        wavelength_m=wavelength_m,
    )

    projection = latitude_projection(
        latitude_deg=latitude_deg,
        latitude_rad=latitude_rad,
        instrument=instrument,
    )
    phi_scale = projection["width_scale"]

    spacing_rad = _safe_arcsin_ratio(wavelength_m / b_m)
    spacing_small_rad = wavelength_m / b_m
    amp_fwhm_rad = 2.0 * _safe_arcsin_ratio(wavelength_m / (6.0 * b_m))
    power_fwhm_rad = 2.0 * _safe_arcsin_ratio(wavelength_m / (8.0 * b_m))
    null_to_null_rad = 2.0 * _safe_arcsin_ratio(wavelength_m / (4.0 * b_m))
    fringe_phi_spacing_rad = spacing_rad * phi_scale
    fringe_phi_spacing_small_rad = spacing_small_rad * phi_scale

    def bundle(angle_rad):
        return {
            "rad": angle_rad,
            "deg": np.degrees(angle_rad),
            "lst_hr": float(radians_to_lst_hours(angle_rad)),
        }

    return {
        "baseline_m": b_m,
        "frequency_MHz": frequency_MHz,
        "wavelength_m": wavelength_m,
        "lambda_over_b": wavelength_m / b_m,
        "latitude_projection": projection,
        "fringe_spacing": bundle(spacing_rad),
        "fringe_spacing_small_angle": bundle(spacing_small_rad),
        "fringe_phi_spacing": bundle(fringe_phi_spacing_rad),
        "fringe_phi_spacing_small_angle": bundle(fringe_phi_spacing_small_rad),
        "amp_fwhm": bundle(amp_fwhm_rad),
        "power_fwhm": bundle(power_fwhm_rad),
        "null_to_null": bundle(null_to_null_rad),
    }


def fringe_fwhm(b_m, frequency_MHz=None, wavelength_m=None, response="amplitude"):
    """Backward-compatible FWHM helper returning radians, degrees, and LST hours."""
    widths = fringe_widths(b_m, frequency_MHz=frequency_MHz, wavelength_m=wavelength_m)
    response = response.lower()
    if response in ("amplitude", "field", "cos"):
        values = widths["amp_fwhm"]
        response_name = "amplitude"
    elif response in ("power", "half-power", "cos2", "cos^2"):
        values = widths["power_fwhm"]
        response_name = "power"
    else:
        raise ValueError("response must be 'amplitude' or 'power'.")

    return {
        "response": response_name,
        "baseline_m": widths["baseline_m"],
        "frequency_MHz": widths["frequency_MHz"],
        "wavelength_m": widths["wavelength_m"],
        "fwhm_rad": values["rad"],
        "fwhm_deg": values["deg"],
        "fwhm_lst_hr": values["lst_hr"],
    }


def fringe_fwhm_deg(b_m, frequency_MHz=None, wavelength_m=None, response="amplitude"):
    """Return only FWHM in degrees."""
    return fringe_fwhm(b_m, frequency_MHz=frequency_MHz, wavelength_m=wavelength_m, response=response)["fwhm_deg"]


def fringe_fwhm_lst_hr(b_m, frequency_MHz=None, wavelength_m=None, response="amplitude"):
    """Return only FWHM in equivalent LST hours."""
    return fringe_fwhm(b_m, frequency_MHz=frequency_MHz, wavelength_m=wavelength_m, response=response)["fwhm_lst_hr"]


def print_fringe_width_table(b_m, frequency_MHz_list, latitude_deg=None, latitude_rad=None, instrument=None):
    """Print a compact table of fringe spacing, latitude-projected spacing, and FWHM quantities."""
    projection = latitude_projection(latitude_deg=latitude_deg, latitude_rad=latitude_rad, instrument=instrument)
    header = (
        "freq [MHz]  lambda [m]  spacing [deg/hr]  "
        "fringe(phi) [deg/hr]  amp FWHM [deg/hr]  power FWHM [deg/hr]"
    )
    print(
        f"Latitude projection: phi={projection['latitude_deg']:.4f} deg, "
        f"cos(phi)={projection['cos_phi']:.4f}, width scale=1/cos(phi)={projection['width_scale']:.4f} "
        f"[{projection['source']}]"
    )
    print(header)
    print("-" * len(header))
    for freq in frequency_MHz_list:
        w = fringe_widths(
            b_m,
            frequency_MHz=freq,
            latitude_deg=latitude_deg,
            latitude_rad=latitude_rad,
            instrument=instrument,
        )
        print(
            f"{freq:10.3f}  "
            f"{w['wavelength_m']:10.3f}  "
            f"{w['fringe_spacing']['deg']:7.3f}/{w['fringe_spacing']['lst_hr']:6.3f}  "
            f"{w['fringe_phi_spacing']['deg']:7.3f}/{w['fringe_phi_spacing']['lst_hr']:6.3f}  "
            f"{w['amp_fwhm']['deg']:7.3f}/{w['amp_fwhm']['lst_hr']:6.3f}  "
            f"{w['power_fwhm']['deg']:7.3f}/{w['power_fwhm']['lst_hr']:6.3f}"
        )


def plot_fringe_concepts_vs_phi():
    """Plot cos(phi) and cos^2(phi), showing phase-domain spacing and FWHM."""
    phi = np.linspace(-2.4 * np.pi, 2.4 * np.pi, 2000)
    amp = np.cos(phi)
    power = amp ** 2

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)

    ax = axes[0]
    ax.plot(phi / np.pi, amp, color="crimson", lw=1.8, label=r"$\cos\phi$")
    ax.axhline(0.5, color="black", ls="--", lw=1, alpha=0.7, label="half amplitude")
    ax.axvspan(-1/3, 1/3, color="crimson", alpha=0.12, label=r"amp FWHM: $2\pi/3$")
    for x in (-2, 0, 2):
        ax.axvline(x, color="0.25", lw=0.8, alpha=0.5)
    ax.annotate(
        "one fringe spacing",
        xy=(0, 1.05), xytext=(2, 1.05),
        arrowprops=dict(arrowstyle="<->", color="0.25"),
        ha="center", va="bottom", fontsize=9,
    )
    ax.set_xlabel(r"phase $\phi / \pi$")
    ax.set_ylabel("real fringe amplitude")
    ax.set_title(r"Amplitude fringe: $\cos\phi$")
    ax.set_ylim(-1.15, 1.2)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)

    ax = axes[1]
    ax.plot(phi / np.pi, power, color="royalblue", lw=1.8, label=r"$\cos^2\phi$")
    ax.axhline(0.5, color="black", ls="--", lw=1, alpha=0.7, label="half power")
    ax.axvspan(-1/4, 1/4, color="royalblue", alpha=0.12, label=r"power FWHM: $\pi/2$")
    for x in (-2, 0, 2):
        ax.axvline(x, color="0.25", lw=0.8, alpha=0.5)
    ax.annotate(
        "one amplitude-fringe spacing",
        xy=(0, 1.05), xytext=(2, 1.05),
        arrowprops=dict(arrowstyle="<->", color="0.25"),
        ha="center", va="bottom", fontsize=9,
    )
    ax.set_xlabel(r"phase $\phi / \pi$")
    ax.set_ylabel("power fringe")
    ax.set_title(r"Power fringe: $\cos^2\phi$")
    ax.set_ylim(-0.05, 1.2)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)

    plt.show()


def plot_fringe_overlays_by_frequency(b_m, frequency_MHz_list, theta_limit_deg=None):
    """Plot real and power fringes versus angular offset for several frequencies."""
    frequency_MHz_list = np.asarray(frequency_MHz_list, dtype=float)
    if theta_limit_deg is None:
        widths = [fringe_widths(b_m, frequency_MHz=f)["fringe_spacing"]["deg"] for f in frequency_MHz_list]
        finite_widths = np.asarray(widths, dtype=float)[np.isfinite(widths)]
        theta_limit_deg = min(45.0, max(5.0, 1.6 * np.nanmax(finite_widths))) if finite_widths.size else 20.0

    theta_deg = np.linspace(-theta_limit_deg, theta_limit_deg, 2400)
    theta_rad = np.radians(theta_deg)

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(frequency_MHz_list)))

    for color, freq in zip(cmap, frequency_MHz_list):
        phi = fringe_phase(theta_rad, b_m, frequency_MHz=freq)
        wavelength_m = C_M_PER_S / (freq * 1e6)
        label = f"{freq:.1f} MHz, lambda={wavelength_m:.2f} m"
        axes[0].plot(theta_deg, np.cos(phi), color=color, lw=1.3, label=label)
        axes[1].plot(theta_deg, np.cos(phi) ** 2, color=color, lw=1.3, label=label)

        w = fringe_widths(b_m, frequency_MHz=freq)
        half_amp = 0.5 * w["amp_fwhm"]["deg"]
        half_power = 0.5 * w["power_fwhm"]["deg"]
        if np.isfinite(half_amp):
            axes[0].axvspan(-half_amp, half_amp, color=color, alpha=0.05, linewidth=0)
        if np.isfinite(half_power):
            axes[1].axvspan(-half_power, half_power, color=color, alpha=0.05, linewidth=0)

    axes[0].axhline(0.5, color="black", ls="--", lw=0.9, alpha=0.65)
    axes[1].axhline(0.5, color="black", ls="--", lw=0.9, alpha=0.65)
    axes[0].set_ylabel(r"$\cos\phi(\theta)$")
    axes[1].set_ylabel(r"$\cos^2\phi(\theta)$")
    axes[1].set_xlabel("angular offset along baseline [deg]")

    def deg_to_lst_hr(x):
        return np.asarray(x) / DEG_PER_LST_HOUR

    def lst_hr_to_deg(x):
        return np.asarray(x) * DEG_PER_LST_HOUR

    secax = axes[0].secondary_xaxis("top", functions=(deg_to_lst_hr, lst_hr_to_deg))
    secax.set_xlabel("equivalent LST offset [hr]")

    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(loc="upper right", fontsize=8)
    axes[0].set_title(f"Fringe overlays for projected baseline b={b_m:.2f} m")
    plt.show()


def plot_fringe_widths_vs_frequency(b_m, frequency_MHz_grid, reference_frequency_MHz=None,
                                    latitude_deg=None, latitude_rad=None, instrument=None):
    """
    Plot fringe spacing, amplitude FWHM, and power FWHM versus frequency.

    ROBUST_SINGLE_FREQUENCY_SWEEP: if the input grid is one frequency, or many
    copies of the same frequency, expand it to a local sweep so curves do not
    collapse to invisible zero-length line segments.
    """
    input_frequency_MHz = np.asarray(frequency_MHz_grid, dtype=float).ravel()
    input_frequency_MHz = input_frequency_MHz[np.isfinite(input_frequency_MHz) & (input_frequency_MHz > 0)]
    if input_frequency_MHz.size == 0:
        raise ValueError("frequency_MHz_grid must contain at least one positive finite frequency.")

    unique_frequency_MHz = np.unique(np.round(input_frequency_MHz, decimals=9))
    f_min = np.nanmin(input_frequency_MHz)
    f_max = np.nanmax(input_frequency_MHz)
    expanded_single_frequency = unique_frequency_MHz.size == 1 or np.isclose(f_min, f_max)

    if expanded_single_frequency:
        center = float(np.nanmedian(input_frequency_MHz))
        half_width = max(5.0, 0.10 * center)
        f_min = max(center - half_width, 1e-6)
        f_max = center + half_width
        plot_frequency_MHz = np.linspace(f_min, f_max, 240)
    else:
        plot_frequency_MHz = np.linspace(f_min, f_max, 240)

    spacing_deg = np.full(plot_frequency_MHz.size, np.nan)
    amp_deg = np.full(plot_frequency_MHz.size, np.nan)
    power_deg = np.full(plot_frequency_MHz.size, np.nan)
    phi_spacing_deg = np.full(plot_frequency_MHz.size, np.nan)

    for i, freq in enumerate(plot_frequency_MHz):
        w = fringe_widths(
            b_m,
            frequency_MHz=freq,
            latitude_deg=latitude_deg,
            latitude_rad=latitude_rad,
            instrument=instrument,
        )
        spacing_deg[i] = w["fringe_spacing"]["deg"]
        phi_spacing_deg[i] = w["fringe_phi_spacing"]["deg"]
        amp_deg[i] = w["amp_fwhm"]["deg"]
        power_deg[i] = w["power_fwhm"]["deg"]

    finite = (
        np.isfinite(plot_frequency_MHz)
        & np.isfinite(spacing_deg)
        & np.isfinite(phi_spacing_deg)
        & np.isfinite(amp_deg)
        & np.isfinite(power_deg)
    )
    print(
        "Width plot frequency range: "
        f"{plot_frequency_MHz[0]:.3f}-{plot_frequency_MHz[-1]:.3f} MHz; "
        f"finite points: {np.count_nonzero(finite)}/{plot_frequency_MHz.size}"
    )
    print(
        "Width plot y-ranges [deg]: "
        f"spacing {np.nanmin(spacing_deg):.3f}-{np.nanmax(spacing_deg):.3f}, "
        f"fringe(phi) {np.nanmin(phi_spacing_deg):.3f}-{np.nanmax(phi_spacing_deg):.3f}, "
        f"amp {np.nanmin(amp_deg):.3f}-{np.nanmax(amp_deg):.3f}, "
        f"power {np.nanmin(power_deg):.3f}-{np.nanmax(power_deg):.3f}"
    )
    if expanded_single_frequency:
        print("Input frequencies collapsed to one value; plotting a local +/-10% sweep around it.")

    fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
    marker_every = max(1, plot_frequency_MHz.size // 24)
    ax.plot(
        plot_frequency_MHz,
        spacing_deg,
        lw=2,
        marker="o",
        markevery=marker_every,
        ms=3,
        color="black",
        label=r"spacing $\approx \lambda/b$",
    )
    ax.plot(
        plot_frequency_MHz,
        phi_spacing_deg,
        lw=2,
        marker="D",
        markevery=marker_every,
        ms=3,
        color="darkorange",
        label=r"latitude-projected spacing $\lambda/[b\cos\phi]$",
    )
    ax.plot(
        plot_frequency_MHz,
        amp_deg,
        lw=2,
        marker="s",
        markevery=marker_every,
        ms=3,
        color="crimson",
        label="amplitude FWHM",
    )
    ax.plot(
        plot_frequency_MHz,
        power_deg,
        lw=2,
        marker="^",
        markevery=marker_every,
        ms=3,
        color="royalblue",
        label="power FWHM",
    )

    if reference_frequency_MHz is None:
        reference_frequency_MHz = unique_frequency_MHz
    reference_frequency_MHz = np.asarray(reference_frequency_MHz, dtype=float).ravel()
    reference_frequency_MHz = np.unique(reference_frequency_MHz[np.isfinite(reference_frequency_MHz) & (reference_frequency_MHz > 0)])
    for i, freq in enumerate(reference_frequency_MHz):
        label = "input/SPW frequency" if i == 0 else None
        ax.axvline(freq, color="0.25", ls=":", lw=1.1, alpha=0.65, label=label)

    ax.set_xlabel("frequency [MHz]")
    ax.set_ylabel("angular width [deg]")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")

    secax = ax.secondary_yaxis(
        "right",
        functions=(degrees_to_lst_hours, lambda h: np.asarray(h) * DEG_PER_LST_HOUR),
    )
    secax.set_ylabel("equivalent LST width [hr]")
    ax.set_title(f"Fringe widths versus frequency for projected baseline b={b_m:.2f} m")
    if expanded_single_frequency:
        ax.text(
            0.02,
            0.02,
            "single frequency input: showing local +/-10% sweep",
            transform=ax.transAxes,
            fontsize=8,
            color="0.25",
            ha="left",
            va="bottom",
        )
    plt.show()

# Example: infer a baseline and representative frequencies from the current UVPSpec dictionary.
try:
    example_uvp = next(iter(combined_uvp_dict.values()))
    example_instrument = example_uvp
    example_baseline_m = np.linalg.norm(example_uvp.bl_vecs[0])
    spw_centers_MHz = np.array([
        0.5 * (freq_range[0] + freq_range[1]) / 1e6
        for _, freq_range in sorted(get_spw_info(example_uvp).items(), key=lambda item: item[0])
        if freq_range is not None
    ])
except NameError:
    example_uvp = None
    example_instrument = None
    example_baseline_m = 29.216
    spw_centers_MHz = np.array([80.0, 120.0, 160.0, 200.0])

if spw_centers_MHz.size == 0:
    spw_centers_MHz = np.array([80.0, 120.0, 160.0, 200.0])

# Use a few representative frequencies for the overlay plot, and all available centers for the width trend.
overlay_indices = np.unique(np.linspace(0, spw_centers_MHz.size - 1, min(5, spw_centers_MHz.size)).round().astype(int))
overlay_frequency_MHz = spw_centers_MHz[overlay_indices]
frequency_grid_MHz = spw_centers_MHz

print(f"Example projected baseline: {example_baseline_m:.3f} m")
print_fringe_width_table(example_baseline_m, overlay_frequency_MHz, instrument=example_instrument)

plot_fringe_concepts_vs_phi()
plot_fringe_overlays_by_frequency(example_baseline_m, overlay_frequency_MHz)
plot_fringe_widths_vs_frequency(
    example_baseline_m,
    frequency_grid_MHz,
    reference_frequency_MHz=spw_centers_MHz,
    instrument=example_instrument,
)

# To use your own values:
# fringe_widths(14.6, frequency_MHz=150.0)  # uses HERA latitude fallback if no instrument is supplied
# fringe_fwhm_lst_hr(14.6, frequency_MHz=150.0, response="amplitude")
# plot_fringe_overlays_by_frequency(14.6, [100.0, 150.0, 200.0])


In [ ]:
# Blackman-Harris bandstop PSPEC using spectral_information_filter() feature selection

import numpy as np
import matplotlib.pyplot as plt

BANDSTOP_GROUP_KEY = None   # None -> use the last key in combined_uvp_dict, matching the FFT cell default
BANDSTOP_POL = "xx"
BANDSTOP_SPW = None         # None -> use the first valid SPW in the selected UVPSpec
BANDSTOP_MAX_FREQUENCY = 6.0
BANDSTOP_SCORE_THRESHOLD = 2.5
BANDSTOP_SKIP_DATA_SPAN_MODE = True

# User-facing width/depth controls for the Blackman-Harris well.
# Set *_CYC_PER_HR to a number to override the automatic bin/continuum-based width.
BANDSTOP_STOP_FLOOR = 0.05              # 0 -> full stop; 0.05 -> leave 5% of the mode
BANDSTOP_FLAT_HALF_WIDTH_CYC_PER_HR = None
BANDSTOP_TAPER_WIDTH_CYC_PER_HR = None
BANDSTOP_FLAT_HALF_WIDTH_BINS = 1.0    # flat-bottom half-width in FFT bins if no explicit width is given
BANDSTOP_TAPER_WIDTH_BINS = 2.0        # Blackman-Harris shoulder width in FFT bins if no explicit width is given
BANDSTOP_FLAT_FRACTION_OF_CONTINUUM_WIDTH = 0.90


def _require_bandstop_dependencies():
    required = [
        "get_lst_pspec_for_fft",
        "_uniform_lst_grid",
        "lst_fft_spectrum",
        "spectral_information_filter",
        "get_spw_info",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Run the earlier LST Fourier spectrum cell first. Missing: " + ", ".join(missing))


def _select_bandstop_context(combined_uvp_dict, combined_uvp_avg_dict, group_key=None, pol="xx", spw=None):
    if group_key is None:
        group_key = list(combined_uvp_dict.keys())[-1]

    uvp = combined_uvp_dict[group_key]
    uvp_avg = combined_uvp_avg_dict[group_key]
    spw_ranges = get_spw_info(uvp)

    if spw is None:
        valid_spws = [key for key, val in sorted(spw_ranges.items(), key=lambda item: item[0]) if val is not None]
        if not valid_spws:
            raise ValueError("No valid SPW frequency ranges found for bandstop filtering.")
        spw = valid_spws[0]

    if spw_ranges.get(spw) is None:
        center_frequency_MHz = np.nan
        freq_range_str = "N/A"
    else:
        freq_min, freq_max = spw_ranges[spw]
        center_frequency_MHz = 0.5 * (freq_min + freq_max) / 1e6
        freq_range_str = f"{freq_min / 1e6:.2f}-{freq_max / 1e6:.2f} MHz"

    return uvp, uvp_avg, group_key, pol, spw, center_frequency_MHz, freq_range_str


def _blackman_harris_smoothstep(q):
    """
    Monotonic 0->1 edge made from the integral of a 4-term Blackman-Harris window.
    """
    q = np.clip(np.asarray(q, dtype=float), 0.0, 1.0)
    a0, a1, a2, a3 = 0.35875, 0.48829, 0.14128, 0.01168
    integral = (
        a0 * q
        - a1 * np.sin(2.0 * np.pi * q) / (2.0 * np.pi)
        + a2 * np.sin(4.0 * np.pi * q) / (4.0 * np.pi)
        - a3 * np.sin(6.0 * np.pi * q) / (6.0 * np.pi)
    )
    return np.clip(integral / a0, 0.0, 1.0)


def blackman_harris_bandstop(freq, wells, stop_floor=0.0):
    """
    Flat-bottom Blackman-Harris-edged bandstop transfer H(f).

    Each well dict needs center, flat_half_width, and taper_width in cycles/hour.
    """
    freq = np.asarray(freq, dtype=float)
    transfer = np.ones_like(freq)
    stop_floor = float(stop_floor)
    depth = 1.0 - stop_floor

    for well in wells:
        center = float(well["center"])
        flat_half = max(0.0, float(well["flat_half_width"]))
        taper = max(0.0, float(well["taper_width"]))

        flat_min = center - flat_half
        flat_max = center + flat_half
        outer_min = flat_min - taper
        outer_max = flat_max + taper

        if outer_max <= 0 or outer_min >= np.nanmax(freq):
            continue

        flat = (freq >= flat_min) & (freq <= flat_max)
        transfer[flat] = np.minimum(transfer[flat], stop_floor)

        if taper > 0:
            left = (freq >= outer_min) & (freq < flat_min)
            q_left = (freq[left] - outer_min) / taper
            transfer[left] = np.minimum(transfer[left], 1.0 - depth * _blackman_harris_smoothstep(q_left))

            right = (freq > flat_max) & (freq <= outer_max)
            q_right = (freq[right] - flat_max) / taper
            transfer[right] = np.minimum(transfer[right], stop_floor + depth * _blackman_harris_smoothstep(q_right))

    return transfer


def _fft_df(freq):
    positive = np.asarray(freq)[np.asarray(freq) > 0]
    return np.nanmedian(np.diff(positive)) if positive.size > 1 else 0.0


def _snap_wells_to_fft_grid(freq, wells):
    """Snap each well center to the nearest actual FFT bin and keep a sensible width."""
    freq = np.asarray(freq, dtype=float)
    df = _fft_df(freq)
    snapped = []
    for well in wells:
        center = float(well["center"])
        if not np.isfinite(center) or center < 0 or center > np.nanmax(freq):
            continue
        snapped_center = freq[int(np.nanargmin(np.abs(freq - center)))]
        flat_half = max(float(well["flat_half_width"]), 0.5 * df)
        taper = max(float(well["taper_width"]), df)
        snapped.append({
            **well,
            "requested_center": center,
            "center": snapped_center,
            "flat_half_width": flat_half,
            "taper_width": taper,
            "df": df,
            "flat_min": max(0.0, snapped_center - flat_half),
            "flat_max": snapped_center + flat_half,
            "outer_min": max(0.0, snapped_center - flat_half - taper),
            "outer_max": snapped_center + flat_half + taper,
        })
    return snapped, df


def _fractional_fft_from_grid(y_grid, dt_hours):
    scale = np.nanmedian(np.abs(y_grid))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    x = y_grid / scale - 1.0
    x_mean = np.nanmean(x)
    x = np.nan_to_num(x - x_mean, nan=0.0, posinf=0.0, neginf=0.0)
    fft = np.fft.rfft(x)
    freq = np.fft.rfftfreq(x.size, d=dt_hours)
    amp = np.abs(fft) / x.size
    if x.size % 2 == 0 and amp.size > 2:
        amp[1:-1] *= 2.0
    elif x.size % 2 == 1 and amp.size > 1:
        amp[1:] *= 2.0
    return freq, fft, amp, scale, x_mean


def apply_lst_bh_bandstop(lst_hours, values, wells, stop_floor=0.0):
    """Apply a Blackman-Harris bandstop well in LST Fourier space."""
    lst_grid, y_grid, dt_hours = _uniform_lst_grid(lst_hours, values)
    freq, fft, amp, scale, x_mean = _fractional_fft_from_grid(y_grid, dt_hours)
    snapped_wells, df = _snap_wells_to_fft_grid(freq, wells)
    transfer = blackman_harris_bandstop(freq, snapped_wells, stop_floor=stop_floor)

    if not snapped_wells:
        print("Warning: requested bandstop lies outside the available LST Fourier grid; no notch applied.")
    else:
        print(
            "Effective Blackman-Harris well(s): "
            + ", ".join(
                f"center={w['center']:.4g}, flat=[{w['flat_min']:.4g}, {w['flat_max']:.4g}], "
                f"outer=[{w['outer_min']:.4g}, {w['outer_max']:.4g}] cyc/hr"
                for w in snapped_wells
            )
            + f"; transfer min={np.nanmin(transfer):.3g}"
        )

    filtered_x = np.fft.irfft(fft * transfer, n=y_grid.size)
    filtered = scale * (filtered_x + x_mean + 1.0)

    return {
        "lst_grid": lst_grid,
        "original_grid": y_grid,
        "filtered": filtered,
        "freq": freq,
        "fft": fft,
        "amp": amp,
        "transfer": transfer,
        "applied_fft_amp": amp * transfer,
        "dt_hours": dt_hours,
        "df": df,
        "wells": snapped_wells,
        "requested_wells": wells,
    }


def _candidate_overlaps_frequency(feature, target_freq, df):
    f0 = feature.get("frequency_cyc_per_hr", np.nan)
    f_min = feature.get("frequency_min_cyc_per_hr", np.nan)
    f_max = feature.get("frequency_max_cyc_per_hr", np.nan)
    tol = max(df, 0.20 * target_freq)
    if np.isfinite(f_min) and np.isfinite(f_max) and f_min <= target_freq <= f_max:
        return True
    return np.isfinite(f0) and abs(f0 - target_freq) <= tol


def _continuum_intersection_width(freq, amp, continuum, f0):
    """Find where a feature peak falls back to the smooth continuum."""
    freq = np.asarray(freq, dtype=float)
    amp = np.asarray(amp, dtype=float)
    continuum = np.asarray(continuum, dtype=float)
    ratio = amp / continuum
    idx = int(np.nanargmin(np.abs(freq - f0)))

    left = idx
    while left > 1 and np.isfinite(ratio[left]) and ratio[left] > 1.0:
        left -= 1

    right = idx
    while right < ratio.size - 2 and np.isfinite(ratio[right]) and ratio[right] > 1.0:
        right += 1

    if left == idx or right == idx:
        return None
    return freq[left], freq[right]


def _well_from_center_and_widths(center, df, flat_half_width=None, taper_width=None,
                                 continuum_bounds=None):
    if continuum_bounds is not None:
        left, right = continuum_bounds
        outer_half = max(abs(center - left), abs(right - center))
    else:
        outer_half = 0.0

    if flat_half_width is None:
        flat_half_width = max(
            BANDSTOP_FLAT_HALF_WIDTH_BINS * df,
            BANDSTOP_FLAT_FRACTION_OF_CONTINUUM_WIDTH * outer_half,
        )
    if taper_width is None:
        taper_width = max(
            BANDSTOP_TAPER_WIDTH_BINS * df,
            outer_half - flat_half_width,
        )

    return {
        "center": center,
        "flat_half_width": flat_half_width,
        "taper_width": taper_width,
        "continuum_bounds": continuum_bounds,
    }


def _strongest_spectral_information_well(lst_array_roll, uvpspec_averaged_power, uvp_power,
                                         max_frequency=6.0, score_threshold=2.5,
                                         skip_data_span_mode=True):
    """
    Select the strongest non-span feature and create a Blackman-Harris well.

    The automatic taper starts from the intersection of the feature with the
    smooth FFT continuum when that intersection can be found.
    """
    candidates = []
    for label, values in [
        ("uvpspec_averaged_power", uvpspec_averaged_power),
        ("uvp_power", uvp_power),
    ]:
        freq, amp, _, _ = lst_fft_spectrum(lst_array_roll, values, normalize="fractional", window="hann")
        result = spectral_information_filter(
            freq,
            amp,
            n_features=8,
            max_frequency=max_frequency,
            score_threshold=score_threshold,
            continuum_window=15,
            continuum_percentile=40,
        )
        for feature in result["features"]:
            candidates.append((label, values, feature, freq, amp, result["continuum"]))

    if not candidates:
        raise RuntimeError("spectral_information_filter() did not return any candidate features.")

    candidates = sorted(
        candidates,
        key=lambda item: item[2].get("region_score", item[2].get("score", -np.inf)),
        reverse=True,
    )

    lst_span_hr = np.nanmax(lst_array_roll) - np.nanmin(lst_array_roll)
    data_span_mode = 1.0 / lst_span_hr if np.isfinite(lst_span_hr) and lst_span_hr > 0 else np.nan

    skipped = []
    chosen = None
    for label, values, feature, freq, amp, continuum in candidates:
        df = _fft_df(freq)
        if skip_data_span_mode and np.isfinite(data_span_mode) and _candidate_overlaps_frequency(feature, data_span_mode, df):
            skipped.append((label, feature))
            continue
        chosen = (label, values, feature, freq, amp, continuum, df)
        break

    if chosen is None:
        label, values, feature, freq, amp, continuum = candidates[min(1, len(candidates) - 1)][:6]
        df = _fft_df(freq)
        chosen = (label, values, feature, freq, amp, continuum, df)

    label, values, feature, freq, amp, continuum, df = chosen
    center = feature["frequency_cyc_per_hr"]
    continuum_bounds = _continuum_intersection_width(freq, amp, continuum, center)

    flat_override = BANDSTOP_FLAT_HALF_WIDTH_CYC_PER_HR
    taper_override = BANDSTOP_TAPER_WIDTH_CYC_PER_HR
    well = _well_from_center_and_widths(
        center,
        df,
        flat_half_width=flat_override,
        taper_width=taper_override,
        continuum_bounds=continuum_bounds,
    )

    return {
        "source_label": label,
        "source_values": values,
        "feature": feature,
        "wells": [well],
        "df": df,
        "data_span_mode_cyc_per_hr": data_span_mode,
        "skipped_data_span_features": skipped,
        "continuum_bounds": continuum_bounds,
    }


def _diagnostic_fft_for_plot(lst_hours, values):
    """Use the same FFT display convention as plot_lst_fourier_spectrum()."""
    freq, amp, _, _ = lst_fft_spectrum(
        lst_hours,
        values,
        normalize="fractional",
        window="hann",
    )
    return freq, amp


def _plot_bandstop_fft_panel(ax, source_label, source_result, plot_max_frequency=6.0):
    """
    Show original PSPEC FFT, H(f)*FFT, post-filter FFT, and H(f).

    The plotted FFT amplitudes intentionally use lst_fft_spectrum(..., window="hann"),
    matching plot_lst_fourier_spectrum(). The actual inverse-filtering step still
    uses the unwindowed FFT, which is the appropriate object to invert.
    """
    freq, amp = _diagnostic_fft_for_plot(source_result["lst_grid"], source_result["original_grid"])
    post_freq, post_amp = _diagnostic_fft_for_plot(source_result["lst_grid"], source_result["filtered"])

    transfer = np.interp(
        freq,
        source_result["freq"],
        source_result["transfer"],
        left=1.0,
        right=1.0,
    )
    applied_amp = amp * transfer

    use = freq > 0
    if plot_max_frequency is not None:
        use &= freq <= plot_max_frequency
    post_use = post_freq > 0
    if plot_max_frequency is not None:
        post_use &= post_freq <= plot_max_frequency

    ax.plot(freq[use], amp[use], color="0.25", linewidth=1.25, label=f"{source_label} FFT")
    ax.plot(freq[use], applied_amp[use], color="darkorange", linewidth=1.35, label="bandstop applied to FFT")
    ax.plot(post_freq[post_use], post_amp[post_use], color="crimson", linewidth=1.1, linestyle="--", label="post-filter PSPEC FFT")

    for i, well in enumerate(source_result["wells"]):
        label = "BH well outer support" if i == 0 else None
        ax.axvspan(well["outer_min"], well["outer_max"], color="orange", alpha=0.12, linewidth=0, zorder=0, label=label)
        ax.axvspan(well["flat_min"], well["flat_max"], color="orange", alpha=0.24, linewidth=0, zorder=0)

    ax_transfer = ax.twinx()
    transfer_freq = source_result["freq"]
    raw_transfer = source_result["transfer"]
    use_transfer = np.isfinite(transfer_freq) & np.isfinite(raw_transfer)
    if plot_max_frequency is not None:
        use_transfer &= transfer_freq <= plot_max_frequency
    ax_transfer.fill_between(transfer_freq[use_transfer], raw_transfer[use_transfer], 1.0, color="black", alpha=0.10, linewidth=0)
    ax_transfer.plot(transfer_freq[use_transfer], raw_transfer[use_transfer], color="black", linewidth=1.4, label="Blackman-Harris bandstop H(f)")
    ax_transfer.set_ylim(-0.05, 1.05)
    ax_transfer.set_ylabel("bandstop transfer")

    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.grid(alpha=0.22, linewidth=0.6)
    if plot_max_frequency is not None:
        ax.set_xlim(0, plot_max_frequency)
    ax.margins(x=0.02)
    ax.set_xlabel("LST Fourier frequency [cycles / hr]")
    ax.set_ylabel("FFT amplitude\n(fractional PSPEC)")

    handles, labels = ax.get_legend_handles_labels()
    t_handles, t_labels = ax_transfer.get_legend_handles_labels()
    ax.legend(handles + t_handles, labels + t_labels, loc="best", frameon=True, borderpad=0.35, handlelength=1.8, fontsize=8)


def _plot_bandstop_filtered_pspec(lst_array_roll, uvpspec_averaged_power, uvp_power,
                                  avg_filtered_result, uvp_filtered_result, source_label,
                                  title, annotation, plot_max_frequency=6.0):
    fig, axes = plt.subplots(
        2, 1, figsize=(11, 7.8), constrained_layout=True,
        gridspec_kw={"height_ratios": [2.25, 1.35]},
    )
    ax = axes[0]
    ax.scatter(lst_array_roll, uvpspec_averaged_power, s=0.5, color="orange", alpha=0.1,
               label="uvpspec_averaged_power original")
    ax.scatter(lst_array_roll, uvp_power, s=0.5, color="green", alpha=0.1,
               label="uvp_power original")
    ax.plot(avg_filtered_result["lst_grid"], avg_filtered_result["filtered"], color="royalblue", lw=1.8,
            label="uvpspec_averaged_power bandstop")
    ax.plot(uvp_filtered_result["lst_grid"], uvp_filtered_result["filtered"], color="crimson", lw=1.8,
            label="uvp_power bandstop")

    all_y = np.concatenate([
        np.asarray(uvpspec_averaged_power, dtype=float),
        np.asarray(uvp_power, dtype=float),
        np.asarray(avg_filtered_result["filtered"], dtype=float),
        np.asarray(uvp_filtered_result["filtered"], dtype=float),
    ])
    if np.nanmin(all_y) > 0:
        ax.set_yscale("log")
    else:
        ax.set_yscale("symlog", linthresh=max(1.0, 0.01 * np.nanmedian(np.abs(all_y))))

    ax.set_xlabel("LST (Hours)")
    ax.set_ylabel(r"PSPEC [mK$^2$]")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)
    # ax.text(0.02, 0.02, annotation, transform=ax.transAxes, fontsize=8, va="bottom")
    ax.set_ylim(1e1, 1e7)

    source_result = uvp_filtered_result if source_label == "uvp_power" else avg_filtered_result
    _plot_bandstop_fft_panel(axes[1], source_label, source_result, plot_max_frequency=plot_max_frequency)
    plt.show()


def plot_spectral_information_bandstop_pspec(combined_uvp_dict, combined_uvp_avg_dict,
                                             group_key=None, pol="xx", spw=None,
                                             max_frequency=6.0, score_threshold=2.5,
                                             stop_floor=0.0):
    """Bandstop the strongest non-span spectral_information_filter() LST Fourier feature."""
    _require_bandstop_dependencies()
    uvp, uvp_avg, group_key, pol, spw, center_frequency_MHz, freq_range_str = _select_bandstop_context(
        combined_uvp_dict, combined_uvp_avg_dict, group_key=group_key, pol=pol, spw=spw,
    )
    (_, _, _, _, lst_array_roll, uvp_power, uvpspec_averaged_power) = get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw)

    region = _strongest_spectral_information_well(
        lst_array_roll, uvpspec_averaged_power, uvp_power,
        max_frequency=max_frequency,
        score_threshold=score_threshold,
        skip_data_span_mode=BANDSTOP_SKIP_DATA_SPAN_MODE,
    )

    avg_filtered = apply_lst_bh_bandstop(lst_array_roll, uvpspec_averaged_power, region["wells"], stop_floor=stop_floor)
    uvp_filtered = apply_lst_bh_bandstop(lst_array_roll, uvp_power, region["wells"], stop_floor=stop_floor)

    feature = region["feature"]
    source_result = uvp_filtered if region["source_label"] == "uvp_power" else avg_filtered
    well = source_result["wells"][0]
    bounds_text = "none"
    if region["continuum_bounds"] is not None:
        bounds_text = f"[{region['continuum_bounds'][0]:.3f}, {region['continuum_bounds'][1]:.3f}]"
    annotation = (
        f"span mode={region['data_span_mode_cyc_per_hr']:.3f} cyc/hr skipped; "
        f"source={region['source_label']}; peak={feature['frequency_cyc_per_hr']:.3f} cyc/hr; "
        f"continuum crossings={bounds_text}; "
        f"flat=[{well['flat_min']:.3f}, {well['flat_max']:.3f}], outer=[{well['outer_min']:.3f}, {well['outer_max']:.3f}] cyc/hr"
    )
    title = f"PSPEC after Blackman-Harris spectral bandstop | group={group_key}, SPW {spw} ({freq_range_str}), pol={pol}"
    _plot_bandstop_filtered_pspec(lst_array_roll, uvpspec_averaged_power, uvp_power,
                                  avg_filtered, uvp_filtered, region["source_label"],
                                  title, annotation, plot_max_frequency=max_frequency)
    print(annotation)
    print(f"Skipped {len(region['skipped_data_span_features'])} data-span feature(s).")
    return {
        "lst_array_roll": lst_array_roll,
        "uvpspec_averaged_power": uvpspec_averaged_power,
        "uvp_power": uvp_power,
        "avg_filtered": avg_filtered,
        "uvp_filtered": uvp_filtered,
        "region": region,
        "group_key": group_key,
        "spw": spw,
        "pol": pol,
    }


spectral_info_bandstop_result = plot_spectral_information_bandstop_pspec(
    combined_uvp_dict,
    combined_uvp_avg_dict,
    group_key=BANDSTOP_GROUP_KEY,
    pol=BANDSTOP_POL,
    spw=BANDSTOP_SPW,
    max_frequency=BANDSTOP_MAX_FREQUENCY,
    score_threshold=BANDSTOP_SCORE_THRESHOLD,
    stop_floor=BANDSTOP_STOP_FLOOR,
)


In [ ]:
# Blackman-Harris bandstop PSPEC using a user-selected LST Fourier mode

import numpy as np
import matplotlib.pyplot as plt

FRINGE_BANDSTOP_GROUP_KEY = globals().get("BANDSTOP_GROUP_KEY", None)
FRINGE_BANDSTOP_POL = globals().get("BANDSTOP_POL", "xx")
FRINGE_BANDSTOP_SPW = globals().get("BANDSTOP_SPW", None)
FRINGE_BANDSTOP_STOP_FLOOR = 0.05

# Toggle for which physical angular scale predicts the LST Fourier mode.
# Options: "fringe" or "beam_lobe".
FRINGE_BANDSTOP_FILTER_MODE = "fringe"

# User-facing width controls for the fringe-mode Blackman-Harris well.
FRINGE_MODE_FLAT_HALF_WIDTH_CYC_PER_HR = None
FRINGE_MODE_TAPER_WIDTH_CYC_PER_HR = None
FRINGE_MODE_FLAT_HALF_WIDTH_BINS = 0.5
FRINGE_MODE_TAPER_WIDTH_BINS = 1.0
FRINGE_MODE_FRACTIONAL_TAPER_WIDTH = 0.9

# Which fringe spacing drives the Fourier-mode notch?
# Options: "fringe_spacing" for raw lambda/b, or
# "fringe_phi_spacing" for latitude-projected lambda/[b cos(phi)].
FRINGE_MODE_SPACING_KEY = "fringe_phi_spacing"

# Beam-main-lobe mode. Set BEAM_MAIN_LOBE_WIDTH_DEG when using filter_mode="beam_lobe".
BEAM_MAIN_LOBE_WIDTH_DEG = None
BEAM_LOBE_SPACING_KEY = "beam_lobe_phi_width"  # or "beam_lobe_width" for no cos(phi) projection
BEAM_LOBE_MODE_FLAT_HALF_WIDTH_CYC_PER_HR = None
BEAM_LOBE_MODE_TAPER_WIDTH_CYC_PER_HR = None
BEAM_LOBE_MODE_FLAT_HALF_WIDTH_BINS = 0.5
BEAM_LOBE_MODE_TAPER_WIDTH_BINS = 1.0
BEAM_LOBE_MODE_FRACTIONAL_TAPER_WIDTH = 0.9


def _require_fringe_bandstop_dependencies():
    required = [
        "fringe_widths",
        "latitude_projection",
        "degrees_to_lst_hours",
        "DEG_PER_LST_HOUR",
        "get_lst_pspec_for_fft",
        "_uniform_lst_grid",
        "_fft_df",
        "get_spw_info",
        "apply_lst_bh_bandstop",
        "_select_bandstop_context",
        "_plot_bandstop_filtered_pspec",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Run the fringe-spacing and spectral-information bandstop cells first. Missing: " + ", ".join(missing))


def _normalize_bandstop_filter_mode(filter_mode):
    if filter_mode is None:
        filter_mode = FRINGE_BANDSTOP_FILTER_MODE
    text = str(filter_mode).strip().lower().replace("-", "_").replace(" ", "_")
    aliases = {
        "fringe": "fringe",
        "fringe_mode": "fringe",
        "fringe_spacing": "fringe",
        "baseline_fringe": "fringe",
        "beam": "beam_lobe",
        "beam_lobe": "beam_lobe",
        "main_lobe": "beam_lobe",
        "beam_main_lobe": "beam_lobe",
        "lobe": "beam_lobe",
        "lobe_mode": "beam_lobe",
    }
    if text not in aliases:
        raise ValueError("filter_mode must be 'fringe' or 'beam_lobe'.")
    return aliases[text]


def _normalize_lobe_spacing_key(lobe_spacing_key):
    if lobe_spacing_key is None:
        lobe_spacing_key = BEAM_LOBE_SPACING_KEY
    text = str(lobe_spacing_key).strip().lower().replace("-", "_").replace(" ", "_")
    aliases = {
        "raw": "beam_lobe_width",
        "no_phi": "beam_lobe_width",
        "without_phi": "beam_lobe_width",
        "beam_lobe_width": "beam_lobe_width",
        "main_lobe_width": "beam_lobe_width",
        "phi": "beam_lobe_phi_width",
        "projected": "beam_lobe_phi_width",
        "with_phi": "beam_lobe_phi_width",
        "cos_phi": "beam_lobe_phi_width",
        "beam_lobe_phi_width": "beam_lobe_phi_width",
        "main_lobe_phi_width": "beam_lobe_phi_width",
    }
    if text not in aliases:
        raise ValueError("lobe_spacing_key must be 'beam_lobe_width' or 'beam_lobe_phi_width'.")
    return aliases[text]


def _well_from_lst_span(lst_array_roll, values_for_resolution, lst_span_hr,
                        flat_half_width_cyc_per_hr=None,
                        taper_width_cyc_per_hr=None,
                        flat_half_width_bins=0.5,
                        taper_width_bins=1.0,
                        fractional_taper_width=0.9):
    """Convert an LST span into a Blackman-Harris well centered at 1/span."""
    lst_span_hr = float(lst_span_hr)
    if not np.isfinite(lst_span_hr) or lst_span_hr <= 0:
        raise ValueError("lst_span_hr must be positive and finite.")

    mode_cyc_per_hr = 1.0 / lst_span_hr
    lst_grid, _, dt_hours = _uniform_lst_grid(lst_array_roll, values_for_resolution)
    freq = np.fft.rfftfreq(lst_grid.size, d=dt_hours)
    df = _fft_df(freq)

    flat_half = flat_half_width_cyc_per_hr
    if flat_half is None:
        flat_half = flat_half_width_bins * df

    taper = taper_width_cyc_per_hr
    if taper is None:
        taper = max(
            taper_width_bins * df,
            fractional_taper_width * mode_cyc_per_hr,
        )

    return {
        "wells": [{
            "center": mode_cyc_per_hr,
            "flat_half_width": flat_half,
            "taper_width": taper,
        }],
        "mode_cyc_per_hr": mode_cyc_per_hr,
        "df": df,
    }


def _fringe_mode_well(lst_array_roll, values_for_resolution, baseline_m, frequency_MHz,
                      instrument=None, spacing_key=None, spacing_lst_hr_override=None):
    """Convert a chosen fringe spacing in LST hours into a Blackman-Harris Fourier well."""
    widths = fringe_widths(baseline_m, frequency_MHz=frequency_MHz, instrument=instrument)
    if spacing_key is None:
        spacing_key = FRINGE_MODE_SPACING_KEY
    if spacing_key not in widths:
        raise ValueError(f"spacing_key={spacing_key!r} not in fringe_widths() result.")

    spacing_lst_hr = widths[spacing_key]["lst_hr"]
    if not np.isfinite(spacing_lst_hr) or spacing_lst_hr <= 0:
        fallback_key = "fringe_phi_spacing_small_angle" if spacing_key == "fringe_phi_spacing" else "fringe_spacing_small_angle"
        spacing_lst_hr = widths[fallback_key]["lst_hr"]

    if spacing_lst_hr_override is not None:
        spacing_lst_hr = float(spacing_lst_hr_override)

    mode_info = _well_from_lst_span(
        lst_array_roll,
        values_for_resolution,
        spacing_lst_hr,
        flat_half_width_cyc_per_hr=FRINGE_MODE_FLAT_HALF_WIDTH_CYC_PER_HR,
        taper_width_cyc_per_hr=FRINGE_MODE_TAPER_WIDTH_CYC_PER_HR,
        flat_half_width_bins=FRINGE_MODE_FLAT_HALF_WIDTH_BINS,
        taper_width_bins=FRINGE_MODE_TAPER_WIDTH_BINS,
        fractional_taper_width=FRINGE_MODE_FRACTIONAL_TAPER_WIDTH,
    )

    return {
        **mode_info,
        "filter_mode": "fringe",
        "fringe_mode_cyc_per_hr": mode_info["mode_cyc_per_hr"],
        "spacing_key": spacing_key,
        "spacing_lst_hr": spacing_lst_hr,
        "spacing_deg": widths[spacing_key]["deg"],
        "raw_spacing_lst_hr": widths["fringe_spacing"]["lst_hr"],
        "raw_spacing_deg": widths["fringe_spacing"]["deg"],
        "fringe_phi_spacing_lst_hr": widths["fringe_phi_spacing"]["lst_hr"],
        "fringe_phi_spacing_deg": widths["fringe_phi_spacing"]["deg"],
        "latitude_projection": widths["latitude_projection"],
    }


def beam_lobe_widths(beam_main_lobe_width_deg, latitude_deg=None, latitude_rad=None, instrument=None):
    """
    Convert a beam main-lobe angular width into raw and latitude-projected LST spans.

    The raw mapping uses 24 LST hr = 360 deg, so width_hr = width_deg / 15.
    The projected mapping uses width_phi = width / |cos(phi)| for the latitude
    circle swept by the drift scan.
    """
    beam_main_lobe_width_deg = float(beam_main_lobe_width_deg)
    if not np.isfinite(beam_main_lobe_width_deg) or beam_main_lobe_width_deg <= 0:
        raise ValueError("beam_main_lobe_width_deg must be positive and finite.")

    raw_lst_hr = float(degrees_to_lst_hours(beam_main_lobe_width_deg))
    projection = latitude_projection(
        latitude_deg=latitude_deg,
        latitude_rad=latitude_rad,
        instrument=instrument,
        width_deg=beam_main_lobe_width_deg,
        width_lst_hr=raw_lst_hr,
    )
    phi_width_deg = float(np.asarray(projection["width_phi_deg"]))
    phi_width_lst_hr = float(np.asarray(projection["width_phi_lst_hr"]))

    return {
        "beam_lobe_width": {
            "deg": beam_main_lobe_width_deg,
            "lst_hr": raw_lst_hr,
        },
        "beam_lobe_phi_width": {
            "deg": phi_width_deg,
            "lst_hr": phi_width_lst_hr,
        },
        "latitude_projection": projection,
    }


def _lobe_mode_well(lst_array_roll, values_for_resolution, beam_main_lobe_width_deg,
                    instrument=None, lobe_spacing_key=None, lobe_lst_hr_override=None,
                    latitude_deg=None, latitude_rad=None):
    """Convert a beam main-lobe LST span into a Blackman-Harris Fourier well."""
    widths = beam_lobe_widths(
        beam_main_lobe_width_deg,
        latitude_deg=latitude_deg,
        latitude_rad=latitude_rad,
        instrument=instrument,
    )
    lobe_spacing_key = _normalize_lobe_spacing_key(lobe_spacing_key)
    lobe_lst_hr = widths[lobe_spacing_key]["lst_hr"]
    lobe_deg = widths[lobe_spacing_key]["deg"]

    if lobe_lst_hr_override is not None:
        lobe_lst_hr = float(lobe_lst_hr_override)
        lobe_deg = lobe_lst_hr * DEG_PER_LST_HOUR

    mode_info = _well_from_lst_span(
        lst_array_roll,
        values_for_resolution,
        lobe_lst_hr,
        flat_half_width_cyc_per_hr=BEAM_LOBE_MODE_FLAT_HALF_WIDTH_CYC_PER_HR,
        taper_width_cyc_per_hr=BEAM_LOBE_MODE_TAPER_WIDTH_CYC_PER_HR,
        flat_half_width_bins=BEAM_LOBE_MODE_FLAT_HALF_WIDTH_BINS,
        taper_width_bins=BEAM_LOBE_MODE_TAPER_WIDTH_BINS,
        fractional_taper_width=BEAM_LOBE_MODE_FRACTIONAL_TAPER_WIDTH,
    )

    return {
        **mode_info,
        "filter_mode": "beam_lobe",
        "lobe_mode_cyc_per_hr": mode_info["mode_cyc_per_hr"],
        "beam_main_lobe_width_deg": float(beam_main_lobe_width_deg),
        "lobe_spacing_key": lobe_spacing_key,
        "lobe_spacing_lst_hr": lobe_lst_hr,
        "lobe_spacing_deg": lobe_deg,
        "raw_lobe_width_lst_hr": widths["beam_lobe_width"]["lst_hr"],
        "raw_lobe_width_deg": widths["beam_lobe_width"]["deg"],
        "beam_lobe_phi_width_lst_hr": widths["beam_lobe_phi_width"]["lst_hr"],
        "beam_lobe_phi_width_deg": widths["beam_lobe_phi_width"]["deg"],
        "latitude_projection": widths["latitude_projection"],
    }


def _annotation_for_physical_mode(region, well):
    projection = region["latitude_projection"]
    if region["filter_mode"] == "beam_lobe":
        return (
            f"mode source=beam main lobe; input lobe={region['beam_main_lobe_width_deg']:.3f} deg; "
            f"spacing key={region['lobe_spacing_key']}; selected span={region['lobe_spacing_lst_hr']:.3f} LST hr "
            f"({region['lobe_spacing_deg']:.3f} deg); raw={region['raw_lobe_width_lst_hr']:.3f} hr, "
            f"phi-projected={region['beam_lobe_phi_width_lst_hr']:.3f} hr; "
            f"phi={projection['latitude_deg']:.3f} deg, cos(phi)={projection['cos_phi']:.3f}; "
            f"mode={region['lobe_mode_cyc_per_hr']:.3f} cyc/hr; "
            f"flat=[{well['flat_min']:.3f}, {well['flat_max']:.3f}], "
            f"outer=[{well['outer_min']:.3f}, {well['outer_max']:.3f}] cyc/hr"
        )

    return (
        f"mode source=baseline fringe; spacing key={region['spacing_key']}; "
        f"selected spacing={region['spacing_lst_hr']:.3f} LST hr ({region['spacing_deg']:.3f} deg); "
        f"raw={region['raw_spacing_lst_hr']:.3f} hr, phi-projected={region['fringe_phi_spacing_lst_hr']:.3f} hr; "
        f"phi={projection['latitude_deg']:.3f} deg, cos(phi)={projection['cos_phi']:.3f}; "
        f"mode={region['fringe_mode_cyc_per_hr']:.3f} cyc/hr; "
        f"flat=[{well['flat_min']:.3f}, {well['flat_max']:.3f}], "
        f"outer=[{well['outer_min']:.3f}, {well['outer_max']:.3f}] cyc/hr"
    )


def plot_fringe_mode_bandstop_pspec(combined_uvp_dict, combined_uvp_avg_dict,
                                    group_key=None, pol="xx", spw=None,
                                    stop_floor=0.0, spacing_key=None,
                                    filter_mode=None,
                                    beam_main_lobe_width_deg=None,
                                    lobe_spacing_key=None):
    """Bandstop an LST Fourier mode predicted by either fringe spacing or beam-lobe width."""
    _require_fringe_bandstop_dependencies()
    filter_mode = _normalize_bandstop_filter_mode(filter_mode)
    uvp, uvp_avg, group_key, pol, spw, center_frequency_MHz, freq_range_str = _select_bandstop_context(
        combined_uvp_dict, combined_uvp_avg_dict, group_key=group_key, pol=pol, spw=spw,
    )

    (_, _, _, _, lst_array_roll, uvp_power, uvpspec_averaged_power) = get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw)
    baseline_m = np.linalg.norm(uvp.bl_vecs[0])

    if filter_mode == "fringe":
        if not np.isfinite(center_frequency_MHz):
            raise ValueError("Need a valid SPW center frequency for the fringe-mode bandstop.")
        region = _fringe_mode_well(
            lst_array_roll,
            uvp_power,
            baseline_m,
            center_frequency_MHz,
            instrument=uvp,
            spacing_key=spacing_key,
        )
        title = f"PSPEC after Blackman-Harris fringe-mode bandstop | b={baseline_m:.2f} m, SPW {spw} ({freq_range_str}), pol={pol}"
    else:
        if beam_main_lobe_width_deg is None:
            beam_main_lobe_width_deg = BEAM_MAIN_LOBE_WIDTH_DEG
        if beam_main_lobe_width_deg is None:
            raise ValueError(
                "Set beam_main_lobe_width_deg, or set BEAM_MAIN_LOBE_WIDTH_DEG before using filter_mode='beam_lobe'."
            )
        region = _lobe_mode_well(
            lst_array_roll,
            uvp_power,
            beam_main_lobe_width_deg,
            instrument=uvp,
            lobe_spacing_key=lobe_spacing_key,
        )
        title = f"PSPEC after Blackman-Harris beam-lobe bandstop | lobe={beam_main_lobe_width_deg:.2f} deg, SPW {spw} ({freq_range_str}), pol={pol}"

    avg_filtered = apply_lst_bh_bandstop(lst_array_roll, uvpspec_averaged_power, region["wells"], stop_floor=stop_floor)
    uvp_filtered = apply_lst_bh_bandstop(lst_array_roll, uvp_power, region["wells"], stop_floor=stop_floor)

    well = uvp_filtered["wells"][0]
    annotation = _annotation_for_physical_mode(region, well)
    _plot_bandstop_filtered_pspec(lst_array_roll, uvpspec_averaged_power, uvp_power,
                                  avg_filtered, uvp_filtered, "uvp_power",
                                  title, annotation,
                                  plot_max_frequency=max(globals().get("BANDSTOP_MAX_FREQUENCY", 6.0), 1.2 * well["outer_max"]))
    print(annotation)
    return {
        "lst_array_roll": lst_array_roll,
        "uvpspec_averaged_power": uvpspec_averaged_power,
        "uvp_power": uvp_power,
        "avg_filtered": avg_filtered,
        "uvp_filtered": uvp_filtered,
        "region": region,
        "group_key": group_key,
        "spw": spw,
        "pol": pol,
        "baseline_m": baseline_m,
        "center_frequency_MHz": center_frequency_MHz,
        "filter_mode": filter_mode,
    }


fringe_mode_bandstop_result = plot_fringe_mode_bandstop_pspec(
    combined_uvp_dict,
    combined_uvp_avg_dict,
    group_key=FRINGE_BANDSTOP_GROUP_KEY,
    pol=FRINGE_BANDSTOP_POL,
    spw=FRINGE_BANDSTOP_SPW,
    stop_floor=FRINGE_BANDSTOP_STOP_FLOOR,
    spacing_key=FRINGE_MODE_SPACING_KEY,
    filter_mode=FRINGE_BANDSTOP_FILTER_MODE,
    beam_main_lobe_width_deg=BEAM_MAIN_LOBE_WIDTH_DEG,
    lobe_spacing_key=BEAM_LOBE_SPACING_KEY,
)


In [ ]:
# Build filtered combined_uvp_dict / combined_uvp_avg_dict copies from a bandstop result

import copy
import numpy as np


def _validate_bandstop_result_for_uvpspec_copy(bandstop_result):
    """Check that the bandstop result has the pieces needed for UVPSpec write-back."""
    required = [
        "group_key",
        "spw",
        "pol",
        "uvp_filtered",
        "avg_filtered",
    ]
    missing = [key for key in required if key not in bandstop_result]
    if missing:
        raise KeyError("bandstop_result is missing required key(s): " + ", ".join(missing))

    for result_key in ("uvp_filtered", "avg_filtered"):
        filtered_result = bandstop_result[result_key]
        for key in ("lst_grid", "filtered"):
            if key not in filtered_result:
                raise KeyError(f"bandstop_result[{result_key!r}] is missing {key!r}.")


def _zero_delay_index_for_uvpspec(uvp, spw):
    """Return the delay-bin index nearest tau=0 for this UVPSpec/SPW."""
    dlys_ns = np.asarray(uvp.get_dlys(spw), dtype=float) * 1e9
    zero_candidates = np.where(np.isclose(dlys_ns, 0.0))[0]
    zero_idx = int(zero_candidates[0]) if zero_candidates.size else int(np.nanargmin(np.abs(dlys_ns)))
    return zero_idx, dlys_ns


def _rolled_lst_hours_for_blpair(uvp, blp):
    """Match get_lst_pspec_for_fft(): radians -> hours, then roll late LSTs through zero."""
    lst_rad = np.asarray(uvp.lst_avg_array[uvp.blpair_to_indices(blp)], dtype=float)
    lst_hr = lst_rad * (12.0 / np.pi)
    return np.where(lst_hr > 20.0, lst_hr - 24.0, lst_hr)


def _unique_sorted_interp_grid(x, y):
    """Return finite, sorted, unique x values and median-combined y values."""
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]
    if x.size == 0:
        raise ValueError("Filtered bandstop result has no finite LST/value samples.")

    order = np.argsort(x)
    x = x[order]
    y = y[order]
    unique_x, inverse = np.unique(x, return_inverse=True)
    if unique_x.size == x.size:
        return x, y

    unique_y = np.empty(unique_x.size, dtype=float)
    for idx in range(unique_x.size):
        unique_y[idx] = np.nanmedian(y[inverse == idx])
    return unique_x, unique_y


def _interpolate_filtered_track_to_lsts(filtered_result, target_lst_hr):
    """Interpolate a uniform-grid filtered track back to arbitrary rolled LST samples."""
    grid, values = _unique_sorted_interp_grid(filtered_result["lst_grid"], filtered_result["filtered"])
    target_lst_hr = np.asarray(target_lst_hr, dtype=float)
    return np.interp(target_lst_hr, grid, values, left=values[0], right=values[-1])


def _stored_delay_index_for_exposed_column(uvp, spw, exposed_zero_idx, exposed_n_delay):
    """
    Map a delay-column index from get_data(key) back to uvp.data_array[spw].

    For unfolded UVPSpec objects this is the identity. For folded objects,
    get_data(key) exposes only the positive-delay tail of data_array[spw].
    """
    raw_n_delay = uvp.data_array[spw].shape[1]
    if not getattr(uvp, "folded", False):
        stored_idx = int(exposed_zero_idx)
    else:
        stored_idx = raw_n_delay - int(exposed_n_delay) + int(exposed_zero_idx)

    if stored_idx < 0 or stored_idx >= raw_n_delay:
        raise IndexError(
            f"Mapped delay index {stored_idx} is outside data_array[{spw!r}] "
            f"delay axis with length {raw_n_delay}."
        )
    return stored_idx


def _write_filtered_zero_delay_track_to_uvpspec(uvp, filtered_result, spw, pol,
                                                preserve_imag=True,
                                                write_abs_real_pspec=True,
                                                rtol=1e-7, atol=1e-7):
    """
    Write an LST-filtered tau~=0 PSPEC track into a UVPSpec object.

    This updates only the selected (spw, blpair, pol) zero-delay column. All other
    delays, metadata, stats, weights, integration arrays, and object attributes are
    left as they were in the copied UVPSpec.

    The source PSPEC track was originally built as abs(real(uvp.get_data(key))) at
    tau~=0. By default, this writes the filtered positive PSPEC values into the
    real component and preserves any existing imaginary component.
    """
    if not hasattr(uvp, "data_array") or spw not in uvp.data_array:
        raise TypeError("Expected a UVPSpec-like object with data_array[spw].")
    if not hasattr(uvp, "key_to_indices"):
        raise TypeError("Expected a UVPSpec-like object with key_to_indices().")

    zero_idx, dlys_ns = _zero_delay_index_for_uvpspec(uvp, spw)
    write_count = 0
    sample_count = 0
    blpair_summaries = []

    for blp in uvp.get_blpairs():
        key = (spw, blp, pol)
        spw_ind, blpairts_inds, polpair_ind = uvp.key_to_indices(key, omit_flags=False)
        exposed_data = np.asarray(uvp.get_data(key))
        if exposed_data.ndim != 2:
            raise ValueError(f"Expected get_data({key!r}) to be 2D (time, delay), got shape {exposed_data.shape}.")
        if zero_idx >= exposed_data.shape[1]:
            raise IndexError(f"zero_idx={zero_idx} outside delay axis for get_data({key!r}) shape {exposed_data.shape}.")

        stored_zero_idx = _stored_delay_index_for_exposed_column(
            uvp,
            spw_ind,
            exposed_zero_idx=zero_idx,
            exposed_n_delay=exposed_data.shape[1],
        )

        lst_roll = _rolled_lst_hours_for_blpair(uvp, blp)
        filtered_values = _interpolate_filtered_track_to_lsts(filtered_result, lst_roll)
        if filtered_values.size != exposed_data.shape[0]:
            raise ValueError(
                f"Filtered values for {key!r} have length {filtered_values.size}, "
                f"but data has {exposed_data.shape[0]} time samples."
            )
        if filtered_values.size != len(blpairts_inds):
            raise ValueError(
                f"Filtered values for {key!r} have length {filtered_values.size}, "
                f"but key_to_indices returned {len(blpairts_inds)} blpair-time rows."
            )

        if write_abs_real_pspec:
            filtered_values = np.abs(np.asarray(filtered_values, dtype=float))

        old_column = np.asarray(uvp.data_array[spw_ind][blpairts_inds, stored_zero_idx, polpair_ind])
        if np.iscomplexobj(uvp.data_array[spw_ind]):
            imag_part = np.imag(old_column) if preserve_imag else 0.0
            new_column = filtered_values.astype(np.real(old_column).dtype, copy=False) + 1j * imag_part
        else:
            new_column = filtered_values.astype(uvp.data_array[spw_ind].dtype, copy=False)

        uvp.data_array[spw_ind][blpairts_inds, stored_zero_idx, polpair_ind] = new_column
        verify_column = np.asarray(uvp.get_data(key)[:, zero_idx])
        if not np.allclose(verify_column, new_column, rtol=rtol, atol=atol, equal_nan=True):
            raise RuntimeError(
                f"Could not verify filtered data write-back for key={key!r}; "
                "the exposed get_data(key) column does not match data_array after assignment."
            )

        write_count += 1
        sample_count += int(filtered_values.size)
        blpair_summaries.append({
            "blpair": blp,
            "n_samples": int(filtered_values.size),
            "lst_min_hr": float(np.nanmin(lst_roll)),
            "lst_max_hr": float(np.nanmax(lst_roll)),
            "filtered_min": float(np.nanmin(filtered_values)),
            "filtered_max": float(np.nanmax(filtered_values)),
        })

    return {
        "spw": spw,
        "pol": pol,
        "zero_idx": int(zero_idx),
        "zero_delay_ns": float(dlys_ns[zero_idx]),
        "n_blpairs_written": int(write_count),
        "n_samples_written": int(sample_count),
        "preserve_imag": bool(preserve_imag),
        "write_abs_real_pspec": bool(write_abs_real_pspec),
        "blpairs": blpair_summaries,
    }

def make_filtered_combined_uvp_dicts_from_bandstop_result(
    bandstop_result,
    combined_uvp_dict,
    combined_uvp_avg_dict,
    preserve_imag=True,
    write_abs_real_pspec=True,
):
    """
    Deep-copy combined_uvp_dict and combined_uvp_avg_dict and write filtered PSPEC tracks.

    Parameters
    ----------
    bandstop_result : dict
        Output from plot_spectral_info_bandstop_pspec() or
        plot_fringe_mode_bandstop_pspec(). The function uses group_key, spw, pol,
        uvp_filtered, and avg_filtered.
    combined_uvp_dict, combined_uvp_avg_dict : dict
        Original UVPSpec dictionaries. They are not modified.
    preserve_imag : bool
        If True, preserve the original imaginary component at tau~=0 while replacing
        the real component with the filtered PSPEC value.
    write_abs_real_pspec : bool
        If True, write abs(filtered) because the plotted PSPEC track was built from
        abs(real(get_data(key))).

    Returns
    -------
    filt_combined_uvp_dict, filt_combined_uvp_avg_dict, write_summary
        Deep-copied dictionaries with only the selected group/SPW/pol zero-delay
        PSPEC track replaced by the filtered values.
    """
    _validate_bandstop_result_for_uvpspec_copy(bandstop_result)

    group_key = bandstop_result["group_key"]
    spw = bandstop_result["spw"]
    pol = bandstop_result["pol"]
    if group_key not in combined_uvp_dict:
        raise KeyError(f"group_key={group_key!r} not found in combined_uvp_dict.")
    if group_key not in combined_uvp_avg_dict:
        raise KeyError(f"group_key={group_key!r} not found in combined_uvp_avg_dict.")

    filt_combined_uvp_dict = copy.deepcopy(combined_uvp_dict)
    filt_combined_uvp_avg_dict = copy.deepcopy(combined_uvp_avg_dict)

    uvp_summary = _write_filtered_zero_delay_track_to_uvpspec(
        filt_combined_uvp_dict[group_key],
        bandstop_result["uvp_filtered"],
        spw,
        pol,
        preserve_imag=preserve_imag,
        write_abs_real_pspec=write_abs_real_pspec,
    )
    avg_summary = _write_filtered_zero_delay_track_to_uvpspec(
        filt_combined_uvp_avg_dict[group_key],
        bandstop_result["avg_filtered"],
        spw,
        pol,
        preserve_imag=preserve_imag,
        write_abs_real_pspec=write_abs_real_pspec,
    )

    write_summary = {
        "group_key": group_key,
        "spw": spw,
        "pol": pol,
        "source_region": bandstop_result.get("region", None),
        "uvp_summary": uvp_summary,
        "avg_summary": avg_summary,
    }
    return filt_combined_uvp_dict, filt_combined_uvp_avg_dict, write_summary


# Default write-back: use the spectral-information bandstop result requested above.
filt_combined_uvp_dict, filt_combined_uvp_avg_dict, filt_bandstop_write_summary = (
    make_filtered_combined_uvp_dicts_from_bandstop_result(
        spectral_info_bandstop_result,
        combined_uvp_dict,
        combined_uvp_avg_dict,
        preserve_imag=True,
        write_abs_real_pspec=True,
    )
)

print(
    "Created filt_combined_uvp_dict and filt_combined_uvp_avg_dict "
    f"for group={filt_bandstop_write_summary['group_key']}, "
    f"SPW={filt_bandstop_write_summary['spw']}, pol={filt_bandstop_write_summary['pol']}."
)
print(
    "Wrote uvp samples:", filt_bandstop_write_summary["uvp_summary"]["n_samples_written"],
    "| avg samples:", filt_bandstop_write_summary["avg_summary"]["n_samples_written"],
    "| zero-delay bin [ns]:", f"{filt_bandstop_write_summary['uvp_summary']['zero_delay_ns']:.3g}",
)

# To build filtered dictionaries from the fringe-mode bandstop instead, run:
filt_fringe_combined_uvp_dict, filt_fringe_combined_uvp_avg_dict, filt_fringe_write_summary = (
    make_filtered_combined_uvp_dicts_from_bandstop_result(
        fringe_mode_bandstop_result,
        combined_uvp_dict,
        combined_uvp_avg_dict,
    )
)


In [ ]:
# Beam-main-lobe mode. Set BEAM_MAIN_LOBE_WIDTH_DEG when using filter_mode="beam_lobe".
FRINGE_BANDSTOP_FILTER_MODE = "beam_lobe"
BEAM_MAIN_LOBE_WIDTH_DEG = 78.0
BEAM_LOBE_SPACING_KEY = "beam_lobe_phi_width"  # or "beam_lobe_width" for no cos(phi) projection
BEAM_LOBE_MODE_FLAT_HALF_WIDTH_CYC_PER_HR = None
BEAM_LOBE_MODE_TAPER_WIDTH_CYC_PER_HR = None
BEAM_LOBE_MODE_FLAT_HALF_WIDTH_BINS = 0.5
BEAM_LOBE_MODE_TAPER_WIDTH_BINS = 0.25
BEAM_LOBE_MODE_FRACTIONAL_TAPER_WIDTH = 0.6

fringe_2_mode_bandstop_result = plot_fringe_mode_bandstop_pspec(
    filt_fringe_combined_uvp_dict,
    filt_fringe_combined_uvp_avg_dict,
    group_key=FRINGE_BANDSTOP_GROUP_KEY,
    pol=FRINGE_BANDSTOP_POL,
    spw=FRINGE_BANDSTOP_SPW,
    stop_floor=FRINGE_BANDSTOP_STOP_FLOOR,
    spacing_key=FRINGE_MODE_SPACING_KEY,
    filter_mode=FRINGE_BANDSTOP_FILTER_MODE,
    beam_main_lobe_width_deg=BEAM_MAIN_LOBE_WIDTH_DEG,
    lobe_spacing_key=BEAM_LOBE_SPACING_KEY,
)


In [ ]:
# To build filtered dictionaries from the fringe-mode bandstop instead, run:
filt_lobe_combined_uvp_dict, filt_lobe_combined_uvp_avg_dict, filt_lobe_write_summary = (
    make_filtered_combined_uvp_dicts_from_bandstop_result(
        fringe_2_mode_bandstop_result,
        filt_fringe_combined_uvp_dict,
        filt_fringe_combined_uvp_avg_dict,
    )
)

In [ ]:
# Beam-main-lobe mode. Set BEAM_MAIN_LOBE_WIDTH_DEG when using filter_mode="beam_lobe".
FRINGE_BANDSTOP_FILTER_MODE = "beam_lobe"
BEAM_MAIN_LOBE_WIDTH_DEG = 100
BEAM_LOBE_SPACING_KEY = "beam_lobe_phi_width"  # or "beam_lobe_width" for no cos(phi) projection
BEAM_LOBE_MODE_FLAT_HALF_WIDTH_CYC_PER_HR = None
BEAM_LOBE_MODE_TAPER_WIDTH_CYC_PER_HR = None
BEAM_LOBE_MODE_FLAT_HALF_WIDTH_BINS = 0.5
BEAM_LOBE_MODE_TAPER_WIDTH_BINS = 0.25
BEAM_LOBE_MODE_FRACTIONAL_TAPER_WIDTH = 0.6

lobe_2_mode_bandstop_result = plot_fringe_mode_bandstop_pspec(
    filt_lobe_combined_uvp_dict,
    filt_lobe_combined_uvp_avg_dict,
    group_key=FRINGE_BANDSTOP_GROUP_KEY,
    pol=FRINGE_BANDSTOP_POL,
    spw=FRINGE_BANDSTOP_SPW,
    stop_floor=FRINGE_BANDSTOP_STOP_FLOOR,
    spacing_key=FRINGE_MODE_SPACING_KEY,
    filter_mode=FRINGE_BANDSTOP_FILTER_MODE,
    beam_main_lobe_width_deg=BEAM_MAIN_LOBE_WIDTH_DEG,
    lobe_spacing_key=BEAM_LOBE_SPACING_KEY,
)


In [ ]:
# Mode Analytics

import numpy as np
import matplotlib.pyplot as plt

# Pick one group/SPW/pol to view. None means use the same default selection as
# the bandstop cells: last group key and first valid SPW.
MODE_ANALYTICS_GROUP_KEY = globals().get("FRINGE_BANDSTOP_GROUP_KEY", globals().get("BANDSTOP_GROUP_KEY", None))
MODE_ANALYTICS_SPW = globals().get("FRINGE_BANDSTOP_SPW", globals().get("BANDSTOP_SPW", None))
MODE_ANALYTICS_POL = globals().get("FRINGE_BANDSTOP_POL", globals().get("BANDSTOP_POL", "xx"))

# Plot controls. Use MODE_ANALYTICS_XSCALE = "log" if desired.
MODE_ANALYTICS_MAX_FREQUENCY = 6.0
MODE_ANALYTICS_XSCALE = "log"
MODE_ANALYTICS_PLOT_MIXED_MODES = True
MODE_ANALYTICS_USE_PHI_MIXED_MODES = True  # True plots the cos(phi)-projected product modes.

MODE_ANALYTICS_WINDOW = "blackman-harris"  # "blackman-harris", "hann", None or "boxcar" gives no FFT window.
MODE_ANALYTICS_NORMALIZE = "none"   # "fractional" or None / "none" / "raw"
MODE_ANALYTICS_METHOD = "direct"            # "auto", "grid", "fft", "direct"/"nonuniform"
MODE_ANALYTICS_REMOVE_MEAN = True         # set False to keep DC
MODE_ANALYTICS_COLLAPSE_REPEATS = False    # collapse repeated LST samples
MODE_ANALYTICS_STATISTIC = "mean"       # "median" or "mean"
MODE_ANALYTICS_RETURN_INFO = False         # return additional information


# Analytic feature widths. These are converted to LST spans and then to
# Fourier centers by f_mode = 1 / Delta_LST.
MODE_ANALYTICS_FRINGE_SPACING_KEY = globals().get("FRINGE_MODE_SPACING_KEY", "fringe_phi_spacing")
MODE_ANALYTICS_LOBE_SPACING_KEY = globals().get("BEAM_LOBE_SPACING_KEY", "beam_lobe_phi_width")
MODE_ANALYTICS_MAIN_LOBE_WIDTH_DEG = 38.0
MODE_ANALYTICS_SIDE_LOBE_WIDTH_DEG = 18.0


def lst_span_to_fourier_mode(lst_span_hr):
    """Convert an LST feature width/span in hours into cycles per LST hour."""
    lst_span_hr = float(lst_span_hr)
    if not np.isfinite(lst_span_hr) or lst_span_hr <= 0:
        raise ValueError("lst_span_hr must be positive and finite.")
    return 1.0 / lst_span_hr


def fourier_mode_to_lst_span(mode_cyc_per_hr):
    """Convert a Fourier mode in cycles/hr back to its LST period/span."""
    mode_cyc_per_hr = float(mode_cyc_per_hr)
    if not np.isfinite(mode_cyc_per_hr) or mode_cyc_per_hr <= 0:
        return np.inf, np.inf
    span_lst_hr = 1.0 / mode_cyc_per_hr
    return span_lst_hr, span_lst_hr * DEG_PER_LST_HOUR


def select_mode_analytics_data(combined_uvp_dict, combined_uvp_avg_dict,
                               group_key=None, spw=None, pol="xx"):
    """Return one PSPEC-vs-LST track and its SPW metadata for the simple mode plot."""
    if group_key is None:
        group_key = list(combined_uvp_dict.keys())[-1]

    uvp = combined_uvp_dict[group_key]
    uvp_avg = combined_uvp_avg_dict[group_key]
    spw_info = get_spw_info(uvp)

    if spw is None:
        valid_spws = [key for key, val in sorted(spw_info.items()) if val is not None]
        if not valid_spws:
            raise ValueError("No valid SPW frequency ranges found.")
        spw = valid_spws[0]

    if spw_info.get(spw) is None:
        center_frequency_MHz = np.nan
        freq_range_str = "N/A"
    else:
        freq_min, freq_max = spw_info[spw]
        center_frequency_MHz = 0.5 * (freq_min + freq_max) / 1e6
        freq_range_str = f"{freq_min / 1e6:.2f}-{freq_max / 1e6:.2f} MHz"

    (_, _, _, _, lst_array_roll,
     uvp_power, uvpspec_averaged_power) = get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw)

    return {
        "group_key": group_key,
        "spw": spw,
        "pol": pol,
        "uvp": uvp,
        "uvp_avg": uvp_avg,
        "baseline_m": float(np.linalg.norm(uvp.bl_vecs[0])),
        "center_frequency_MHz": center_frequency_MHz,
        "freq_range_str": freq_range_str,
        "lst_array_roll": lst_array_roll,
        "uvp_power": uvp_power,
        "uvpspec_averaged_power": uvpspec_averaged_power,
    }


def analytic_mode_centers(baseline_m, frequency_MHz, instrument=None,
                          fringe_spacing_key=MODE_ANALYTICS_FRINGE_SPACING_KEY,
                          lobe_spacing_key=MODE_ANALYTICS_LOBE_SPACING_KEY,
                          main_lobe_width_deg=MODE_ANALYTICS_MAIN_LOBE_WIDTH_DEG,
                          side_lobe_width_deg=MODE_ANALYTICS_SIDE_LOBE_WIDTH_DEG):
    """Compute the basic analytic centers for fringe, main-lobe, and side-lobe scales."""
    centers = []

    fringe = fringe_widths(baseline_m, frequency_MHz=frequency_MHz, instrument=instrument)
    fringe_span_hr = fringe[fringe_spacing_key]["lst_hr"]
    centers.append({
        "label": "fringe",
        "mode_cyc_per_hr": lst_span_to_fourier_mode(fringe_span_hr),
        "span_lst_hr": fringe_span_hr,
        "span_deg": fringe[fringe_spacing_key]["deg"],
        "color": "black",
        "linestyle": "-",
    })

    for label, width_deg, color, linestyle in [
        ("main lobe", main_lobe_width_deg, "darkorange", "--"),
        ("side lobe", side_lobe_width_deg, "seagreen", ":"),
    ]:
        lobe = beam_lobe_widths(width_deg, instrument=instrument)
        lobe_span_hr = lobe[lobe_spacing_key]["lst_hr"]
        centers.append({
            "label": label,
            "mode_cyc_per_hr": lst_span_to_fourier_mode(lobe_span_hr),
            "span_lst_hr": lobe_span_hr,
            "span_deg": lobe[lobe_spacing_key]["deg"],
            "input_width_deg": width_deg,
            "color": color,
            "linestyle": linestyle,
        })

    return centers


def product_mode_pair(label, mode_a, mode_b, projection, color):
    """
    Combine two Fourier modes from a product f_a(LST) f_b(LST).

    For two cosine-like features with modes k_a and k_b, multiplication gives
    product modes k_plus = k_a + k_b and k_minus = |k_a - k_b|.  The returned
    LST spans are lambda_plus = 1/k_plus and lambda_minus = 1/k_minus.
    """
    k_plus = mode_a + mode_b
    k_minus = abs(mode_a - mode_b)
    out = []
    for suffix, mode, linestyle in [("k+", k_plus, "-."), ("k-", k_minus, (0, (1, 1)))]:
        span_hr, span_deg = fourier_mode_to_lst_span(mode)
        out.append({
            "label": f"{label} {suffix}",
            "projection": projection,
            "mode_cyc_per_hr": mode,
            "span_lst_hr": span_hr,
            "span_deg": span_deg,
            "color": color,
            "linestyle": linestyle,
        })
    return out


def fringe_lobe_product_modes(baseline_m, frequency_MHz, lobe_width_deg, label,
                              instrument=None, color="purple"):
    """
    Return main/side-lobe x fringe product modes with and without cos(phi).

    The raw branch uses lambda/b fringe spacing and the input lobe width as-is.
    The phi branch uses the latitude-projected widths, width_phi = width/cos(phi).
    Both branches return k+ = k_lobe + k_fringe and k- = |k_lobe - k_fringe|,
    plus their equivalent LST wavelengths, lambda_LST = 1/k.
    """
    fringe = fringe_widths(baseline_m, frequency_MHz=frequency_MHz, instrument=instrument)
    lobe = beam_lobe_widths(lobe_width_deg, instrument=instrument)

    branches = {
        "raw": ("fringe_spacing", "beam_lobe_width", "no cos(phi)"),
        "phi": ("fringe_phi_spacing", "beam_lobe_phi_width", "cos(phi)"),
    }
    mixed = {}
    for key, (fringe_key, lobe_key, projection) in branches.items():
        k_fringe = lst_span_to_fourier_mode(fringe[fringe_key]["lst_hr"])
        k_lobe = lst_span_to_fourier_mode(lobe[lobe_key]["lst_hr"])
        modes = product_mode_pair(label, k_lobe, k_fringe, projection, color)
        for mode in modes:
            mode.update({
                "lobe_width_input_deg": lobe_width_deg,
                "lobe_mode_cyc_per_hr": k_lobe,
                "fringe_mode_cyc_per_hr": k_fringe,
                "lobe_span_lst_hr": lobe[lobe_key]["lst_hr"],
                "fringe_span_lst_hr": fringe[fringe_key]["lst_hr"],
                "lobe_span_deg": lobe[lobe_key]["deg"],
                "fringe_span_deg": fringe[fringe_key]["deg"],
            })
        mixed[key] = modes
    return mixed


def all_fringe_lobe_product_modes(baseline_m, frequency_MHz, instrument=None,
                                  main_lobe_width_deg=MODE_ANALYTICS_MAIN_LOBE_WIDTH_DEG,
                                  side_lobe_width_deg=MODE_ANALYTICS_SIDE_LOBE_WIDTH_DEG,
                                  use_phi=True):
    """Return all raw/phi product modes and the branch selected for plotting."""
    all_modes = {
        "main_lobe_x_fringe": fringe_lobe_product_modes(
            baseline_m, frequency_MHz, main_lobe_width_deg, "main x fringe", instrument=instrument, color="purple"
        ),
        "side_lobe_x_fringe": fringe_lobe_product_modes(
            baseline_m, frequency_MHz, side_lobe_width_deg, "side x fringe", instrument=instrument, color="teal"
        ),
    }
    branch = "phi" if use_phi else "raw"
    plotted = []
    for modes_by_projection in all_modes.values():
        plotted.extend(modes_by_projection[branch])
    return all_modes, plotted


def plot_simple_mode_analytics(combined_uvp_dict, combined_uvp_avg_dict,
                               group_key=MODE_ANALYTICS_GROUP_KEY,
                               spw=MODE_ANALYTICS_SPW,
                               pol=MODE_ANALYTICS_POL,
                               max_frequency=MODE_ANALYTICS_MAX_FREQUENCY,
                               window=MODE_ANALYTICS_WINDOW,
                               xscale=MODE_ANALYTICS_XSCALE,
                               plot_mixed_modes=MODE_ANALYTICS_PLOT_MIXED_MODES,
                               use_phi_mixed_modes=MODE_ANALYTICS_USE_PHI_MIXED_MODES,
                               normalize=MODE_ANALYTICS_NORMALIZE,
                               method=MODE_ANALYTICS_METHOD,
                               remove_mean=MODE_ANALYTICS_REMOVE_MEAN,
                               collapse_repeats=MODE_ANALYTICS_COLLAPSE_REPEATS,
                               statistic=MODE_ANALYTICS_STATISTIC,
                               return_info=MODE_ANALYTICS_RETURN_INFO
                               ):
    """Simple one-panel PSPEC FFT with analytic and product-mode centers as vertical lines."""
    data = select_mode_analytics_data(combined_uvp_dict, combined_uvp_avg_dict, group_key, spw, pol)
    print(method)
    freq_avg, amp_avg, _, _ = lst_fft_spectrum(
        data["lst_array_roll"],
        data["uvpspec_averaged_power"],
        normalize=normalize,
        window=window,
        method=method,
        remove_mean=remove_mean,
        collapse_repeats=collapse_repeats,
        statistic=statistic,
        return_info=return_info,
    )
    print(method)
    freq_uvp, amp_uvp, _, _ = lst_fft_spectrum(
        data["lst_array_roll"],
        data["uvp_power"],
        normalize=normalize,
        window=window,
        method=method,
        remove_mean=remove_mean,
        collapse_repeats=collapse_repeats,
        statistic=statistic,
        return_info=return_info,
    )

    use_avg = freq_avg > 0
    use_uvp = freq_uvp > 0
    if max_frequency is not None:
        use_avg &= freq_avg <= max_frequency
        use_uvp &= freq_uvp <= max_frequency

    centers = analytic_mode_centers(
        data["baseline_m"],
        data["center_frequency_MHz"],
        instrument=data["uvp"],
    )
    mixed_modes, plotted_mixed_centers = all_fringe_lobe_product_modes(
        data["baseline_m"],
        data["center_frequency_MHz"],
        instrument=data["uvp"],
        use_phi=use_phi_mixed_modes,
    )
    plot_centers = centers + (plotted_mixed_centers if plot_mixed_modes else [])

    fig, ax = plt.subplots(figsize=(9.5, 5.2), constrained_layout=True)
    ax.plot(freq_avg[use_avg], amp_avg[use_avg], color="royalblue", lw=1.2, label="uvpspec_averaged_power")
    ax.plot(freq_uvp[use_uvp], amp_uvp[use_uvp], color="crimson", lw=1.2, alpha=0.85, label="uvp_power")

    for center in plot_centers:
        f0 = center["mode_cyc_per_hr"]
        if not np.isfinite(f0) or f0 <= 0:
            continue
        ax.axvline(
            f0,
            color=center["color"],
            linestyle=center["linestyle"],
            lw=2.0,
            label=f"{center['label']}: {f0:.3g} cyc/hr",
        )

    ax.set_yscale("log")
    ax.set_xscale(xscale)

    positive_freq = np.concatenate([
        freq_avg[use_avg],
        freq_uvp[use_uvp],
        np.array([c["mode_cyc_per_hr"] for c in plot_centers], dtype=float),
    ])
    positive_freq = positive_freq[np.isfinite(positive_freq) & (positive_freq > 0)]
    if positive_freq.size and max_frequency is not None:
        ax.set_xlim(0.8 * np.nanmin(positive_freq), max_frequency)

    ax.grid(alpha=0.25, lw=0.6)
    ax.set_xlabel("LST Fourier frequency [cycles / hr]")
    ax.set_ylabel("FFT amplitude\n(fractional PSPEC)")
    ax.set_title(
        f"Analytic mode centers on PSPEC FFT | b={data['baseline_m']:.2f} m, "
        f"SPW {data['spw']} ({data['freq_range_str']}), pol={data['pol']}"
    )
    ax.legend(fontsize=8, loc="best")
    plt.show()

    print("Analytic centers:")
    for center in centers:
        print(
            f"  {center['label']}: {center['mode_cyc_per_hr']:.4g} cyc/hr; "
            f"span={center['span_lst_hr']:.4g} hr = {center['span_deg']:.4g} deg"
        )

    branch = "phi" if use_phi_mixed_modes else "raw"
    print(f"Product-mode centers plotted from branch: {branch}")
    for center in plotted_mixed_centers:
        print(
            f"  {center['label']}: {center['mode_cyc_per_hr']:.4g} cyc/hr; "
            f"lambda_LST={center['span_lst_hr']:.4g} hr = {center['span_deg']:.4g} deg; "
            f"from k_lobe={center['lobe_mode_cyc_per_hr']:.4g}, "
            f"k_fringe={center['fringe_mode_cyc_per_hr']:.4g} cyc/hr"
        )

    return {
        "fig": fig,
        "ax": ax,
        "data": data,
        "freq_avg": freq_avg,
        "amp_avg": amp_avg,
        "freq_uvp": freq_uvp,
        "amp_uvp": amp_uvp,
        "centers": centers,
        "mixed_modes": mixed_modes,
        "plotted_mixed_centers": plotted_mixed_centers,
    }


mode_analytics_result = plot_simple_mode_analytics(combined_uvp_dict, combined_uvp_avg_dict)


In [ ]:
# Read and plot beam

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from pyuvdata import UVBeam


# ============================================
# USER INPUTS
# ============================================
# Set these to the directory and beamfits filename you want to inspect.
# If you already have a full path, put it in read_beam_dir and set
# read_beam_filename = None.
read_beam_dir = "/home/herastore02-1/HERA_Validation_rchandra/"
read_beam_filename = "airy_beam_14.0m_freqconst_ref80MHz_decay_0.3dBdeg_start_70.0deg.fits"

# Plot/extraction controls. The ZA cut uses the azimuth closest to
# read_beam_az_cut_deg and the frequency closest to read_beam_target_freq_mhz.
read_beam_target_freq_mhz = 80.0
read_beam_az_cut_deg = 0.0
read_beam_feed_index = 0
read_beam_vec_indices = (0, 1)
read_beam_floor_db = -80.0
read_beam_plot_in_db = True
read_beam_save_png = False
read_beam_png_path = None  # if None and read_beam_save_png=True, uses filename with .png


def _angular_distance_deg(angle_deg, target_deg):
    """Smallest signed angular separation, in degrees, for circular azimuth lookup."""
    return (np.asarray(angle_deg) - target_deg + 180.0) % 360.0 - 180.0


def read_azza_efield_beam(beam_dir, beam_filename=None):
    """
    Read an az_za efield UVBeam from a beamfits file.

    Parameters
    ----------
    beam_dir : str or Path
        Directory containing the beamfits file, or a full beamfits path if
        beam_filename is None.
    beam_filename : str or None
        Beamfits filename inside beam_dir.

    Returns
    -------
    beam : UVBeam
        Loaded pyuvdata UVBeam object.
    beam_path : Path
        Full path used for the read.
    """
    beam_path = Path(beam_dir).expanduser()
    if beam_filename is not None:
        beam_path = beam_path / beam_filename

    beam = UVBeam()
    beam.read_beamfits(str(beam_path))

    if beam.pixel_coordinate_system != "az_za":
        raise ValueError(
            f"Expected an az_za beamfits file, got {beam.pixel_coordinate_system!r} for {beam_path}."
        )
    if beam.beam_type != "efield":
        raise ValueError(
            f"Expected an efield beamfits file, got {beam.beam_type!r} for {beam_path}."
        )

    return beam, beam_path


def beam_za_power_cut(
    beam,
    target_freq_mhz=150.0,
    az_deg=0.0,
    feed_index=0,
    vec_indices=(0, 1),
):
    """
    Extract normalized power versus zenith angle for one azimuth cut.

    This matches the plot_airy_za_cut convention:
        power(ZA) = |E_vec0(ZA, az)|^2 + |E_vec1(ZA, az)|^2

    Returns
    -------
    za_deg : ndarray
        Zenith-angle axis in degrees.
    norm_power : ndarray
        Linear power normalized by its maximum along this cut.
    actual_freq_mhz : float
        Frequency actually selected from the beam file.
    actual_az_deg : float
        Azimuth actually selected from the beam file.
    """
    target_freq_hz = target_freq_mhz * 1e6
    freq_idx = int(np.argmin(np.abs(beam.freq_array - target_freq_hz)))

    az_array_deg = np.rad2deg(beam.axis1_array)
    az_idx = int(np.argmin(np.abs(_angular_distance_deg(az_array_deg, az_deg))))

    za_deg = np.rad2deg(beam.axis2_array)
    e_vec0 = beam.data_array[vec_indices[0], feed_index, freq_idx, :, az_idx]
    e_vec1 = beam.data_array[vec_indices[1], feed_index, freq_idx, :, az_idx]
    power = np.abs(e_vec0) ** 2 + np.abs(e_vec1) ** 2

    peak_power = np.nanmax(power)
    if not np.isfinite(peak_power) or peak_power <= 0:
        raise ValueError("Cannot normalize beam power: peak power is non-positive or non-finite.")

    norm_power = power / peak_power
    actual_freq_mhz = beam.freq_array[freq_idx] / 1e6
    actual_az_deg = az_array_deg[az_idx]

    return za_deg, norm_power, actual_freq_mhz, actual_az_deg


def read_and_plot_beam_za_cut(
    beam_dir,
    beam_filename=None,
    target_freq_mhz=150.0,
    az_deg=0.0,
    feed_index=0,
    vec_indices=(0, 1),
    plot_in_db=True,
    floor_db=-80.0,
    save_png=False,
    png_path=None,
):
    """
    Read a beamfits file, plot one ZA cut, and return the cut arrays.

    The returned dict contains za_deg and norm_power arrays for downstream use.
    """
    beam, beam_path = read_azza_efield_beam(beam_dir, beam_filename=beam_filename)
    za_deg, norm_power, actual_freq_mhz, actual_az_deg = beam_za_power_cut(
        beam,
        target_freq_mhz=target_freq_mhz,
        az_deg=az_deg,
        feed_index=feed_index,
        vec_indices=vec_indices,
    )

    floor_linear = 10 ** (floor_db / 10.0)
    power_db = 10.0 * np.log10(np.maximum(norm_power, floor_linear))

    plot_y = power_db if plot_in_db else norm_power
    y_label = "Normalized Power (dB)" if plot_in_db else "Normalized Power"

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(za_deg, plot_y, "b-", linewidth=2, label=beam_path.name)
    ax.set_xlabel("Zenith Angle (degrees)", fontsize=12)
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_title(
        f"Beam ZA cut at {actual_freq_mhz:.1f} MHz, az={actual_az_deg:.1f} deg\n"
        f"{beam_path.name}",
        fontsize=13,
    )
    ax.set_xlim(0, 180)
    if plot_in_db:
        ax.set_ylim(floor_db, 5)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()

    saved_png_path = None
    if save_png:
        saved_png_path = Path(png_path).expanduser() if png_path is not None else beam_path.with_suffix(".png")
        fig.savefig(saved_png_path, dpi=150)
        print(f"Plot saved: {saved_png_path}")

    plt.show()

    return {
        "beam": beam,
        "beam_path": beam_path,
        "za_deg": za_deg,
        "norm_power": norm_power,
        "power_db": power_db,
        "actual_freq_mhz": actual_freq_mhz,
        "actual_az_deg": actual_az_deg,
        "png_path": saved_png_path,
        "fig": fig,
        "ax": ax,
    }


read_beam_result = read_and_plot_beam_za_cut(
    read_beam_dir,
    beam_filename=read_beam_filename,
    target_freq_mhz=read_beam_target_freq_mhz,
    az_deg=read_beam_az_cut_deg,
    feed_index=read_beam_feed_index,
    vec_indices=read_beam_vec_indices,
    plot_in_db=read_beam_plot_in_db,
    floor_db=read_beam_floor_db,
    save_png=read_beam_save_png,
    png_path=read_beam_png_path,
)

# Arrays requested for downstream use.
read_beam_za_deg = read_beam_result["za_deg"]
read_beam_norm_power = read_beam_result["norm_power"]


In [ ]:
# Beam FFT in LST
# Flip and symmetrize read_beam_norm_power, convert zangle to LST -90 to 90 deg, then divide both data by cos(phi) to get accurate projection, then convert to LST given that 1 degree is equal to 24/360 hours. 
# Then we take the FFT of the symmetrized and projected beam cut to get the beam mode spectrum. We can then plot this spectrum to see the main lobe and side lobes in Fourier space, which correspond to the angular features of the beam.

# This cell expects read_beam_za_deg and read_beam_norm_power from the previous cell.
# The latitude projection is applied to the angular/LST coordinate only:
#     theta_phi = theta / |cos(phi)|
# The normalized beam power itself is not divided by cos(phi).

# -----------------------------
# User controls
# -----------------------------
BEAM_FFT_MAX_ZA_DEG = 90.0       # use above-horizon ZA range and mirror it to -ZA...+ZA
BEAM_FFT_USE_LATITUDE_PROJECTION = True
BEAM_FFT_LATITUDE_DEG = None     # None -> infer from data/instrument if available, else HERA fallback
BEAM_FFT_NORMALIZE = None        # None/'none' keeps beam power units; 'fractional' uses P/median(P)-1
BEAM_FFT_REMOVE_MEAN = True      # True suppresses the DC mode before plotting side/main-lobe modes
BEAM_FFT_WINDOW = "blackman-harris"  # None/'boxcar', 'hann', or 'blackman-harris'
BEAM_FFT_METHOD = "auto"        # 'auto', 'fft', 'grid', or 'direct'/'nonuniform'
BEAM_FFT_PLOT_MAX_CYC_PER_HR = 6.0
BEAM_FFT_PLOT_BEAM_IN_DB = True  # only affects the left diagnostic panel, not the FFT input
BEAM_FFT_BEAM_FLOOR_DB = globals().get("read_beam_floor_db", -80.0)
BEAM_FFT_MARK_FEATURES = True
BEAM_FFT_FEATURE_SCORE_THRESHOLD = 2.0
BEAM_FFT_CONTINUUM_WINDOW = 15
BEAM_FFT_CONTINUUM_PERCENTILE = 40


def symmetrize_beam_cut_to_signed_za(za_deg, norm_power, max_za_deg=90.0):
    """
    Build a signed ZA cut from a one-sided beam cut.

    Input ZA is assumed to be a radial zenith angle, usually 0..180 deg.
    We keep 0..max_za_deg and mirror it about zenith to produce -max..+max.
    The duplicated zenith sample is kept only once.
    """
    za_deg = np.asarray(za_deg, dtype=float)
    norm_power = np.asarray(norm_power, dtype=float)
    finite = np.isfinite(za_deg) & np.isfinite(norm_power)
    za_deg = za_deg[finite]
    norm_power = norm_power[finite]

    order = np.argsort(za_deg)
    za_deg = za_deg[order]
    norm_power = norm_power[order]

    use = (za_deg >= 0.0) & (za_deg <= max_za_deg)
    za_pos = za_deg[use]
    power_pos = norm_power[use]
    if za_pos.size < 2:
        raise ValueError("Need at least two finite ZA samples in the selected above-horizon range.")

    nonzero = za_pos > 0.0
    signed_za_deg = np.concatenate((-za_pos[nonzero][::-1], za_pos))
    signed_power = np.concatenate((power_pos[nonzero][::-1], power_pos))

    order = np.argsort(signed_za_deg)
    return signed_za_deg[order], signed_power[order]


def beam_signed_za_to_projected_lst_hours(signed_za_deg, latitude_deg=None, use_projection=True):
    """
    Convert signed beam angle to the equivalent LST coordinate in hours.

    Latitude projection stretches the angular coordinate by 1/|cos(phi)|,
    matching the smaller latitude-circle circumference used in the fringe/lobe cells.
    Then 360 deg corresponds to 24 LST hr, so 1 deg = 24/360 hr.
    """
    if use_projection:
        projection = latitude_projection(latitude_deg=latitude_deg)
        projected_angle_deg = np.asarray(signed_za_deg, dtype=float) / projection["cos_phi"]
    else:
        projection = {
            "latitude_deg": np.nan,
            "latitude_rad": np.nan,
            "cos_phi": 1.0,
            "width_scale": 1.0,
            "source": "projection disabled",
        }
        projected_angle_deg = np.asarray(signed_za_deg, dtype=float)

    lst_hours = projected_angle_deg * (24.0 / 360.0)
    return lst_hours, projected_angle_deg, projection


def plot_beam_fft_in_lst(za_deg,
                         norm_power,
                         max_za_deg=90.0,
                         latitude_deg=None,
                         use_projection=True,
                         normalize=None,
                         remove_mean=True,
                         window="blackman-harris",
                         method="auto",
                         plot_max_frequency=6.0,
                         plot_beam_in_db=True,
                         beam_floor_db=-80.0,
                         mark_features=True,
                         feature_score_threshold=2.0,
                         continuum_window=15,
                         continuum_percentile=40):
    """Symmetrize a beam ZA cut, project to LST, FFT it, and plot both domains."""
    signed_za_deg, signed_power = symmetrize_beam_cut_to_signed_za(
        za_deg,
        norm_power,
        max_za_deg=max_za_deg,
    )
    beam_lst_hr, projected_angle_deg, projection = beam_signed_za_to_projected_lst_hours(
        signed_za_deg,
        latitude_deg=latitude_deg,
        use_projection=use_projection,
    )

    freq_cyc_per_hr, beam_fft_amp, fft_lst_grid, fft_input_power, fft_info = lst_fft_spectrum(
        beam_lst_hr,
        signed_power,
        normalize=normalize,
        remove_mean=remove_mean,
        window=window,
        method=method,
        collapse_repeats=True,
        statistic="median",
        return_info=True,
    )

    use_freq = freq_cyc_per_hr > 0
    if plot_max_frequency is not None:
        use_freq &= freq_cyc_per_hr <= plot_max_frequency

    feature_result = None
    if mark_features and "spectral_information_filter" in globals():
        feature_result = spectral_information_filter(
            freq_cyc_per_hr,
            beam_fft_amp,
            n_features=6,
            continuum_window=continuum_window,
            continuum_percentile=continuum_percentile,
            score_threshold=feature_score_threshold,
            max_frequency=plot_max_frequency,
        )

    if plot_beam_in_db:
        beam_floor_linear = 10.0 ** (beam_floor_db / 10.0)
        left_plot_y = 10.0 * np.log10(np.maximum(signed_power, beam_floor_linear))
        left_ylabel = "normalized beam power [dB]"
        left_ylim = (beam_floor_db, 5.0)
        left_label = "symmetrized beam power [dB]"
    else:
        left_plot_y = signed_power
        left_ylabel = "normalized beam power"
        left_ylim = None
        left_label = "symmetrized beam power"

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), constrained_layout=True)
    axes[0].plot(beam_lst_hr, left_plot_y, color="royalblue", lw=1.8, label=left_label)
    axes[0].set_xlabel("projected LST coordinate [hr]")
    axes[0].set_ylabel(left_ylabel)
    axes[0].set_title("Beam cut after mirror symmetry and latitude projection")
    if left_ylim is not None:
        axes[0].set_ylim(*left_ylim)
    axes[0].grid(alpha=0.25)
    axes[0].legend(loc="best")

    plot_amp = np.maximum(beam_fft_amp[use_freq], np.finfo(float).tiny)
    axes[1].plot(freq_cyc_per_hr[use_freq], plot_amp, color="crimson", lw=1.5, label="beam FFT")
    if feature_result is not None:
        continuum = feature_result["continuum"]
        axes[1].plot(
            freq_cyc_per_hr[use_freq],
            np.maximum(continuum[use_freq], np.finfo(float).tiny),
            color="crimson",
            ls="--",
            alpha=0.45,
            lw=0.9,
            label="smooth continuum",
        )
        for feat in feature_result["features"]:
            axes[1].axvspan(
                feat["frequency_min_cyc_per_hr"],
                feat["frequency_max_cyc_per_hr"],
                color="crimson",
                alpha=0.08,
                lw=0,
                zorder=0,
            )
            axes[1].scatter(
                feat["frequency_cyc_per_hr"],
                max(feat["amplitude"], np.finfo(float).tiny),
                s=38,
                marker="o",
                facecolors="none",
                edgecolors="crimson",
                linewidths=1.4,
                zorder=5,
            )
    axes[1].set_yscale("log")
    axes[1].set_xlabel("LST Fourier frequency [cycles / hr]")
    ylabel = "FFT amplitude"
    if normalize == "fractional":
        ylabel += "\n(fractional beam power)"
    axes[1].set_ylabel(ylabel)
    axes[1].set_title("Beam mode spectrum in LST Fourier space")
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc="best")
    if plot_max_frequency is not None:
        axes[1].set_xlim(0, plot_max_frequency)

    fig.suptitle(
        f"Beam FFT in LST | phi={projection['latitude_deg']:.3f} deg, "
        f"cos(phi)={projection['cos_phi']:.3f}, method={fft_info['method']}, "
        f"window={window}, remove_mean={remove_mean}"
    )
    plt.show()

    print(
        f"Latitude projection: source={projection['source']}; "
        f"phi={projection['latitude_deg']:.4f} deg; cos(phi)={projection['cos_phi']:.4f}; "
        f"width scale={projection['width_scale']:.4f}"
    )
    print(
        f"Beam FFT: method={fft_info['method']}, N={fft_info['n_samples']}, "
        f"dt={fft_info['dt_hours']:.5g} hr, normalize={normalize}, "
        f"remove_mean={remove_mean}, window={window}"
    )
    print(
        "Left beam panel note: plotted in dB for visual comparison; "
        "FFT input remains the linear normalized signed_power array."
    )
    if feature_result is not None:
        if feature_result["features"]:
            formatted = ", ".join(
                f"{feat['frequency_cyc_per_hr']:.4g} cyc/hr "
                f"(period {feat['period_hr']:.3g} hr, score {feat['score']:.2g})"
                for feat in feature_result["features"]
            )
            print(f"Beam FFT feature modes: {formatted}")
        else:
            print("Beam FFT feature modes: none above the current threshold")

    return {
        "signed_za_deg": signed_za_deg,
        "projected_angle_deg": projected_angle_deg,
        "beam_lst_hr": beam_lst_hr,
        "signed_power": signed_power,
        "signed_power_db": 10.0 * np.log10(np.maximum(signed_power, 10.0 ** (beam_floor_db / 10.0))),
        "freq_cyc_per_hr": freq_cyc_per_hr,
        "fft_amp": beam_fft_amp,
        "fft_lst_grid": fft_lst_grid,
        "fft_input_power": fft_input_power,
        "latitude_projection": projection,
        "fft_info": fft_info,
        "feature_result": feature_result,
        "fig": fig,
        "axes": axes,
    }


beam_fft_lst_result = plot_beam_fft_in_lst(
    read_beam_za_deg,
    read_beam_norm_power,
    max_za_deg=BEAM_FFT_MAX_ZA_DEG,
    latitude_deg=BEAM_FFT_LATITUDE_DEG,
    use_projection=BEAM_FFT_USE_LATITUDE_PROJECTION,
    normalize=BEAM_FFT_NORMALIZE,
    remove_mean=BEAM_FFT_REMOVE_MEAN,
    window=BEAM_FFT_WINDOW,
    method=BEAM_FFT_METHOD,
    plot_max_frequency=BEAM_FFT_PLOT_MAX_CYC_PER_HR,
    plot_beam_in_db=BEAM_FFT_PLOT_BEAM_IN_DB,
    beam_floor_db=BEAM_FFT_BEAM_FLOOR_DB,
    mark_features=BEAM_FFT_MARK_FEATURES,
    feature_score_threshold=BEAM_FFT_FEATURE_SCORE_THRESHOLD,
    continuum_window=BEAM_FFT_CONTINUUM_WINDOW,
    continuum_percentile=BEAM_FFT_CONTINUUM_PERCENTILE,
)

# Result arrays are available in beam_fft_lst_result.


In [ ]:
# Percent signal-loss style plot after spectral-information bandstop

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator


def _bandstop_percent_difference(uvp_power, uvpspec_averaged_power):
    """Same percent-difference convention as plot_percent_difference()."""
    uvp_power = np.asarray(uvp_power, dtype=float)
    uvpspec_averaged_power = np.asarray(uvpspec_averaged_power, dtype=float)
    return 100.0 * (uvp_power - uvpspec_averaged_power) / np.where(
        uvpspec_averaged_power != 0,
        uvpspec_averaged_power,
        np.nan,
    )


def _aligned_filtered_percent_difference(result):
    """Return filtered percent difference on a common filtered LST grid."""
    avg_lst = np.asarray(result["avg_filtered"]["lst_grid"], dtype=float)
    avg_filtered = np.asarray(result["avg_filtered"]["filtered"], dtype=float)
    uvp_lst = np.asarray(result["uvp_filtered"]["lst_grid"], dtype=float)
    uvp_filtered = np.asarray(result["uvp_filtered"]["filtered"], dtype=float)

    if avg_lst.shape != uvp_lst.shape or not np.allclose(avg_lst, uvp_lst, equal_nan=True):
        order = np.argsort(uvp_lst)
        uvp_filtered = np.interp(avg_lst, uvp_lst[order], uvp_filtered[order])

    return avg_lst, _bandstop_percent_difference(uvp_filtered, avg_filtered)


def plot_bandstop_percent_difference_result(result,
                                            method_label,
                                            color="purple",
                                            xlim=(0, 8),
                                            show_original=True):
    """
    Plot percent difference/signal-loss after a bandstop filter using the same
    convention as plot_percent_difference().
    """
    lst_original = np.asarray(result["lst_array_roll"], dtype=float)
    original_pct = _bandstop_percent_difference(
        result["uvp_power"],
        result["uvpspec_averaged_power"],
    )

    lst_filtered, filtered_pct = _aligned_filtered_percent_difference(result)

    fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)

    ax.axhline(10, color="blue", alpha=0.5, linewidth=1.0)
    ax.axhline(-10, color="blue", alpha=0.5, linewidth=1.0)
    ax.axhline(5, color="green", alpha=0.5, linewidth=1.0)
    ax.axhline(-5, color="green", alpha=0.5, linewidth=1.0)
    ax.axhline(0, color="black", alpha=0.5, linewidth=1.0)

    if show_original:
        ax.scatter(
            lst_original,
            original_pct,
            s=5,
            marker="o",
            color="0.45",
            alpha=0.25,
            label="original % difference",
        )

    ax.plot(
        lst_filtered,
        filtered_pct,
        color=color,
        linewidth=1.8,
        label="bandstop-filtered % difference",
    )
    ax.scatter(
        lst_filtered,
        filtered_pct,
        s=8,
        marker="o",
        color=color,
        alpha=0.65,
    )

    finite_vals = np.concatenate([
        original_pct[np.isfinite(original_pct)],
        filtered_pct[np.isfinite(filtered_pct)],
    ])
    if finite_vals.size:
        lo, hi = np.nanpercentile(finite_vals, [1, 99])
        lo = min(lo, -12.0)
        hi = max(hi, 12.0)
        pad = 0.10 * max(hi - lo, 1.0)
        ax.set_ylim(lo - pad, hi + pad)

    ax.xaxis.set_major_locator(MultipleLocator(1))
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.grid(False)
    ax.set_ylabel(r"% difference $100(P_{uvp}-P_{avg})/P_{avg}$", fontsize=11)
    ax.set_xlabel("LST (Hours)", fontsize=11)

    group_key = result.get("group_key", "N/A")
    spw = result.get("spw", "N/A")
    pol = result.get("pol", "N/A")
    ax.set_title(f"{method_label} bandstop % difference | group={group_key}, SPW {spw}, pol={pol}", fontsize=12)
    # ax.set_ylim(-20, 20)
    ax.set_ylim(-1, 1)
    ax.set_xlim(0, 8)
    ax.legend(fontsize=9, loc="best")

    print(f"{method_label} original median % diff: {np.nanmedian(original_pct):.3f}")
    print(f"{method_label} filtered median % diff: {np.nanmedian(filtered_pct):.3f}")
    print(f"{method_label} original mean % diff: {np.nanmean(original_pct):.3f}")
    print(f"{method_label} filtered mean % diff: {np.nanmean(filtered_pct):.3f}")

    plt.show()
    return {
        "lst_original": lst_original,
        "original_percent_diff": original_pct,
        "lst_filtered": lst_filtered,
        "filtered_percent_diff": filtered_pct,
    }


spectral_info_bandstop_percent_result = plot_bandstop_percent_difference_result(
    spectral_info_bandstop_result,
    method_label="Spectral-information",
    color="purple",
)


In [ ]:
# Percent signal-loss style plot after fringe-mode bandstop

fringe_mode_bandstop_percent_result = plot_bandstop_percent_difference_result(
    fringe_mode_bandstop_result,
    method_label="Fringe-mode",
    color="darkorange",
)


In [ ]:
plot_percent_difference(combined_uvp_dict, combined_uvp_avg_dict)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


def plot_full_delay_waterfalls(combined_uvp_dict,
                               combined_uvp_avg_dict,
                               pol='xx',
                               spw=0):
    """
    Plot full delay vs LST waterfalls for each grp_key
    comparing:
        - original UVPSpec
        - averaged UVPSpec

    Parameters
    ----------
    combined_uvp_dict : dict
        {grp_key: UVPSpec}
    combined_uvp_avg_dict : dict
        {grp_key: UVPSpec}
    pol : str
        Polarization (e.g. 'xx' or 'pI')
    spw : int
        Spectral window index
    """

    for grp_key in combined_uvp_dict:

        # --- Access uvp objects via group key (important subtlety) ---
        uvp     = combined_uvp_dict[grp_key]
        uvp_avg = combined_uvp_avg_dict[grp_key]

        # --- Delay axis (ns) ---
        dlys = uvp.get_dlys(spw) * 1e9  # convert to ns
        Ndlys = len(dlys)

        # --- Containers for all baseline-pair blocks ---
        lst_blocks = []
        spec_blocks = []
        spec_avg_blocks = []

        # --- Loop over ALL baseline pairs ---
        for blp in uvp.get_blpairs():

            key = (spw, blp, pol)
            print("key ", key)

            # LST indices corresponding to this baseline pair
            inds = uvp.blpair_to_indices(blp)

            lst_block = uvp.lst_avg_array[inds]

            spec_block = np.real(uvp.get_data(key))       # (Ntimes_bl, Ndlys)
            spec_avg_block = np.real(uvp_avg.get_data(key))

            lst_blocks.append(lst_block)
            spec_blocks.append(spec_block)
            spec_avg_blocks.append(spec_avg_block)

        # --- Concatenate across baseline pairs ---
        lst_all = np.concatenate(lst_blocks)
        spec_all = np.concatenate(spec_blocks, axis=0)
        spec_avg_all = np.concatenate(spec_avg_blocks, axis=0)

        # --- Sort by LST ---
        order = np.argsort(lst_all)

        lst_sorted = lst_all[order]
        spec_sorted = spec_all[order, :]
        spec_avg_sorted = spec_avg_all[order, :]

        # --- Convert LST from radians to hours ---
        # LST [hours] = LST [radians] * (12 / π)
        lst_hours = lst_sorted * (12.0 / np.pi)

        # Optional rolling (to center window)
        lst_hours = np.where(lst_hours > 20, lst_hours - 24, lst_hours)

        # --- Plot ---
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

        extent = [
            lst_hours.min(),
            lst_hours.max(),
            dlys.min(),
            dlys.max()
        ]
        
        # Combine both datasets to compute common limits
        combined_power = np.abs(
            np.concatenate([spec_sorted.flatten(),
                            spec_avg_sorted.flatten()])
        )

        vmin = np.nanpercentile(combined_power, 5)
        vmax = np.nanpercentile(combined_power, 99)

        common_norm = LogNorm(vmin=vmin, vmax=vmax)

        im0 = axes[0].imshow(
        np.abs(spec_sorted.T),
        aspect='auto',
        origin='lower',
        extent=extent,
        norm=common_norm

        )


        im1 = axes[1].imshow(
        np.abs(spec_avg_sorted.T),
        aspect='auto',
        origin='lower',
        extent=extent,
        norm=common_norm

        )


        axes[0].set_title(f"Coherent Avg UVP\nGrp: {grp_key}")
        axes[1].set_title(f"Incoherent Avg UVP\nGrp: {grp_key}")

        for ax in axes:
            ax.set_xlabel("LST (Hours)")
        axes[0].set_ylabel("Delay (ns)")

        fig.colorbar(im0, ax=axes[0], label="|P(k)|")
        fig.colorbar(im1, ax=axes[1], label="|P(k)|")

        plt.tight_layout()
        plt.show()


plot_full_delay_waterfalls(combined_uvp_dict, combined_uvp_avg_dict)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_percent_sigloss_from_uvp(combined_uvp_dict,
                                  combined_uvp_avg_dict,
                                  pol='xx',
                                  spw=0,
                                  vmin=-20,
                                  vmax=20):
    """
    Compute and plot full 2D % signal loss heatmap
    over ALL delays and ALL LSTs for each grp_key.

    Parameters
    ----------
    combined_uvp_dict       : dict of coherent UVPSpec objects
    combined_uvp_avg_dict   : dict of incoherent UVPSpec objects
    pol                     : polarization string
    spw                     : spectral window index
    vmin, vmax              : colorbar range for % loss
    """

    for grp_key in combined_uvp_dict:

        print(f"Processing grp_key: {grp_key}")

        uvp_coh   = combined_uvp_dict[grp_key]
        uvp_incoh = combined_uvp_avg_dict[grp_key]

        # -----------------------------
        # Delay axis (ns)
        # -----------------------------
        dlys = uvp_coh.get_dlys(spw) * 1e9

        # -----------------------------
        # Gather all baseline-pairs
        # -----------------------------
        lst_list = []
        spec_coh_list = []
        spec_incoh_list = []

        for blp in uvp_coh.get_blpairs():

            key = (spw, blp, pol)

            # LST (radians) → convert to hours
            lst_block = uvp_coh.lst_avg_array[
                uvp_coh.blpair_to_indices(blp)
            ]

            # Spectra: shape (Ntimes, Ndlys)
            spec_coh_block = np.real(uvp_coh.get_data(key))
            spec_incoh_block = np.real(uvp_incoh.get_data(key))

            lst_list.append(lst_block)
            spec_coh_list.append(spec_coh_block)
            spec_incoh_list.append(spec_incoh_block)

        # -----------------------------
        # Concatenate over baselines
        # -----------------------------
        lst_all = np.concatenate(lst_list)
        spec_coh_all = np.vstack(spec_coh_list)
        spec_incoh_all = np.vstack(spec_incoh_list)

        # -----------------------------
        # Sort by LST
        # -----------------------------
        order = np.argsort(lst_all)

        lst_sorted_rad = lst_all[order]
        spec_coh_sorted = spec_coh_all[order, :]
        spec_incoh_sorted = spec_incoh_all[order, :]

        # Convert LST → hours
        lst_sorted_hr = lst_sorted_rad * (12 / np.pi)

        # Optional: roll LST
        lst_sorted_hr = np.where(lst_sorted_hr > 20,
                                 lst_sorted_hr - 24,
                                 lst_sorted_hr)

        # -----------------------------
        # Compute % signal loss (2D)
        # -----------------------------
        P_coh = np.abs(spec_coh_sorted)
        P_incoh = np.abs(spec_incoh_sorted)

        percent_loss = 100.0 * (P_coh - P_incoh) / np.where(
            P_incoh != 0,
            P_incoh,
            np.nan
        )

        print("2D percent_loss shape:", percent_loss.shape)

        # -----------------------------
        # Plot heatmap
        # -----------------------------
        fig, ax = plt.subplots(figsize=(8, 6))

        extent = [
            lst_sorted_hr.min(),
            lst_sorted_hr.max(),
            dlys.min(),
            dlys.max()
        ]

        im = ax.imshow(
            percent_loss.T,
            aspect='auto',
            origin='lower',
            extent=extent,
            cmap='RdBu_r',
            vmin=vmin,
            vmax=vmax
        )

        ax.set_xlabel("LST (Hours)")
        ax.set_ylabel("Delay (ns)")
        ax.set_title(f"% Signal Loss\nGrp: {grp_key}, SPW: {spw}, Pol: {pol}")

        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label("% Signal Loss")

        plt.tight_layout()
        plt.show()

        # Optional diagnostics
        print("Mean % loss:", np.nanmean(percent_loss))
        print("Median % loss:", np.nanmedian(percent_loss))


plot_percent_sigloss_from_uvp(
    combined_uvp_dict,
    combined_uvp_avg_dict,
    pol='xx',
    spw=0,
    vmin=-100,
    vmax=100
)


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator, FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable



import numpy as np
from scipy.optimize import least_squares
import matplotlib.pyplot as plt

import numpy as np
from scipy.signal import find_peaks, savgol_filter

def bump_n_val_segments_by_dips(lst, pspec, pspec2,
                          smooth_window=11,
                          polyorder=3,
                          dip_prominence=0.1,
                          dip_width=1.0,
                          frac=0.25):
    """
    Split (lst, pspec) into consecutive “bump” segments by finding dips
    (local minima) in the smoothed pspec curve, and also return the
    25%-of-peak-height valley region around each dip.

    Parameters
    ----------
    lst : 1d array, shape (N,)
        Monotonic LST axis.
    pspec : 1d array, shape (N,)
        Power-spectrum (or whatever) values.
    smooth_window : int
        Odd window length for Savitzky-Golay smoothing.
    polyorder : int
        Polynomial order for Savitzky-Golay filter.
    dip_prominence : float
        Minimum prominence of each dip (passed to `find_peaks` on `-pspec`).
    dip_width : float
        Minimum width of each dip (in samples).
    frac : float
        Fraction of each adjacent peak’s height (default 0.25) at which
        to cut off the valley region around each dip.

    Returns
    -------
    segments : list of (lst_seg, pspec_seg)
        Each entry is the full bump (from one dip to the next).
    dip_indices : 1d array
        The indices of the dips in the original arrays.
    valley_ranges : list of (lst_val, pspec_val)
        For each dip, the slice of (lst, pspec) between the two points
        where pspec rises back above `frac *` its neighboring‐peak height.
    """
    # 1) smooth
    psmooth = savgol_filter(pspec, smooth_window, polyorder)

    # 2) find dips
    dip_indices, _ = find_peaks(-psmooth,
                                prominence=dip_prominence,
                                width=dip_width)

    # 3) full‐bump boundaries
    bounds = np.concatenate(([0], dip_indices, [len(pspec)-1]))
    segments = []
    for i in range(len(bounds)-1):
        i0, i1 = bounds[i], bounds[i+1]
        segments.append((lst[i0:i1+1], pspec[i0:i1+1]))

    # 4) find all peaks in the smoothed curve
    peak_indices, _ = find_peaks(psmooth)

    # 5) for each dip, locate the valley region at 25% of its two flanking peaks
    valley_ranges = []
    valley_ranges2 = []
    for pd in dip_indices:
        # find nearest left/right peaks
        lefts  = peak_indices[peak_indices < pd]
        rights = peak_indices[peak_indices > pd]
        if len(lefts)==0 or len(rights)==0:
            continue
        pl, pr = lefts.max(), rights.min()

        # thresholds
        thr_l = frac * pspec[pl]
        thr_r = frac * pspec[pr]

        # find where it last exceeded thr_l before the dip
        left_ok = np.nonzero(pspec[:pd] > thr_l)[0]
        start = (left_ok.max()+1) if left_ok.size else 0

        # find where it first exceeds thr_r after the dip
        right_ok = np.nonzero(pspec[pd+1:] > thr_r)[0]
        end = (pd + right_ok.min()) if right_ok.size else len(pspec)-1

        valley_ranges.append((lst[start:end+1], pspec[start:end+1]))
        valley_ranges2.append((lst[start:end+1], pspec2[start:end+1]))

    return segments, dip_indices, valley_ranges, valley_ranges2


def piecewise_linear(params, x):
    """
    params = [m1, b1, m2, b2, x0]
    returns f(x) defined piecewise.
    """
    m1, b1, m2, b2, x0 = params
    return np.where(x <= x0,
                    m1 * x + b1,
                    m2 * x + b2)


def residuals(params, x, y):
    """Compute model minus data."""
    return piecewise_linear(params, x) - y

def fit_piecewise_linear(x, y, method='trf', manual_fallback=True):
    """
    Fit a two-segment piecewise linear model. By default uses TRF solver;
    if it fails and manual_fallback=True, falls back to grid search.
    Returns dict with slopes, intercepts, breakpoint, and diagnostics.
    """
    # initial guess & bounds
    p0 = [1.0, 0.0, 1.0, 0.0, np.median(x)]
    lower = [-np.inf, -np.inf, -np.inf, -np.inf, np.min(x)]
    upper = [ np.inf,  np.inf,  np.inf,  np.inf, np.max(x)]
    x_scale = np.array([1.0, 1.0, 1.0, 1.0, np.ptp(x)])
    diff_step = x_scale * 0.1

    try:
        result = least_squares(
            residuals, p0, args=(x, y), bounds=(lower, upper),
            method=method, x_scale=x_scale, diff_step=diff_step,
            ftol=1e-10, xtol=1e-10, gtol=1e-10
        )
        success = result.success
    except Exception:
        success = False
        result = None
        
    success=False

    if not success and manual_fallback:
        fit = manual_piecewise_linear_fit(x, y)
        fit['method'] = 'manual'
        return fit

    m1, b1, m2, b2, x0 = result.x
    return dict(
        m1=m1, b1=b1,
        m2=m2, b2=b2,
        x0=x0,
        success=result.success,
        cost=result.cost,
        nfev=result.nfev,
        message=result.message,
        method='trf'
    )

def manual_piecewise_linear_fit(x, y, num_breaks=200):
    """
    Brute-force grid search over breakpoints. Fits two segments via np.polyfit
    and selects breakpoint minimizing sum of squared errors.
    """
#     print("Doing manual")
    #print("x ", x)
    candidates = np.linspace(np.min(x), np.max(x), num_breaks)
    best = {'cost': np.inf}
    for x0 in candidates:
        # split
        left = x <= x0
        right = x > x0
        if np.sum(left) < 2 or np.sum(right) < 2:
            continue
        # fit segments
        m1, b1 = np.polyfit(x[left], y[left], 1)
        m2, b2 = np.polyfit(x[right], y[right], 1)
        # compute cost
        y_pred = np.empty_like(y)
        y_pred[left] = m1 * x[left] + b1
        y_pred[right] = m2 * x[right] + b2
        cost = np.sum((y_pred - y)**2)
        #print("cost ", cost)
        if cost < best['cost']:
            best.update(dict(m1=m1, b1=b1, m2=m2, b2=b2, x0=x0, cost=cost))
    return {**best, 'success': True, 'n_breaks': num_breaks}


def joint_piecewise_linear_fit(uvp_power_list, uvpspec_avg_power_list,
                               method='trf', manual_fallback=True):
    """
    Perform a single two‐segment piecewise‐linear fit to the union of
    all your P_coh vs P_inc data across N redundant‐group datasets.

    Parameters
    ----------
    uvp_power_list : list of 1D arrays
        Each entry is the delay‐0 coherent PSPEC for one redundant group.
    uvpspec_avg_power_list : list of 1D arrays
        Each entry is the delay‐0 incoherent PSPEC for the same group.
    method : str, optional
        Passed to `least_squares` ("trf", "dogbox", etc).
    manual_fallback : bool, optional
        If the solver fails, fall back on the brute‐force grid.

    Returns
    -------
    fit : dict
        Same keys as `fit_piecewise_linear`: 
        `{m1, b1, m2, b2, x0, success, cost, ...}` on the combined dataset.
    """
    # 1) Gather and mask each pair, log‐space
    xs, ys = [], []
    for pcoh, pinc in zip(uvp_power_list, uvpspec_avg_power_list):
        # only keep finite log‐points
        mask = np.isfinite(pcoh) & np.isfinite(pinc) & (pcoh>0) & (pinc>0)
        if not np.any(mask):
            continue
        xs.append(np.log10(pcoh[mask]))
        ys.append(np.log10(pinc[mask]))
    if len(xs) == 0:
        raise ValueError("No valid data points found in any group.")
    x_all = np.concatenate(xs)
    y_all = np.concatenate(ys)

    # 2) Call your existing fit
    return fit_piecewise_linear(x_all, y_all,
                                method=method,
                                manual_fallback=manual_fallback), x_all



def fit_linear_combo(y, fx, gx, snx=None):
    """
    Fit the model y ≈ a·f(x) + b·g(x) + c via ordinary least squares.

    Parameters
    ----------
    y  : array‐like, shape (N,)
        Observed dependent variable.
    fx : array‐like, shape (N,)
        First regressor f(x).
    gx : array‐like, shape (N,)
        Second regressor g(x).

    Returns
    -------
    a, b, c : floats
        Best‐fit coefficients such that y ≈ a·fx + b·gx + c.
    """
    y  = np.asarray(y,  float)
    fx = np.asarray(fx, float)
    gx = np.asarray(gx, float)
    
    # Build design matrix with columns [fx, gx, 1]
#     X = np.column_stack([fx, gx])#, np.ones_like(fx)])
    if snx is not None :
        snx = np.asarray(snx, float)
        X = np.column_stack([fx, gx, snx])#, np.ones_like(fx)])
    else:
        X = np.column_stack([fx, gx])#, np.ones_like(fx)])
    # Solve the normal equations: minimize ||X @ [a,b,c] – y||^2
#     (a, b, c), *_ = np.linalg.lstsq(X, y, rcond=None)
#     return a, b, c
    coefs = np.dot(np.linalg.pinv(X), y)
    return coefs



# Example usage:
# a, b, c = fit_linear_combo(y_array, f_of_x_array, g_of_x_array)

def predict_from_coeff(a, b, c, fx, gx):
# def predict_from_coeff(a, b, c, sn, fx, gx, snx):
    """
    Given fit coefficients a, b, c and your original regressors fx = f(x),
    gx = g(x), return the model prediction y_hat for each gx:
        y_hat = a*fx + b*gx + c

    Parameters
    ----------
    a, b, c : float
        Fitted coefficients.
    fx : array‐like, shape (N,)
        Values of f(x) at your sample points.
    gx : array‐like, shape (N,)
        Values of g(x) at your sample points.

    Returns
    -------
    y_hat : ndarray, shape (N,)
        The predicted y values, ready to plot versus gx.
    """
    fx = np.asarray(fx)
    gx = np.asarray(gx)
#     snx = np.asarray(snx)
    
    return a*fx + b*gx #+ c
#     return a*fx + b*gx + sn*snx #+ c


# Example usage:
# a, b, c = fit_linear_combo(y, fx, gx)
# y_pred = predict_from_g(a, b, c, fx, gx)
# plt.plot(gx, y_pred, '-', label='model vs g(x)')


def fit_joint_PN_linear_combo(uvp_power_list, uvpspec_avg_power_list, uvpspec_noise_list):
    """
    Fit the model y ≈ a·f(x) + b·g(x) + c via ordinary least squares.

    Parameters
    ----------
    y  : arraylike, shape (N,)
        Observed dependent variable.
    fx : arraylike, shape (N,)
        First regressor f(x).
    gx : arraylike, shape (N,)
        Second regressor g(x).

    Returns
    -------
    a, b, c : floats
        Bestfit coefficients such that y ≈ a·fx + b·gx + c.
    """
    Ys, PN_cols, coh_cols, mask = [], [], [], []
    for up, ua, un in zip(uvp_power_list, uvpspec_avg_power_list, uvpspec_noise_list):
        m = np.isfinite(ua)&np.isfinite(un)&np.isfinite(up)
        Ys.append(ua[m])
        PN_cols.append(un[m])
        # PN_cols.append(un[m]/un[m]) # No PN fit
        coh_cols.append(up[m])
        mask.append(m)
        
#     print("Ys shapes:", [arr.shape for arr in Ys])
#     print("PN_cols shapes:", [arr.shape for arr in PN_cols])
#     print("coh_cols shapes:", [arr.shape for arr in coh_cols])

    Y   = np.concatenate(Ys)
#     PNm = [np.concatenate(col) for col in PN_cols]
#     PNm = PN_cols
    C   = np.concatenate(coh_cols)
    
    # 3) Build “block‐diagonal” PN columns
    Ntot = Y.size
    X_cols = []
    start = 0
    for PN in PN_cols:
        Ni = PN.size
        col_full = np.zeros(Ntot, dtype=float)
        col_full[start:start+Ni] = PN
        X_cols.append(col_full)
        start += Ni

#     # design: [PN_0, PN_1, … , PN_{D-1},  C]
#     X = np.column_stack(PNm + [C])
    # 4) Now column‐stack all PN_i and the shared C
    X = np.column_stack(X_cols + [C])
    # solve for [Na_0 … Na_{D-1}, slopeloss]
    coefs    = np.linalg.pinv(X) @ Y
    Na_list  = coefs[:-1]
    slopeloss = coefs[-1]
    
    # predict & plot
    Y_pred = X @ coefs
    
    return Na_list, slopeloss, X, Y, Y_pred, mask

def fit_joint_PSN_linear_combo(uvp_power_list, uvpspec_avg_power_list, uvpspec_noise_list, uvpspec_signalnoise_list):
    """
    Fit the model y ≈ a·f(x) + b·g(x) + c via ordinary least squares.

    Parameters
    ----------
    y  : array‐like, shape (N,)
        Observed dependent variable.
    fx : array‐like, shape (N,)
        First regressor f(x).
    gx : array‐like, shape (N,)
        Second regressor g(x).

    Returns
    -------
    a, b, c : floats
        Best‐fit coefficients such that y ≈ a·fx + b·gx + c.
    """
    Ys, PN_cols, PSN_cols, coh_cols = [], [], [], []
    for up, ua, un, usn in zip(uvp_power_list, uvpspec_avg_power_list, uvpspec_noise_list, uvpspec_signalnoise_list):
        m = np.isfinite(ua)&np.isfinite(un)&np.isfinite(up)
        Ys.append(ua[m])
        PN_cols.append(un[m])
        PSN_cols.append(usn[m])
        coh_cols.append(up[m])
        
#     print("Ys shapes:", [arr.shape for arr in Ys])
#     print("PN_cols shapes:", [arr.shape for arr in PN_cols])
#     print("PSN_cols shapes:", [arr.shape for arr in PSN_cols])
#     print("coh_cols shapes:", [arr.shape for arr in coh_cols])

    Y   = np.concatenate(Ys)
#     PNm = [np.concatenate(col) for col in PN_cols]
#     PNm = PN_cols
    C   = np.concatenate(coh_cols)
    
    # 3) Build “block‐diagonal” PN columns
    Ntot = Y.size
    X_cols = []
    start = 0
    for PN in PN_cols:
        Ni = PN.size
        col_full = np.zeros(Ntot, dtype=float)
        col_full[start:start+Ni] = PN
        X_cols.append(col_full)
        start += Ni
        
    Ntot = Y.size
    XS_cols = []
    start = 0
    for PSN in PSN_cols:
        Ni = PSN.size
        col_full = np.zeros(Ntot, dtype=float)
        col_full[start:start+Ni] = PSN
        XS_cols.append(col_full)
        start += Ni

#     # design: [PN_0, PN_1, … , PN_{D-1},  C]
#     X = np.column_stack(PNm + [C])
    # 4) Now column‐stack all PN_i and the shared C
    X = np.column_stack(X_cols + XS_cols + [C])
    # solve for [Na_0 … Na_{D-1}, slopeloss]
    coefs    = np.linalg.pinv(X) @ Y
#     print("coefs shape ", coefs.shape)
    Na_list  = coefs[0:len(PN_cols)]
    Nb_list  = coefs[len(PN_cols):-1]
    slopeloss = coefs[-1]
    
    # predict & plot
    Y_pred = X @ coefs
    
    return Na_list, Nb_list, slopeloss, X, Y, Y_pred

def fit_joint_SNR_linear_combo(uvp_power_list, uvpspec_avg_power_list, uvpspec_SNR_list):
    """
    Fit the model y ≈ a·f(x) + b·g(x) + c via ordinary least squares.

    Parameters
    ----------
    y  : array‐like, shape (N,)
        Observed dependent variable.
    fx : array‐like, shape (N,)
        First regressor f(x).
    gx : array‐like, shape (N,)
        Second regressor g(x).

    Returns
    -------
    a, b, c : floats
        Best‐fit coefficients such that y ≈ a·fx + b·gx + c.
    """
    Ys, SNR_cols, coh_cols = [], [], []
    for up, ua, usn in zip(uvp_power_list, uvpspec_avg_power_list, uvpspec_SNR_list):
        m = np.isfinite(ua)&np.isfinite(usn)&np.isfinite(up)
        Ys.append(ua[m])
        SNR_cols.append(usn[m])
        # PN_cols.append(un[m]/un[m]) # No PN fit
        coh_cols.append(up[m])
        
#     print("Ys shapes:", [arr.shape for arr in Ys])
#     print("PN_cols shapes:", [arr.shape for arr in SNR_cols])
#     print("coh_cols shapes:", [arr.shape for arr in coh_cols])

    Y   = np.concatenate(Ys)
#     PNm = [np.concatenate(col) for col in PN_cols]
#     PNm = PN_cols
    C   = np.concatenate(coh_cols)
    
    # 3) Build “block‐diagonal” PN columns
    Ntot = Y.size
    X_cols = []
    start = 0
    for SNR in SNR_cols:
        Ni = SNR.size
        col_full = np.zeros(Ntot, dtype=float)
        col_full[start:start+Ni] = SNR
        X_cols.append(col_full)
        start += Ni

#     # design: [PN_0, PN_1, … , PN_{D-1},  C]
#     X = np.column_stack(PNm + [C])
    # 4) Now column‐stack all PN_i and the shared C
    X = np.column_stack(X_cols + [C])
    # solve for [Na_0 … Na_{D-1}, slopeloss]
    coefs    = np.linalg.pinv(X) @ Y
    SNRa_list  = coefs[:-1]
    slopeloss = coefs[-1]
    
    # predict & plot
    Y_pred = X @ coefs
    
    return SNRa_list, slopeloss, X, Y, Y_pred

def fit_joint_noPN_linear_combo(uvp_power_list, uvpspec_avg_power_list):
    """
    Fit the model y ≈ a·f(x) + b·g(x) + c via ordinary least squares.

    Parameters
    ----------
    y  : array‐like, shape (N,)
        Observed dependent variable.
    fx : array‐like, shape (N,)
        First regressor f(x).
    gx : array‐like, shape (N,)
        Second regressor g(x).

    Returns
    -------
    a, b, c : floats
        Best‐fit coefficients such that y ≈ a·fx + b·gx + c.
    """
    Ys, coh_cols = [], []
    for up, ua in zip(uvp_power_list, uvpspec_avg_power_list):
        m = np.isfinite(ua)&np.isfinite(up)
        Ys.append(ua[m])
        coh_cols.append(up[m])
        
#     print("Ys shapes:", [arr.shape for arr in Ys])
#     print("coh_cols shapes:", [arr.shape for arr in coh_cols])

    Y   = np.concatenate(Ys)
    C   = np.concatenate(coh_cols)


    X = np.column_stack([C])
    # solve for [Na_0 … Na_{D-1}, slopeloss]
    coefs    = np.linalg.pinv(X) @ Y
    slopeloss = coefs[-1]
    
    # predict & plot
    Y_pred = X @ coefs
    
    return slopeloss, X, Y, Y_pred



def compute_1hr_bin_average(lst_array, data_array):
    """
    Compute a 1-hour bin average of data_array centered around each LST value.
    """
    data_array_1hrbin = np.zeros_like(data_array)
    lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
    for i, lst in enumerate(lst_array_roll):
        # Find indices within ±0.5 hours of the current LST
        indices_in_bin = np.where((lst_array_roll >= lst - 0.5) & (lst_array_roll <= lst + 0.5))[0]
        # if( lst>0.001 and lst<0.002 ): 
            # print(" indices_in_bin ", lst_array[indices_in_bin] )
        if len(indices_in_bin) > 0:
            data_array_1hrbin[i] = np.nanmean(data_array[indices_in_bin])
        else:
            data_array_1hrbin[i] = np.nan
    return data_array_1hrbin

def get_spw_info(uvp):
    """
    Returns a dictionary mapping each SPW (from uvp.spw_array) to its (min, max) frequency range in Hz.
    """
    spw_indices = uvp.spw_array       # e.g., shape (14,)
    freq = uvp.freq_array             # e.g., shape (1114,)
    spw_freq = uvp.spw_freq_array     # e.g., shape (1114,)
    
    spw_info = {}
    for spw in spw_indices:
        mask = (spw_freq == spw)
        if np.any(mask):
            freq_min = np.min(freq[mask])
            freq_max = np.max(freq[mask])
            spw_info[spw] = (freq_min, freq_max)
        else:
            spw_info[spw] = None
    return spw_info

def get_plot_data(uvp, uvpspec_averaged, pol, idx):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    blp = uvp.get_blpairs()[0]
    # key = (idx, blp, 'xx')
    key = (idx, blp, pol)

    # Retrieve LST array and convert to hours.
    tarr = uvp.lst_avg_array
    tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
    lst_array_rad = tarr
#         print("lst_array_rad ", lst_array_rad)
    lst_array = tarrq

    dlys = uvp.get_dlys(idx) * 1e9
    index_of_zero = np.where(np.isclose(dlys, 0))[0][0]

    uvp_power = np.abs((uvp.get_data(key)))[:, index_of_zero]
    uvpspec_averaged_power = np.abs((uvpspec_averaged.get_data(key)))[:, index_of_zero]
    
    uvpspec_averaged_noise = np.abs(uvpspec_averaged.get_stats('autos_diag',key))[:, index_of_zero]
    
    uvp_power_S = (uvp.get_data(key))[:, index_of_zero].real

    percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
#     median_val = np.nanmedian(percent_diff)
#     median_vals.append(median_val)
#     mean_val = np.nanmean(percent_diff)
#     mean_vals.append(mean_val)

    lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        
    return dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, uvpspec_averaged_noise, uvp_power_S, percent_diff


def get_plot_data_all_blpairs(uvp, uvp_avg, pol, spw):
    """
    Gather τ≈0 spectra from *all* baseline–pairs, concatenate by LST,
    and compute percent-difference between uvp and its averaged version.

    Parameters
    ----------
    uvp, uvp_avg : UVPSpec
        Original and incoherently-averaged spectra.
    pol          : str              (e.g. 'xx' or 'pI')
    spw          : int              (spectral-window index)

    Returns
    -------
    dlys               : (Ndlys,)  delay values [ns]
    zero_idx           : int       index of τ≈0 in `dlys`
    lst_rad_sorted     : (Ntimes,) LST in radians, sorted
    lst_hr_sorted      : (Ntimes,) LST in hours,  "
    uvp_pow_sorted     : (Ntimes,) |P|  from uvp          (τ≈0)
    uvp_avg_pow_sorted : (Ntimes,) |P|  from uvp_avg      (τ≈0)
    pct_diff_sorted    : (Ntimes,) 100*(uvp-avg)/avg
    mean_pct           : float      mean of pct_diff
    median_pct         : float      median of pct_diff
    """
    LST_strt = 0
    Lst_end = 7
    # ---------- fixed per-spw info ----------
    dlys = uvp.get_dlys(spw) * 1e9          # ns
    zero_idx = int(np.where(np.isclose(dlys, 0))[0][0])

    # ---------- gather blocks ----------
    lst_list = []
    uvp_pow_list = []
    uvp_avg_pow_list = []
    uvp_avg_noise_list = []
    uvp_power_S_list = []
    uvp_SNR_list = []

    for blp in uvp.get_blpairs():
        key = (spw, blp, pol)
        print("key ", key)

        # LST (radians) and convert now (same for both uvp and uvp_avg)
        lst_block = uvp.lst_avg_array[uvp.blpair_to_indices(blp)]
        lst_list.append(lst_block)

        # spectra, pick τ≈0 and |.| for power
        uvp_pow_block     = np.abs((uvp        .get_data(key)))[:, zero_idx]
#         print("uvp_pow_block ", uvp_pow_block)
        uvp_avg_pow_block = np.abs((uvp_avg    .get_data(key)))[:, zero_idx]
#         print("uvp_avg_pow_block ", uvp_avg_pow_block)
        uvp_avg_noise_block = np.abs(uvp_avg.get_stats('autos_diag',key))[:, zero_idx]
        uvp_power_S_block = (uvp.get_data(key))[:, zero_idx].real        
        uvp_SNR_block =  np.abs(uvp.get_data(key) / uvp_avg.get_stats('autos_diag', key).real )[:, zero_idx] 
        print("zero_idx ", zero_idx)
        print("uvp.get_data(key) ", uvp.get_data(key) )
        print("uvp_avg.get_stats('autos_diag', key).real ", uvp_avg.get_stats('autos_diag', key).real[:, zero_idx]  )
        print("uvp_SNR_block ", uvp_SNR_block )

        uvp_pow_list    .append(uvp_pow_block)
        uvp_avg_pow_list.append(uvp_avg_pow_block)
        uvp_avg_noise_list.append(uvp_avg_noise_block)
        uvp_power_S_list.append(uvp_power_S_block)
        uvp_SNR_list.append(uvp_SNR_block)

    # ---------- concatenate and sort by LST ----------
    lst_all         = np.concatenate(lst_list)
    uvp_pow_all     = np.concatenate(uvp_pow_list)
    uvp_avg_pow_all = np.concatenate(uvp_avg_pow_list)
    uvp_avg_noise_all = np.concatenate(uvp_avg_noise_list)
    uvp_power_S_all = np.concatenate(uvp_power_S_list)
    uvp_SNR_all = np.concatenate(uvp_SNR_list)

    order           = np.argsort(lst_all)
    lst_rad_sorted  = lst_all        [order]
    uvp_pow_sorted  = uvp_pow_all    [order]
    uvp_avg_sorted  = uvp_avg_pow_all[order]
    uvp_avg_noise_sorted  = uvp_avg_noise_all[order]
    uvp_power_S_sorted  = uvp_power_S_all[order]
    uvp_SNR_sorted  = uvp_SNR_all[order]
    
    print("uvp_SNR_sorted ", len(uvp_SNR_sorted), uvp_SNR_sorted)

    # ---------- compute percent difference ----------
    pct_diff_sorted = 100.0 * (uvp_pow_sorted - uvp_avg_sorted) / np.where(
                         uvp_avg_sorted != 0, uvp_avg_sorted, np.nan
                     )

    mean_pct   = np.nanmean(pct_diff_sorted)
    median_pct = np.nanmedian(pct_diff_sorted)

    # LST in hours
    lst_hr_sorted = lst_rad_sorted * (12 / np.pi)
    
    print("lst_hr_sorted ", len(lst_hr_sorted), lst_hr_sorted)
    LST_cutndx = np.where( (lst_hr_sorted>LST_strt) & (lst_hr_sorted<Lst_end) ) 
    print("lst_hr_sorted cut ", len(lst_hr_sorted[LST_cutndx]), lst_hr_sorted[LST_cutndx] )
    
    lst_array_roll = np.where(lst_hr_sorted > 20, lst_hr_sorted - 24, lst_hr_sorted)

    return (dlys, zero_idx,
            lst_rad_sorted[LST_cutndx], lst_hr_sorted[LST_cutndx], lst_array_roll[LST_cutndx],
            uvp_pow_sorted[LST_cutndx], uvp_avg_sorted[LST_cutndx], uvp_avg_noise_sorted[LST_cutndx], uvp_power_S_sorted[LST_cutndx], uvp_SNR_sorted[LST_cutndx],
            pct_diff_sorted[LST_cutndx], 
           )

import numpy as np

def goodness_of_fit(X, Y, Y_pred):
    """
    Compute goodness-of-fit metrics between data Y and predictions Y_pred.

    Parameters
    ----------
    Y       : array-like, shape (N,)
        Observed dependent variable.
    Y_pred  : array-like, shape (N,)
        Model predictions.

    Returns
    -------
    metrics : dict
        A dictionary containing:
         - residuals : Y - Y_pred
         - SSE       : Sum of squared errors
         - MSE       : Mean squared error
         - RMSE      : Root mean squared error
         - MAE       : Mean absolute error
         - R2        : Coefficient of determination
    """
    Y  = np.asarray(Y,  float)
    Yp = np.asarray(Y_pred, float)
    if Y.shape != Yp.shape:
        raise ValueError("Y and Y_pred must have the same shape")

    # residuals
    res = Y - Yp
#     print("Y ", Y)
#     print("Yp ", Yp)
#     print("res ", res)

    # sum of squares
    SSE = np.sum(res**2)                     # sum squared errors
#     print("SSE ", SSE)
    SST = np.sum((Y - np.mean(Y))**2)        # total sum of squares

    # basic metrics
    MSE  = SSE / Y.size
    RMSE = np.sqrt(MSE)
    MAE  = np.mean(np.abs(res))

    # R-squared
    R2 = 1 - SSE/SST if SST != 0 else np.nan
    
    # estimate variance and covariance
    N, p = X.shape
#     print("N, p ", N, p)
    MSE_samp = SSE / (N - p)
#     print("MSE_samp ", MSE_samp)
    XtX = X.T @ X
    inv_XtX = np.linalg.pinv(XtX)
#     print("inv_XtX ", inv_XtX)
    cov_beta = MSE_samp * inv_XtX

    return {
        'residuals': res,
        'SSE':    SSE,
        'MSE':    MSE,
        'RMSE':   RMSE,
        'MAE':    MAE,
        'R2':     R2,
        'MSE_samp': MSE_samp
    }, cov_beta


# median_vals = []
# median_vals_err = []
# median_vals_mse = []

noPN_median_vals = []
noPN_median_vals_err = []
noPN_median_vals_mse = []

psn_median_vals = []
psn_median_vals_err = []
psn_median_vals_mse = []

snr_median_vals = []
snr_median_vals_err = []
snr_median_vals_mse = []

mean_vals = []



def plot_percent_difference_with_1hrbin(group,lst_array, pol, uvp, uvpspec_averaged,
                                        zero_delay_pspec_spw=None, avg_pspec=None,
                                        joint_fit=False):
    """
    It's a mess but it works
    """
    reds = [
    "#F08080",  # LightCoral: a soft, pastel red :contentReference[oaicite:0]{index=0}
    "#FA8072",  # Salmon: a mildly orange-tinged red :contentReference[oaicite:1]{index=1}
    "#E9967A",  # DarkSalmon: deeper, muted salmon red :contentReference[oaicite:2]{index=2}
    "#CD5C5C",  # IndianRed: warm, brick-like red :contentReference[oaicite:3]{index=3}
    "#DC143C",  # Crimson: vivid, slightly bluish red :contentReference[oaicite:4]{index=4}
    "#FF0000",  # Red: pure, primary web red :contentReference[oaicite:5]{index=5}
    "#FF4500",  # OrangeRed: intense red with an orange tint :contentReference[oaicite:6]{index=6}
    "#B22222",  # FireBrick: dark, brownish-red :contentReference[oaicite:7]{index=7}
    "#8B0000",  # DarkRed: very deep, almost maroon red :contentReference[oaicite:8]{index=8}
    "#800000",  # Maroon: classic dark wine-red :contentReference[oaicite:9]{index=9}
    ]
    
    # Retrieve SPW info from the UVP object
    spw_info = get_spw_info(combined_uvp_dict[(bl_len[0],bl_ang[0],pol[0])])
    
    num_spws = len(spw_info)
    fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharex=False)
    axes = axes.flatten(order='C')
    
    # Create residual axes appended below each main axis
    residual_axes = []
    for ax in axes:
        divider = make_axes_locatable(ax)
        ax_res = divider.append_axes("bottom", size="30%", pad=0.6)
        residual_axes.append(ax_res)
        ax.tick_params(labelbottom=True)
    
    # --- Baseline Information ---
    bl_vec = combined_uvp_dict[(bl_len[0],bl_ang[0], pol[0])].bl_vecs[0]  # using the first baseline vector
    baseline_length = np.linalg.norm(bl_vec)
    angle_rad = np.arctan2(bl_vec[1], bl_vec[0])
    baseline_angle_deg = np.degrees(angle_rad)
    forced_angle = 0  # Force to 0° if desired
    
    median_vals = []
    median_vals_err = []
    median_vals_mse = []
    minmax_SNR = []
    
    # --- Loop Over SPWs ---
    # Iterate over sorted spw_info items (by the key)
    valid_spw = [0]#,1,2,4,5,8,9,12]
    ax_counter=-1
    for idx, (spw_key, spw_val) in enumerate(sorted(spw_info.items(), key=lambda x: x[0])):
        if idx in valid_spw:
            print("idx ", idx)
            ax_counter=ax_counter+1
            if spw_val is not None:
                freq_min, freq_max = spw_val
                freq_range_str = f"{freq_min/1e6:.2f}-{freq_max/1e6:.2f} MHz"
            else:
                freq_range_str = "N/A"

#             pol='yy'
#             pol='pI'
            # Data retrieval using a baseline pair (adjust as needed)
#             blp = combined_uvp_dict[(bl_len[i],bl_ang[i])].get_blpairs()[0]
#             key = (idx, blp, pol)


#             for i in range(1):#len(bl_len)):
#                 dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, uvpspec_averaged_noise, uvp_power_S, percent_diff = get_plot_data(combined_uvp_dict[bl_len[i]], combined_uvp_avg_dict[bl_len[i]], pol, idx)    
#             P_SN = np.sqrt(np.sqrt(2) * uvp_power_S * np.sqrt(77)*uvpspec_averaged_noise + (np.sqrt(77)*uvpspec_averaged_noise)**2)

            group_Nbls = [77,56,58, 51]

            uvp_power_list = []
            uvpspec_avg_power_list = []
            uvpspec_noise_list = []
            uvp_power_S_list = []
            lst_roll_list = []
            uvpspec_signalnoise_list = []
            uvpspec_SNR_list = []
            
            for i in range(1): #len(bl_len)):
                *_, lst_r, up, ua, un, ups, usnr, _ = get_plot_data_all_blpairs(
#                     combined_uvp_dict[bl_len[i]],
#                     combined_uvp_avg_dict[bl_len[i]],
                    uvp,
                    uvpspec_averaged,
                    pol[0], idx
                )
                print("usnr ", len(usnr), usnr)
                uvp_power_list.append(up)
                uvpspec_avg_power_list.append(ua)
                uvpspec_noise_list.append(un)
                uvp_power_S_list.append(ups)
                lst_roll_list.append(lst_r)
    #             uvpspec_signalnoise_list.append( (np.sqrt(np.sqrt(2) * ups * np.sqrt(77)*un + (np.sqrt(group_Nbls[i])*un)**2))  )
                uvpspec_signalnoise_list.append( (np.sqrt(np.sqrt(2) * ups * un + (un)**2))  )
                uvpspec_SNR_list.append( usnr )                                   # SNRs.append(np.ravel(( uvp.get_data(key) / uvp.get_stats('P_N', key).real )[:, high_dlys]))
            
            print("uvpspec_SNR_list ", uvpspec_SNR_list[0])
            idx_min = np.array(uvpspec_SNR_list[0]).argmin()          # or np.argmin(arr)
            print("idx_min ", idx_min)
            print("min SNR and LST ", uvpspec_SNR_list[0][idx_min], lst_roll_list[0][idx_min])
            idx_max = np.array(uvpspec_SNR_list[0]).argmax()          # or np.argmin(arr)
            print("max SNR and LST ", uvpspec_SNR_list[0][idx_max], lst_roll_list[0][idx_max])
            
            minmax_SNR.append([(f"{bl_len[group]}m {bl_ang[group]}° {idx}spw: "),('min SNR[LST]', uvpspec_SNR_list[0][idx_min], lst_roll_list[0][idx_min]),('max SNR[LST]', uvpspec_SNR_list[0][idx_max], lst_roll_list[0][idx_max])])


            segs = []
            dips = []
            valleys = []
            valleys2 = []
            for i in range(1):#len(bl_len)):            
                # Valleys for threshold to peak range
                s, d, v1, v2 = bump_n_val_segments_by_dips(
                    lst_roll_list[i], uvp_power_list[i], uvpspec_avg_power_list[i],
                    smooth_window=51, polyorder=3,
                    dip_prominence=1e10, dip_width=5,
                    frac=0.05
                )
                segs.append(s)
                dips.append(d)
                valleys.append(v1)
                valleys2.append(v2)


            # Group axes: use 2 main axes per SPW (and 2 corresponding residual axes)
            a = 4 * (ax_counter // 2) + (ax_counter % 2)
            b = a + 2
    #         print((a, b))
    #         ax_group = axes[idx:idx+1] 
            ax_group = axes[a:b+1] # axes[2*idx:(2*idx)+2]
    #         ax_res_group = residual_axes[idx:idx+1] 
            ax_res_group = residual_axes[a:b+1] # residual_axes[2*idx:(2*idx)+2]

            # --- Main Scatter Plots ---

            #Plot Data
            plot_data = True
            if plot_data :
                for i in range(1):#len(bl_len)):
                    linestyle=['-', '--', ':']
            #         ax_group[-1].plot(lst_array_roll, uvp_power, linewidth=1, linestyle="-", color="blue", alpha=0.5,
            #                    label=f"P_coh SPW {freq_range_str}")
                    print("uvpspec_noise_list[i] ", uvpspec_noise_list[i])
                    ax_group[-1].plot(lst_roll_list[i], (10**i)*uvpspec_noise_list[i], linewidth=1, 
                                      linestyle=linestyle[0], 
                                      color=reds[i],
                                      label=f"{(10**i)}P_N {bl_len[i]}m {bl_ang[i]}°",
                                      alpha=0.9)
            #         ax_group[-1].plot(lst_array_roll, P_SN, linewidth=0.5, linestyle="-", color="blue",
            #                    label=f"P_SN SPW {freq_range_str}")
                    from itertools import cycle
                    color_cycle = cycle(['#1f77b4','#ff7f0e','#2ca02c','#d62728'])  # any palette
                    style_cycle = cycle([':','--'])  # any palette
                    cmaps = [plt.cm.viridis, plt.cm.plasma, plt.cm.ocean, plt.cm.terrain]
                    n = len(segs[i])+1

                    for j,(lst_seg,ps_seg) in enumerate(segs[i]):
            #             print(f"--- bump #{i} spans LST = {lst_seg[0]:.2f} → {lst_seg[-1]:.2f}")
                        if(j==0):
                            ax_group[-1].plot(lst_seg, (10**i)*ps_seg, linestyle=next(style_cycle), #"-", 
                                              color=cmaps[i](j/(n-1)), #next(color_cycle),
                                              label=f"{(10**i)}P_coh {bl_len[i]}m {bl_ang[i]}°",
                                              linewidth=1.5,
                                              alpha=0.9)
                        else:
                            ax_group[-1].plot(lst_seg, (10**i)*ps_seg, linestyle=next(style_cycle), #"-", 
                                              color=cmaps[i](j/(n-1)), #next(color_cycle),
                                              linewidth=1.5,
                                              alpha=0.9)
        #             for j, (lseg, pseg) in enumerate(valleys[i]):
        #                 ax_group[-1].plot(lseg, pseg, linestyle="-", color='orange',
        #                            alpha=1)

            plot_SNR = True
            if plot_SNR:
                cmaps = ['viridis', 'plasma', 'ocean', 'terrain']
                for i in range(1): #len(bl_len)):            
    #                 scatter_res2 = ax_group[-1].scatter((uvp_power_list[i]), (uvp_power_list[i]/uvpspec_avg_power_list[i]),
    #                                                    s=10, marker=".", linestyle="-",
    #                                                    label=f" P_coh/P_inc {bl_len[i]}m {bl_ang[i]}°",
    # #                                                    color='royalblue',
    #                                                    c=lst_roll_list[i],
    #                                                    cmap=cmaps[i], 
    #                                                    alpha=0.2)

                    scatter_res2 = ax_group[-1].scatter( lst_roll_list[i], uvpspec_SNR_list[i],
                                                       s=10, marker=".", linestyle="-",
                                                       label=f"SNR {bl_len[i]}m {bl_ang[i]}°",
                                                       color='royalblue',
#                                                        c=lst_roll_list[i],
#                                                        cmap=cmaps[i], 
                                                       alpha=0.2)

#                     scatter_res2 = ax_group[-1].scatter((uvpspec_SNR_list[i]), (uvp_power_list[i]/uvpspec_avg_power_list[i]),
#                                                        s=10, marker=".", linestyle="-",
#                                                        label=f" P_coh/P_inc {bl_len[i]}m {bl_ang[i]}°",
#     #                                                    color='royalblue',
#                                                        c=lst_roll_list[i],
#                                                        cmap=cmaps[i], 
#                                                        alpha=0.2)

                    ax_group[-1].axhline(1, 
                                         linewidth=2,
                                         color='black',
                                         label=f"1"
                                        ) 

            # SCATTER PLOT WITH FIT

            #_____2-LINE FIT_______________________________________________________________________________________________________________________________________________________________________________
            twoline_fit=True   
            if twoline_fit:       
    #             mask = ~ (np.isneginf(np.log10(uvp_power)) | np.isneginf(np.log10(uvpspec_averaged_power)))
    #             xin = np.log10(uvp_power)[mask]
    #             yin = np.log10(uvpspec_averaged_power)[mask]
    #     #         print("inputs ", xin, yin)
    #             fit = fit_piecewise_linear(xin, yin)
    #     #         print(fit.keys() )
    #     #         print(f"Method: {fit['method']}")
    #     #         print(f"m1={fit['m1']:.3f}, b1={fit['b1']:.3f}")
    #     #         print(f"m2={fit['m2']:.3f}, b2={fit['b2']:.3f}")
    #     #         print(f"x0={fit['x0']:.3f}, cost={fit.get('cost',np.nan):.3f}")

    #             m1, b1 = fit['m1'], fit['b1']
    #             m2, b2 = fit['m2'], fit['b2']
    #             x0 = fit['x0']
    #             print(f"SPW {idx}: 2-line fit Loss {-1*(1-m2)*100:.2f}%")

    #             left_mask = xin < x0
    #             n_left   = np.count_nonzero(left_mask)
    #             n_total  = xin.size
    #             fraction = n_left / n_total

    #     #         print(f"{n_left} points are to the left of x0 = {x0:.3f}, out of {n_total} total")
    #     #         print(f"Fraction left of breakpoint = {fraction:.2%}")

    #             inp_x = xin #np.log10(uvp_power)
    #             x_min, x_max = np.min(inp_x), np.max(inp_x)
    #             x1 = np.linspace(x_min, x0, 200)
    #             x2 = np.linspace(x0, x_max, 200)
    #             ax_group[0].plot(x1, m1*x1 + b1, linewidth=1, label=f'Segment 1 {n_left} LSTs', color='blue', alpha=0.7 )
    #             ax_group[0].plot(x2, m2*x2 + b2, linewidth=1, label=f'Segment 2 Loss {(1-m2)*100:.2f} %', color='blue', alpha=0.7 )
    #             ax_group[0].axvline(x0, linestyle='--', label=f'Inflection, divergent {n_left} LSTs') # x0={x0:.2f}')


                fit, xin = joint_piecewise_linear_fit(uvp_power_list, uvpspec_avg_power_list)

    #             print("Joint fit results:")
    #             print(f"  Segment 1: slope = {fit['m1']:.3f}, intercept = {fit['b1']:.3f}")
    #             print(f"  Segment 2: slope = {fit['m2']:.3f}, intercept = {fit['b2']:.3f}")
    #             print(f"  Breakpoint x0 = {fit['x0']:.3f}")

                m1, b1 = fit['m1'], fit['b1']
                m2, b2 = fit['m2'], fit['b2']
                x0 = fit['x0']
    #             print(f"SPW {idx}: 2-line fit Loss {-1*(1-m2)*100:.2f}%")

    #             median_vals.append((1-m2)*100)

                left_mask = xin < x0
                n_left   = np.count_nonzero(left_mask)
                n_total  = xin.size
                fraction = n_left / n_total

                inp_x = xin #np.log10(uvp_power)
                x_min, x_max = np.min(inp_x), np.max(inp_x)
                x1 = np.linspace(x_min, x0, 200)
                x2 = np.linspace(x0, x_max, 200)
                ax_res_group[0].plot(x1, (m1-1)*x1 + b1, linewidth=1, label=f'Segment 1 {n_left} LSTs', color='blue', alpha=0.7 )
                ax_res_group[0].plot(x2, (m2-1)*x2 + b2, linewidth=1, label=f'Segment 2 Loss {(1-m2)*100:.2f} %', color='blue', alpha=0.7 )
                ax_res_group[0].axvline(x0, linestyle='--', label=f'Inflection, divergent {n_left} LSTs') # x0={x0:.2f}')





            #_____SINGLE FIT_______________________________________________________________________________________________________________________________________________________________________________
            single_fit=False   
            if single_fit:
                NaN_mask = np.isfinite(uvpspec_averaged_power) & np.isfinite(uvpspec_averaged_noise) & np.isfinite(uvp_power)
                pinc = uvpspec_averaged_power[NaN_mask]
                P_N = uvpspec_averaged_noise[NaN_mask]
                pcoh = uvp_power[NaN_mask]
                P_SN_in = P_SN[NaN_mask]
                cond = np.linalg.cond(np.column_stack([P_N, pcoh]))
        #         print("Condition number:", cond)
                c=1

                na, slopeloss = fit_linear_combo(y=pinc, fx=P_N, gx=pcoh)
        #         na, slopeloss, sn = fit_linear_combo(pinc, P_N, pcoh, P_SN_in)

    #             print("P_N fit N_a, P_slope ", na, slopeloss)
        #         print("P_N fit N_a, P_slope ", na, slopeloss, sn)

    #             print(f"P_N fit loss {-1*(1-(1/slopeloss))*100:.2f} %")

                pinc_pred = predict_from_coeff(na, slopeloss, c, P_N, pcoh)
        #         pinc_pred = predict_from_coeff(na, slopeloss, c, sn, P_N, pcoh, P_SN_in)

        #         scatter1 = ax_group[0].scatter(np.log10(pcoh), np.log10(pinc_pred), s=2, linewidth=1, label=f'P_N Fit slope={slopeloss:.2f}, loss={(slopeloss-1)*100:.2f}%', color='red', alpha=0.6,) 
        #                             c=lst_array_roll[NaN_mask], cmap='viridis' )
                scatter2 = ax_group[0].scatter(np.log10(pinc), np.log10(pinc_pred), s=2, linewidth=1, label=f'Pinc_data vs Pinc_fit slope={slopeloss:.2f}, loss={(slopeloss-1)*100:.2f}%',  alpha=0.7, 
                                    c=lst_array_roll[NaN_mask], cmap='viridis' )

                cbar = plt.colorbar(scatter2)#, ticks=np.arange(12))
                cbar.set_label('LST')

                ax_group[-1].plot(lst_array_roll[NaN_mask], pcoh, linewidth=0.5, linestyle="-", color="red",
                           label=f"Mask Pcoh SPW {freq_range_str}")


            #_____JOINT FIT_______________________________________________________________________________________________________________________________________________________________________________
            # --- build per-dataset lists for either joint or single fit ---
            joint_fit=True   
            if joint_fit:

                # NO PN
                model_no_PN = False
                if model_no_PN:
                    nopn_slopeloss_joint, X_noPN, nopn_Y, nopn_Y_pred = fit_joint_noPN_linear_combo(uvp_power_list, uvpspec_avg_power_list)
                    print("recon shape nopn_slopeloss_joint, X_noPN, nopn_Y ", nopn_slopeloss_joint.shape, X_noPN.shape, nopn_Y.shape)
                    print(X_noPN[:,0].shape)
                    print(X_noPN[500:600,0])
                    print(nopn_Y)

                    metrics_with_noPN, noPN_cov_beta = goodness_of_fit(X_noPN, nopn_Y, nopn_Y_pred)
                    noPN_median_vals_err.append(np.sqrt(noPN_cov_beta[-1,-1]) )
                    noPN_median_vals_mse.append( metrics_with_noPN['MSE_samp'] )

    #                 print(">>> no PN JOINT FIT slopeloss:", nopn_slopeloss_joint)

    #                 noPN_median_vals.append((1-1/nopn_slopeloss_joint)*100)
    #                 ax_group[0].scatter(np.log10(nopn_Y), np.log10(nopn_Y_pred),
    #                                     s=2, label=f'No PN all group fit slope={nopn_slopeloss_joint:.2f}, loss={(1-1/nopn_slopeloss_joint)*100:.2f}%',  alpha=0.2, 
    #                                     color='red'
    #                                    )

                    ax_group[0].scatter(np.log10(X_noPN[:,0]), ( X_noPN[:,0]/nopn_Y ),
                                                   s=10, marker=".", linestyle="-",
                                                   label=f" noPN P_coh/P_inc(fit) All Redgrp",
                                                   color='royalblue',
    #                                                c=lst_roll_list[i],
    #                                                cmap=cmaps[i], 
                                                   alpha=0.2)

                    if joint_fit:
                        ax_group[0].axhline(1/nopn_slopeloss_joint, 
                                         linewidth=4,
                                         color='royalblue',
                                         label=f"All Group noPN Fit Loss Ratio = {1/nopn_slopeloss_joint:.2f}"
                                        ) 

                # PN
                model_PN = True
                if model_PN:
                    Na_list, slopeloss_joint, X_PN, Y, Y_pred, mask_PN = fit_joint_PN_linear_combo(uvp_power_list, uvpspec_avg_power_list, uvpspec_noise_list)
                    print("recon shape Na_list, slopeloss_joint, X_PN ", Na_list.shape, slopeloss_joint.shape, X_PN.shape)

                    metrics_with_PN, PN_cov_beta = goodness_of_fit(X_PN, Y, Y_pred)
                    median_vals_err.append(np.sqrt(PN_cov_beta[-1,-1]) )
                    median_vals_mse.append( metrics_with_PN['MSE_samp'] )
    #                 print("PN_cov_beta ", PN_cov_beta)
    #                 print("Joint PN fit R²:", metrics_with_PN['R2'])
    #                 print("Joint PN fit RMSE:", metrics_with_PN['RMSE'])

    #                 print(">>> PN JOINT FIT slopeloss:", slopeloss_joint)
    #                 for i, Na in enumerate(Na_list):
    #                     print(f"    dataset {i} → N_a = {Na:.3f}")

    #                 median_vals.append((1-1/slopeloss_joint)*100)
    #                 ax_group[0].scatter(np.log10(Y), np.log10(Y_pred),
    #                                     s=2, label=f'PN all group fit slope={slopeloss_joint:.2f}, loss={(1-1/slopeloss_joint)*100:.2f}%',  alpha=0.7, 
    #                                     color='royalblue'
    #                                    )

                    median_vals.append((1-1/slopeloss_joint)*100)
                    print("PN loss ", (1-1/slopeloss_joint)*100)
                    SNR_flatten = np.concatenate(uvpspec_SNR_list)
                    mask_flatten = np.concatenate(mask_PN)
                    lst_roll_list_flatten = lst_roll_list[i]

                    ax_group[0].scatter( 
    #                                     (X_PN[:,3]),
                                         np.log10(SNR_flatten[mask_flatten]),
                                        ( X_PN[:,-1]/(Y-(X_PN[:,0:-1]@Na_list)) ),
                                        # ( X_PN[:,-1]/(Y) ),
                                           s=10, marker=".", linestyle="-",
                                           label=f" PN P_coh/P_inc(fit) All Redgrp",
    #                                        color='red',
                                           c=lst_roll_list_flatten[mask_flatten],
                                           cmap=cmaps[i], 
                                           alpha=0.2)

                    if joint_fit:
                        ax_group[0].axhline(1/slopeloss_joint, 
                                         linewidth=4,
                                         color='red',
                                         label=f"All Group PN Fit Loss Ratio = {1/slopeloss_joint:.2f}"
                                        ) 

                        ax_group[0].axhline(1, 
                                         linewidth=2,
                                         color='black',
                                         label=f"1"
                                        ) 

                # PN + PSN
                model_PN_PSN = False
                if model_PN_PSN:
                    NNa_list, SNa_list, psn_slopeloss_joint, X_PSN, psn_Y, psn_Y_pred = fit_joint_PSN_linear_combo(uvp_power_list, uvpspec_avg_power_list, uvpspec_noise_list, uvpspec_signalnoise_list)

                    metrics_with_PSN, PSN_cov_beta = goodness_of_fit(X_PSN, psn_Y, psn_Y_pred)
                    psn_median_vals_err.append(np.sqrt(PSN_cov_beta[-1,-1]) )
                    psn_median_vals_mse.append( metrics_with_PSN['MSE_samp'] )
    #                 print("PSN_cov_beta ", PSN_cov_beta)

    #                 print(">>> PN+PSN JOINT FIT slopeloss:", psn_slopeloss_joint)
    #                 print("SNa_list ", SNa_list)
    #                 for i, Na in enumerate(SNa_list):
    #                     print(f"    dataset {i} → N_b = {Na:.3f}")

                    psn_median_vals.append((1-1/psn_slopeloss_joint)*100)
                    ax_group[0].scatter(np.log10(psn_Y), np.log10(psn_Y_pred),
                                        s=2, label=f'PN+PSN all group fit slope={psn_slopeloss_joint:.2f}, loss={(1-1/psn_slopeloss_joint)*100:.2f}%',  alpha=0.2, 
                                        color='green'
                                       )

                # SNR 
                model_SNR = False
                if model_SNR:
                    SNRa_list, snr_slopeloss_joint, X_SNR, snr_Y, snr_Y_pred = fit_joint_SNR_linear_combo(uvp_power_list, uvpspec_avg_power_list, uvpspec_SNR_list)

                    metrics_with_SNR, SNR_cov_beta = goodness_of_fit(X_SNR, snr_Y, snr_Y_pred)
                    snr_median_vals_err.append(np.sqrt(SNR_cov_beta[-1,-1]) )
                    snr_median_vals_mse.append( metrics_with_SNR['MSE_samp'] )
    #                 print("PSN_cov_beta ", SNR_cov_beta)

    #                 print(">>> PN+PSN JOINT FIT slopeloss:", snr_slopeloss_joint)
    #                 print("SNa_list ", SNRa_list)
    #                 for i, Na in enumerate(SNRa_list):
    #                     print(f"    dataset {i} → N_b = {Na:.3f}")

                    snr_median_vals.append((1-1/snr_slopeloss_joint)*100)
                    ax_group[0].scatter(np.log10(snr_Y), np.log10(snr_Y_pred),
                                        s=2, label=f'SNR all group fit slope={snr_slopeloss_joint:.2f}, loss={(1-1/snr_slopeloss_joint)*100:.2f}%',  alpha=0.2, 
                                        color='orange'
                                       )

                # PN Convolved
                model_PN_2 = False
                if model_no_PN:
                    placeholder = []


    #         left_mask = np.log10(pcoh) < x0
    #         n_left   = np.count_nonzero(left_mask)
    #         ax_group[0].axvline(x0, linestyle='--', label=f'Inflection, divergent {n_left} LSTs') # x0={x0:.2f}')


    #         cbar1 = fig.colorbar(scatter1, ax=ax_group[0])
    #         cbar1.set_label("LST (Hr)", fontsize=12)


    #         scatter2 = ax_group[-1].scatter(np.log10(uvp_power_1hrbin), np.log10(uvpspec_averaged_power_1hrbin),
    #                                        s=20, marker=".", linestyle="-",
    #                                        label=f"1Hr Coh vs 1Hr Incoh PSPEC\n SPW {freq_range_str}",
    #                                        c=lst_array, cmap='OrRd', alpha=0.7)
    #         cbar2 = fig.colorbar(scatter2, ax=ax_group[-1])
    #         cbar2.set_label("LST (Hr)", fontsize=12)

            # --- Residual Scatter Plots ---
            cmaps = ['viridis', 'plasma', 'ocean', 'terrain']
            for i in range(1):#len(bl_len)):            
                scatter_res1 = ax_res_group[0].scatter(np.log10(uvp_power_list[i]), np.log10(uvpspec_avg_power_list[i])-np.log10(uvp_power_list[i]),
                                                       s=10, marker=".", linestyle="-",
                                                       label=f" P_coh vs P_inc-P_coh {bl_len[i]}m {bl_ang[i]}°",
                                                       c=lst_roll_list[i],
                                                       cmap=cmaps[i], 
                                                       alpha=0.2
                                                      )
    #         scatter_res2 = ax_res_group[-1].scatter(np.log10(uvp_power_1hrbin), np.log10(uvpspec_averaged_power_1hrbin)-np.log10(uvp_power_1hrbin),
    #                                                s=20, marker=".", linestyle="-",
    #                                                label=f"Residual 1hr: Coh vs Incoh\n SPW {freq_range_str}",
    #                                                c=lst_array, cmap='OrRd', alpha=0.7)

            if joint_fit:
                # PN
                model_PN = True
                if model_PN:
                    ax_res_group[-1].axhline(1/slopeloss_joint, 
                                    linewidth=4,
                                    color='royalblue',
                                    label=f"All Group Fit Loss Ratio = {1/slopeloss_joint:.2f}"
                                    ) 
                model_no_PN = False
                if model_no_PN:
                    ax_res_group[-1].axhline(1/nopn_slopeloss_joint, 
                                    linewidth=4,
                                    color='royalblue',
                                    label=f"All Group Fit Loss Ratio = {1/nopn_slopeloss_joint:.2f}"
                                    )        
            for i in range(1):#len(bl_len)):            
                scatter_res2 = ax_res_group[-1].scatter(np.log10(uvp_power_list[i]), (uvp_power_list[i]/uvpspec_avg_power_list[i]),
                                                s=10, marker=".", linestyle="-",
                                                label=f" P_coh/P_inc {bl_len[i]}m {bl_ang[i]}°",
                                                color='royalblue',
#                                                c=lst_roll_list[i],
#                                                cmap=cmaps[i], 
                                                alpha=0.2)


    #         for i in range(len(bl_len)):
    #             for j, ((_, pseg1), (_, pseg2)) in enumerate(zip(valleys[i], valleys2[i])):
    #                 ax_res_group[-1].scatter(np.log10(pseg1), (pseg1/pseg2),
    #                                                    s=20, marker=".", linestyle="-",
    #     #                                                label=f" PSPEC ratio : Coh/Incoh\n SPW {freq_range_str}",
    #                                                    color='orange', alpha=0.7)



            # --- Add Diagonal (Slope=1) Lines to Main Axes ---
            line_x1 = np.linspace(*ax_group[0].get_xlim(), num=100)
    #         ax_group[0].plot(line_x1, line_x1, linestyle='-', color='gray', label='Slope = 1', linewidth=1)
    #         line_x2 = np.linspace(*ax_group[-1].get_xlim(), num=100)
    #         ax_group[-1].plot(line_x2, line_x2, linestyle='-', color='gray', label='Slope = 1', linewidth=1)

            # --- Add Zero Lines to Residual Axes ---
            line_rx1 = np.linspace(*ax_res_group[0].get_xlim(), num=100)
            ax_res_group[0].plot(line_rx1, 0*line_rx1, linestyle='-', color='gray', label='Zero Residual', linewidth=1)
    #         line_rx2 = np.linspace(*ax_res_group[-1].get_xlim(), num=100)
    #         ax_res_group[-1].plot(line_rx2, 0*line_rx2, linestyle='-', color='gray', label='Zero Residual', linewidth=1)

            # --- Synchronize Limits for Main Axes ---
    #         xlim = ax_group[0].get_xlim()
    #         ylim = ax_group[0].get_ylim()
    #         overall_min = min(xlim[0], ylim[0])
    #         overall_max = max(xlim[1], ylim[1])
    #         ax_group[0].set_xlim(overall_min, overall_max)
    #         ax_group[0].set_ylim(overall_min, overall_max)

            # --- Set Labels for Main Axes ---
    #         ax_group[0].set_ylabel(r'log$_{10}$(P_inc) (fit) (mK)$^2$', fontsize=12)
    #         ax_group[0].set_xlabel(r'log$_{10}$(P_inc) (data) (mK)$^2$', fontsize=12)
    #         ax_group[0].set_xlabel(r'Coherent log$_{10}$(PSPEC) (mK)$^2$', fontsize=12)
    #         ax_group[0].set_ylabel(r'PSPEC(Coh/Incoh(fit))', fontsize=12)
    #         ax_group[0].set_ylim(0.9, 1.1)
            ax_group[0].set_xlabel(r'log$_{10}$(SNR)', fontsize=12)
            ax_group[0].set_ylabel(r'PSPEC(Coh/Incoh(fit))', fontsize=12)
            ax_group[0].set_ylim(0.8, 1.1)
            # ax_group[0].set_xlim(-0.1, 2.6)

            ax_res_group[0].set_xlabel(r'log$_{10}$(P_coh) (data) (mK)$^2$', fontsize=12)
            ax_res_group[0].set_ylabel(r'$\Delta$ = log$_{10}$(P_inc)-log$_{10}$(P_coh) (mK)$^2$', fontsize=12)

            ax_group[-1].set_ylabel(r'log$_{10}$(PSPEC) (mK)$^2$', fontsize=12)
            ax_group[-1].set_yscale('log')
            ax_group[-1].set_xlabel(r'LST [Hrs]', fontsize=12)
    #         ax_group[-1].set_xlabel(r'Coherent log$_{10}$(PSPEC) (mK)$^2$', fontsize=12)
    #         ax_group[-1].set_ylabel(r'PSPEC(Coh/Incoh(fit))', fontsize=12)
    #         ax_group[-1].set_ylim(0.9, 1.1)
    #         ax_group[-1].set_xlabel(r'log$_{10}$(SNR)', fontsize=12)
    #         ax_group[-1].set_ylabel(r'PSPEC(Coh/Incoh(data))', fontsize=12)
    #         ax_group[-1].set_ylim(0.9, 1.1)

            ax_res_group[-1].set_xlabel(r'Coherent log$_{10}$(PSPEC) (mK)$^2$', fontsize=12)
            ax_res_group[-1].set_ylabel(r'PSPEC(Coh/Incoh)', fontsize=12)
            ax_res_group[-1].set_ylim(0.9, 1)

            # --- Set Titles and Legends (Using Baseline and Frequency Info) ---
            title_str = (f"Len_ {int(bl_len[group])}m, Ang {bl_ang[group]}°, "
                         f"SPW {idx}, {freq_range_str}, Pol: {pol}")
            ax_group[0].set_title(title_str, fontsize=14)
            ax_group[-1].set_title(title_str, fontsize=14)
            ax_group[0].legend(fontsize=10)
            ax_group[-1].legend(fontsize=10)
            ax_res_group[0].legend(fontsize=10)
            ax_res_group[-1].legend(fontsize=10)

            ax_group[0].grid(False)
    #         ax_group[-1].grid(False)
    
    
    # Hide unused subplots.
    # for idx in range(len(spw_info)*2, len(axes)):
    #     fig.delaxes(axes[idx])
    # for idx in range(len(spw_info)*2, len(residual_axes)):
    #     fig.delaxes(residual_axes[idx])
    
    fig.suptitle("Coh vs Incoh PSPEC Scatter Plots", fontsize=16)
    # plt.tight_layout()
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    if(group<4):
        plt.show()
    
    return median_vals, minmax_SNR

# # Example usage:
# plot_percent_difference_with_1hrbin(
#     lst_array = uvp.lst_avg_array * (24 / (2 * np.pi)),  # LST in hours
#     pol = 'xx',
#     uvp = combined_uvp_dict,
#     uvpspec_averaged = combined_uvp_avg_dict
# )


In [ ]:
set_median_vals = []
maxmin_SNR_nsmp_perLST_perspw = []

bl_len = [14.0]#, 25, 25, 29, 29, 29, 39, 39, 39, 39, 44, 44, 44, 51, 51, 53, 53, 53, 53, 58, 58 ] 
bl_ang = [0.0]#, 150, 30, 0, 60, 120, 19, 161, 139, 41, 0, 60, 120, 30, 150, 166, 14, 46, 134, 0, 120 ] 

print(combined_uvp_dict.keys())
print("pol ", pol_root[0])

for i in range(0,len(bl_len)):
    grp_median_vals, minmax_SNR = plot_percent_difference_with_1hrbin(
        i,
        lst_array = uvp.lst_avg_array * (24 / (2 * np.pi)),  # LST in hours
        pol = pol_root,
        uvp = combined_uvp_dict[(bl_len[i],bl_ang[i], pol_root[0])], 
        uvpspec_averaged = combined_uvp_avg_dict[(bl_len[i],bl_ang[i], pol_root[0])] 
    )
    print("grp_median_vals ", grp_median_vals)
    set_median_vals.append(grp_median_vals)
    print("minmax SNR nsample ", minmax_SNR)
    maxmin_SNR_nsmp_perLST_perspw.append(minmax_SNR)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator, FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable



import numpy as np
from scipy.optimize import least_squares
import matplotlib.pyplot as plt

def piecewise_linear(params, x):
    """
    params = [m1, b1, m2, b2, x0]
    returns f(x) defined piecewise.
    """
    m1, b1, m2, b2, x0 = params
    return np.where(x <= x0,
                    m1 * x + b1,
                    m2 * x + b2)


def residuals(params, x, y):
    """Compute model minus data."""
    return piecewise_linear(params, x) - y

def fit_piecewise_linear(x, y, method='trf', manual_fallback=True):
    """
    Fit a two-segment piecewise linear model. By default uses TRF solver;
    if it fails and manual_fallback=True, falls back to grid search.
    Returns dict with slopes, intercepts, breakpoint, and diagnostics.
    """
    # initial guess & bounds
    p0 = [1.0, 0.0, 1.0, 0.0, np.median(x)]
    lower = [-np.inf, -np.inf, -np.inf, -np.inf, np.min(x)]
    upper = [ np.inf,  np.inf,  np.inf,  np.inf, np.max(x)]
    x_scale = np.array([1.0, 1.0, 1.0, 1.0, np.ptp(x)])
    diff_step = x_scale * 0.1

    try:
        result = least_squares(
            residuals, p0, args=(x, y), bounds=(lower, upper),
            method=method, x_scale=x_scale, diff_step=diff_step,
            ftol=1e-10, xtol=1e-10, gtol=1e-10
        )
        success = result.success
    except Exception:
        success = False
        result = None
        
    success=False

    if not success and manual_fallback:
        fit = manual_piecewise_linear_fit(x, y)
        fit['method'] = 'manual'
        return fit

    m1, b1, m2, b2, x0 = result.x
    return dict(
        m1=m1, b1=b1,
        m2=m2, b2=b2,
        x0=x0,
        success=result.success,
        cost=result.cost,
        nfev=result.nfev,
        message=result.message,
        method='trf'
    )

def manual_piecewise_linear_fit(x, y, num_breaks=200):
    """
    Brute-force grid search over breakpoints. Fits two segments via np.polyfit
    and selects breakpoint minimizing sum of squared errors.
    """
#     print("Doing manual")
    #print("x ", x)
    candidates = np.linspace(np.min(x), np.max(x), num_breaks)
    best = {'cost': np.inf}
    for x0 in candidates:
        # split
        left = x <= x0
        right = x > x0
        if np.sum(left) < 2 or np.sum(right) < 2:
            continue
        # fit segments
        m1, b1 = np.polyfit(x[left], y[left], 1)
        m2, b2 = np.polyfit(x[right], y[right], 1)
        # compute cost
        y_pred = np.empty_like(y)
        y_pred[left] = m1 * x[left] + b1
        y_pred[right] = m2 * x[right] + b2
        cost = np.sum((y_pred - y)**2)
        #print("cost ", cost)
        if cost < best['cost']:
            best.update(dict(m1=m1, b1=b1, m2=m2, b2=b2, x0=x0, cost=cost))
    return {**best, 'success': True, 'n_breaks': num_breaks}


def fit_linear_combo(y, fx, gx, snx=None):
    """
    Fit the model y ≈ a·f(x) + b·g(x) + c via ordinary least squares.

    Parameters
    ----------
    y  : array‐like, shape (N,)
        Observed dependent variable.
    fx : array‐like, shape (N,)
        First regressor f(x).
    gx : array‐like, shape (N,)
        Second regressor g(x).

    Returns
    -------
    a, b, c : floats
        Best‐fit coefficients such that y ≈ a·fx + b·gx + c.
    """
    y  = np.asarray(y,  float)
    fx = np.asarray(fx, float)
    gx = np.asarray(gx, float)
    
    # Build design matrix with columns [fx, gx, 1]
#     X = np.column_stack([fx, gx])#, np.ones_like(fx)])
    if snx is not None :
        snx = np.asarray(snx, float)
        X = np.column_stack([fx, gx, snx])#, np.ones_like(fx)])
    else:
        X = np.column_stack([fx, gx])#, np.ones_like(fx)])
    # Solve the normal equations: minimize ||X @ [a,b,c] – y||^2
#     (a, b, c), *_ = np.linalg.lstsq(X, y, rcond=None)
#     return a, b, c
    coefs = np.dot(np.linalg.pinv(X), y)
    return coefs



# Example usage:
# a, b, c = fit_linear_combo(y_array, f_of_x_array, g_of_x_array)

def predict_from_coeff(a, b, c, fx, gx):
# def predict_from_coeff(a, b, c, sn, fx, gx, snx):
    """
    Given fit coefficients a, b, c and your original regressors fx = f(x),
    gx = g(x), return the model prediction y_hat for each gx:
        y_hat = a*fx + b*gx + c

    Parameters
    ----------
    a, b, c : float
        Fitted coefficients.
    fx : array‐like, shape (N,)
        Values of f(x) at your sample points.
    gx : array‐like, shape (N,)
        Values of g(x) at your sample points.

    Returns
    -------
    y_hat : ndarray, shape (N,)
        The predicted y values, ready to plot versus gx.
    """
    fx = np.asarray(fx)
    gx = np.asarray(gx)
#     snx = np.asarray(snx)
    
    return a*fx + b*gx #+ c
#     return a*fx + b*gx + sn*snx #+ c


# Example usage:
# a, b, c = fit_linear_combo(y, fx, gx)
# y_pred = predict_from_g(a, b, c, fx, gx)
# plt.plot(gx, y_pred, '-', label='model vs g(x)')




def compute_1hr_bin_average(lst_array, data_array):
    """
    Compute a 1-hour bin average of data_array centered around each LST value.
    """
    data_array_1hrbin = np.zeros_like(data_array)
    lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
    for i, lst in enumerate(lst_array_roll):
        # Find indices within ±0.5 hours of the current LST
        indices_in_bin = np.where((lst_array_roll >= lst - 0.5) & (lst_array_roll <= lst + 0.5))[0]
        # if( lst>0.001 and lst<0.002 ): 
            # print(" indices_in_bin ", lst_array[indices_in_bin] )
        if len(indices_in_bin) > 0:
            data_array_1hrbin[i] = np.nanmean(data_array[indices_in_bin])
        else:
            data_array_1hrbin[i] = np.nan
    return data_array_1hrbin

def get_spw_info(uvp):
    """
    Returns a dictionary mapping each SPW (from uvp.spw_array) to its (min, max) frequency range in Hz.
    """
    spw_indices = uvp.spw_array       # e.g., shape (14,)
    freq = uvp.freq_array             # e.g., shape (1114,)
    spw_freq = uvp.spw_freq_array     # e.g., shape (1114,)
    
    spw_info = {}
    for spw in spw_indices:
        mask = (spw_freq == spw)
        if np.any(mask):
            freq_min = np.min(freq[mask])
            freq_max = np.max(freq[mask])
            spw_info[spw] = (freq_min, freq_max)
        else:
            spw_info[spw] = None
    return spw_info

def get_plot_data(uvp, uvpspec_averaged, pol, idx):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    blp = uvp.get_blpairs()[0]
    # key = (idx, blp, 'xx')
    key = (idx, blp, pol)

    # Retrieve LST array and convert to hours.
    tarr = uvp.lst_avg_array
    tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
    lst_array_rad = tarr
#         print("lst_array_rad ", lst_array_rad)
    lst_array = tarrq

    dlys = uvp.get_dlys(idx) * 1e9
    index_of_zero = np.where(np.isclose(dlys, 0))[0][0]

    uvp_power = np.abs((uvp.get_data(key)))[:, index_of_zero]
    uvpspec_averaged_power = np.abs((uvpspec_averaged.get_data(key)))[:, index_of_zero]
    
    uvpspec_averaged_noise = np.abs(uvpspec_averaged.get_stats('autos_diag',key))[:, index_of_zero]
    
    uvp_power_S = (uvp.get_data(key))[:, index_of_zero].real

    percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
    median_val = np.nanmedian(percent_diff)
    median_vals.append(median_val)
    mean_val = np.nanmean(percent_diff)
    mean_vals.append(mean_val)

    lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        
    return dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, uvpspec_averaged_noise, uvp_power_S, percent_diff, mean_vals

median_vals = []
mean_vals = []


def plot_percent_difference_with_1hrbin(lst_array, pol, uvp, uvpspec_averaged,
                                        zero_delay_pspec_spw=None, avg_pspec=None):
    """
    Plot percent difference (using 1-hr bin averages) between uvp and uvpspec_averaged versus LST for each SPW.
    
    For each SPW, the plot has two main axes (with corresponding residual axes below):
      - Main scatter plots (for full data and 1-hr binned data)
      - Residual scatter plots (showing the difference on a log10 scale)
    
    Titles and legends include:
      - Baseline length (in m)
      - Baseline angle (forced to 0° here, while also showing the computed angle in degrees)
      - SPW index and its full frequency range (in MHz)
      - Polarization.
    """
    # Ensure the LST array is sorted (in hours)
    # lst_sorted_indices = np.argsort(lst_array)
    # lst_array = lst_array[lst_sorted_indices]
    
    # Retrieve SPW info from the UVP object
    spw_info = get_spw_info(combined_uvp_dict[(bl_len[0], bl_ang[0], pol)])  # Using the first baseline as reference
    
    num_spws = len(spw_info)
    fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharex=False)
    axes = axes.flatten(order='C')
    
    # Create residual axes appended below each main axis
    residual_axes = []
    for ax in axes:
        divider = make_axes_locatable(ax)
        ax_res = divider.append_axes("bottom", size="30%", pad=0.5)
        residual_axes.append(ax_res)
        ax.tick_params(labelbottom=True)
    
    # --- Baseline Information ---
    bl_vec = combined_uvp_dict[(bl_len[0], bl_ang[0], pol) ].bl_vecs[0]  # using the first baseline vector
    baseline_length = np.linalg.norm(bl_vec)
    angle_rad = np.arctan2(bl_vec[1], bl_vec[0])
    baseline_angle_deg = np.degrees(angle_rad)
    forced_angle = 0  # Force to 0° if desired
    
    # --- Loop Over SPWs ---
    # Iterate over sorted spw_info items (by the key)
    for idx, (spw_key, spw_val) in enumerate(sorted(spw_info.items(), key=lambda x: x[0])):
        if spw_val is not None:
            freq_min, freq_max = spw_val
            freq_range_str = f"{freq_min/1e6:.2f}-{freq_max/1e6:.2f} MHz"
        else:
            freq_range_str = "N/A"
        
        # Data retrieval using a baseline pair (adjust as needed)
        blp = combined_uvp_dict[(bl_len[0], bl_ang[0], pol)].get_blpairs()[0]
        key = (idx, blp, 'xx')
        
#         for i in range(1):#len(bl_len)):
        i=0
        dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, uvpspec_averaged_noise, uvp_power_S, percent_diff, mean_vals = get_plot_data(combined_uvp_dict[(bl_len[i], bl_ang[i], pol)], combined_uvp_avg_dict[(bl_len[i], bl_ang[i], pol)], pol, idx)
            
        P_SN = np.sqrt(np.sqrt(2) * uvp_power_S * np.sqrt(77)*uvpspec_averaged_noise + (np.sqrt(77)*uvpspec_averaged_noise)**2)
            
        # Valleys for threshold to peak range
        segs, dips, valleys, valleys2 = bump_n_val_segments_by_dips(
            lst_array_roll, uvp_power, uvpspec_averaged_power,
            smooth_window=51, polyorder=3,
            dip_prominence=1e10, dip_width=5,
            frac=0.05
        )
#         for i, (lseg, pseg) in enumerate(valleys):
#             print(f"Valley {i}: LST {lseg[0]:.3f} → {lseg[-1]:.3f}, {len(lseg)} points")

#         print(f"Found {len(segs)} bumps between dips at indices {dips}")
        
#         dlys = combined_uvp_dict[bl_len[0]].get_dlys(idx) * 1e9
#         index_of_zero = np.where(np.isclose(dlys, 0))[0][0]
        
#         uvp_power = np.abs(np.real(uvp.get_data(key)))[:, index_of_zero]
#         uvpspec_averaged_power = np.abs(np.real(uvpspec_averaged.get_data(key)))[:, index_of_zero]
        
#         uvp_power = uvp_power[lst_sorted_indices]
#         uvpspec_averaged_power = uvpspec_averaged_power[lst_sorted_indices]
        
        # Compute 1-hr bin averages
        uvpspec_averaged_power_1hrbin = compute_1hr_bin_average(lst_array, uvpspec_averaged_power)
        uvp_power_1hrbin = compute_1hr_bin_average(lst_array, uvp_power)
        
        # Compute percent differences:
        percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power_1hrbin != 0,
                                                                              uvpspec_averaged_power_1hrbin,
                                                                              np.nan)
        percent_diff_raw = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0,
                                                                                  uvpspec_averaged_power,
                                                                                  np.nan)
        
        median_val = np.nanmedian(percent_diff)
        mean_val = np.nanmean(percent_diff)
        
        # Group axes: use 2 main axes per SPW (and 2 corresponding residual axes)
        a = 4 * (idx // 2) + (idx % 2)
        b = a + 2
#         print((a, b))
#         ax_group = axes[idx:idx+1] 
        ax_group = axes[a:b+1] # axes[2*idx:(2*idx)+2]
#         ax_res_group = residual_axes[idx:idx+1] 
        ax_res_group = residual_axes[a:b+1] # residual_axes[2*idx:(2*idx)+2]
        
        # --- Main Scatter Plots ---
        
#         ax_group[-1].plot(lst_array_roll, uvp_power, linewidth=1, linestyle="-", color="blue", alpha=0.5,
#                    label=f"P_coh SPW {freq_range_str}")
        ax_group[-1].plot(lst_array_roll, uvpspec_averaged_noise, linewidth=0.5, linestyle="-", color="red",
                   label=f"P_N SPW {freq_range_str}")
        ax_group[-1].plot(lst_array_roll, P_SN, linewidth=0.5, linestyle="-", color="blue",
                   label=f"P_SN SPW {freq_range_str}")
        from itertools import cycle
#         color_cycle = cycle(['#1f77b4','#ff7f0e','#2ca02c','#d62728'])  # any palette
        style_cycle = cycle(['-',':'])  # any palette
        cmap = plt.cm.viridis
        n = len(segs) 
        n = 2

        for i,(lst_seg,ps_seg) in enumerate(segs):
#             print(f"--- bump #{i} spans LST = {lst_seg[0]:.2f} → {lst_seg[-1]:.2f}")
            ax_group[-1].plot(lst_seg, ps_seg, linestyle=next(style_cycle), #"-", 
                                 color=cmap(i/(n-1)), #next(color_cycle),
                                 linewidth=2,
                                 alpha=0.9)
        for i, (lseg, pseg) in enumerate(valleys):
            ax_group[-1].plot(lseg, pseg, linestyle="-", color='orange',
                       alpha=1)
            
        # SCATTER PLOT WITH FIT
        
        # 3) build an integer “segment index” for each point in your full lst_array
        segment_id = np.full_like(lst_array_roll, fill_value=-1, dtype=int)
        for i, (lst_seg, _) in enumerate(segs):
            # if segs gives you the exact LST values in each bump:
            mask = np.isin(lst_array_roll, lst_seg)
            # otherwise if lst_seg is a contiguous slice, you can do:
            # mask = (lst_array_roll >= lst_seg.min()) & (lst_array_roll <= lst_seg.max())
            segment_id[mask] = i
            
        # 4) grab a discrete OrRd with N colors
        from matplotlib import cm
        cmap_fit = cm.get_cmap('viridis', n)

        # 5) set up a norm so each integer picks its own block
        from matplotlib.colors import BoundaryNorm
        norm = BoundaryNorm(np.arange(n+1) - 0.5, n)
        
#         scatter1 = ax_group[0].scatter(np.log10(uvp_power), np.log10(uvpspec_averaged_power),
#                                        s=20, marker=".", linestyle="-",
#                                        label=f"Coh vs Incoh PSPEC\n SPW {freq_range_str}",
#                                        c=segment_id, #lst_array, 
#                                        cmap=cmap_fit, #'OrRd', 
#                                        alpha=0.7)
#         cbar = plt.colorbar(scatter1, ticks=np.arange(n))
#         cbar.set_label('bump index')
        
#         for i, ((_, pseg1), (_, pseg2)) in enumerate(zip(valleys, valleys2)):
# #             ax_group[-1].scatter(lseg, pseg, s=2, marker="o", linestyle="-", color='black',
# #                        alpha=1)
#             print("valleys", i, pseg1.shape, pseg2.shape)
#             ax_group[0].scatter(np.log10(pseg1), np.log10(pseg2),
#                                        s=20, marker=".", linestyle="-",
# #                                        label=f"Nulls PSPEC\n SPW {freq_range_str}",
# #                                        c=segment_id, #lst_array, 
#                                        color='orange', #'OrRd', 
#                                        alpha=1)

#         scatter2 = ax_group[0].scatter(np.log10(uvp_power+uvpspec_averaged_noise), np.log10(uvpspec_averaged_power),
#                                        s=20, marker=".", linestyle="-",
#                                        label=f"Coh vs Incoh PSPEC\n SPW {freq_range_str}",
#                                        c=lst_array, cmap='YlGn', alpha=0.7)
        
#         print("np.log10(uvp_power) ", (uvp_power))
#         print("np.log10(uvpspec_averaged_power) ", (uvpspec_averaged_power))
        mask = ~ (np.isneginf(np.log10(uvp_power)) | np.isneginf(np.log10(uvpspec_averaged_power)))
        xin = np.log10(uvp_power)[mask]
        yin = np.log10(uvpspec_averaged_power)[mask]
#         print("inputs ", xin, yin)
        fit = fit_piecewise_linear(xin, yin)
#         print(fit.keys() )
#         print(f"Method: {fit['method']}")
#         print(f"m1={fit['m1']:.3f}, b1={fit['b1']:.3f}")
#         print(f"m2={fit['m2']:.3f}, b2={fit['b2']:.3f}")
#         print(f"x0={fit['x0']:.3f}, cost={fit.get('cost',np.nan):.3f}")
        
        m1, b1 = fit['m1'], fit['b1']
        m2, b2 = fit['m2'], fit['b2']
        x0 = fit['x0']
        print(f"SPW {idx}: 2-line fit Loss {-1*(1-m2)*100:.2f}%")
        
        left_mask = xin < x0
        n_left   = np.count_nonzero(left_mask)
        n_total  = xin.size
        fraction = n_left / n_total

#         print(f"{n_left} points are to the left of x0 = {x0:.3f}, out of {n_total} total")
#         print(f"Fraction left of breakpoint = {fraction:.2%}")
        
        inp_x = xin #np.log10(uvp_power)
        x_min, x_max = np.min(inp_x), np.max(inp_x)
        x1 = np.linspace(x_min, x0, 200)
        x2 = np.linspace(x0, x_max, 200)
#         ax_group[0].plot(x1, m1*x1 + b1, linewidth=1, label=f'Segment 1 {n_left} LSTs', color='blue', alpha=0.7 )
#         ax_group[0].plot(x2, m2*x2 + b2, linewidth=1, label=f'Segment 2 Loss {(1-m2)*100:.2f} %', color='blue', alpha=0.7 )
#         ax_group[0].axvline(x0, linestyle='--', label=f'Inflection, divergent {n_left} LSTs') # x0={x0:.2f}')
        
        NaN_mask = np.isfinite(uvpspec_averaged_power) & np.isfinite(uvpspec_averaged_noise) & np.isfinite(uvp_power)
        pinc = uvpspec_averaged_power[NaN_mask]
        P_N = uvpspec_averaged_noise[NaN_mask]
        pcoh = uvp_power[NaN_mask]
        P_SN_in = P_SN[NaN_mask]
        cond = np.linalg.cond(np.column_stack([P_N, pcoh]))
#         print("Condition number:", cond)
        c=1
        print("pinc, pcoh ", len(pinc), len(pcoh))
    
        na, slopeloss = fit_linear_combo(y=pinc, fx=P_N, gx=pcoh)
#         na, slopeloss, sn = fit_linear_combo(pinc, P_N, pcoh, P_SN_in)
    
        print("P_N fit N_a, P_slope ", na, slopeloss)
#         print("P_N fit N_a, P_slope ", na, slopeloss, sn)
    
        print(f"P_N fit loss {-1*(1-(1/slopeloss))*100:.2f} %")
        
        pinc_pred = predict_from_coeff(na, slopeloss, c, P_N, pcoh)
#         pinc_pred = predict_from_coeff(na, slopeloss, c, sn, P_N, pcoh, P_SN_in)
    
#         scatter1 = ax_group[0].scatter(np.log10(pcoh), np.log10(pinc_pred), s=2, linewidth=1, label=f'P_N Fit slope={slopeloss:.2f}, loss={(slopeloss-1)*100:.2f}%', color='red', alpha=0.6,) 
#                             c=lst_array_roll[NaN_mask], cmap='viridis' )
        # scatter2 = ax_group[0].scatter(np.log10(pinc), np.log10(pinc_pred), s=2, linewidth=1, label=f'Pinc_data vs Pinc_fit slope={slopeloss:.2f}, loss={(slopeloss-1)*100:.2f}%',  alpha=0.7, 
        #                     c=lst_array_roll[NaN_mask], cmap='viridis' )
        scatter2 = ax_group[0].scatter(np.log10(pinc), np.log10(pcoh), s=2, linewidth=1, label=f'Pinc_data vs Pinc_fit slope={slopeloss:.2f}, loss={(slopeloss-1)*100:.2f}%',  alpha=0.7, 
                            c=lst_array_roll[NaN_mask], cmap='viridis' )
        
        cbar = plt.colorbar(scatter2)#, ticks=np.arange(12))
        cbar.set_label('LST')
        
        ax_group[-1].plot(lst_array_roll[NaN_mask], pcoh, linewidth=0.5, linestyle="-", color="red",
                   label=f"Mask Pcoh SPW {freq_range_str}")
        
#         left_mask = np.log10(pcoh) < x0
#         n_left   = np.count_nonzero(left_mask)
#         ax_group[0].axvline(x0, linestyle='--', label=f'Inflection, divergent {n_left} LSTs') # x0={x0:.2f}')
        
        
#         cbar1 = fig.colorbar(scatter1, ax=ax_group[0])
#         cbar1.set_label("LST (Hr)", fontsize=12)
    
        
#         scatter2 = ax_group[-1].scatter(np.log10(uvp_power_1hrbin), np.log10(uvpspec_averaged_power_1hrbin),
#                                        s=20, marker=".", linestyle="-",
#                                        label=f"1Hr Coh vs 1Hr Incoh PSPEC\n SPW {freq_range_str}",
#                                        c=lst_array, cmap='OrRd', alpha=0.7)
#         cbar2 = fig.colorbar(scatter2, ax=ax_group[-1])
#         cbar2.set_label("LST (Hr)", fontsize=12)
        
        # --- Residual Scatter Plots ---
        scatter_res1 = ax_res_group[0].scatter(
                                               np.log10(uvp_power), 
                                            #    np.log10(uvpspec_averaged_power)-np.log10(uvp_power),
                                               (uvpspec_averaged_power-uvp_power)/uvpspec_averaged_power,
                                               s=20, marker=".", linestyle="-",
                                               label=f"Residual: Coh vs Incoh\n SPW {freq_range_str}",
                                               c=segment_id, cmap=cmap_fit, alpha=0.7)
#         scatter_res2 = ax_res_group[-1].scatter(np.log10(uvp_power_1hrbin), np.log10(uvpspec_averaged_power_1hrbin)-np.log10(uvp_power_1hrbin),
#                                                s=20, marker=".", linestyle="-",
#                                                label=f"Residual 1hr: Coh vs Incoh\n SPW {freq_range_str}",
#                                                c=lst_array, cmap='OrRd', alpha=0.7)
        scatter_res2 = ax_res_group[-1].scatter(np.log10(uvp_power), (uvp_power/uvpspec_averaged_power),
                                               s=20, marker=".", linestyle="-",
                                               label=f" PSPEC ratio : Coh/Incoh\n SPW {freq_range_str}",
                                               c=segment_id, cmap=cmap_fit, alpha=0.7)
    
        for i, ((_, pseg1), (_, pseg2)) in enumerate(zip(valleys, valleys2)):
#             print("valleys", i, pseg1.shape, pseg2.shape)
            ax_res_group[-1].scatter(np.log10(pseg1), (pseg1/pseg2),
                                               s=20, marker=".", linestyle="-",
#                                                label=f" PSPEC ratio : Coh/Incoh\n SPW {freq_range_str}",
                                               color='orange', alpha=0.7)
        
        # --- Overlay Subset Points (where uvp_power > uvpspec_averaged_power) ---
        mask_full = uvp_power > uvpspec_averaged_power
        subset_uvp_power = uvp_power[mask_full]
        subset_uvpspec_averaged_power = uvpspec_averaged_power[mask_full]
        
        mask_1hr = uvp_power_1hrbin > uvpspec_averaged_power_1hrbin
        subset_uvp_power_1hrbin = uvp_power_1hrbin[mask_1hr]
        subset_uvpspec_averaged_power_1hrbin = uvpspec_averaged_power_1hrbin[mask_1hr]
        
#         ax_group[0].scatter(np.log10(subset_uvp_power), np.log10(subset_uvpspec_averaged_power),
#                             s=100, marker="o", facecolors='none', edgecolors='r', linewidth=1.5,
#                             label=f"UVP > UVPSPEC ({np.sum(mask_full)})")
#         ax_group[1].scatter(np.log10(subset_uvp_power_1hrbin), np.log10(subset_uvpspec_averaged_power_1hrbin),
#                             s=100, marker="o", facecolors='none', edgecolors='r', linewidth=1.5,
#                             label=f"UVP > UVPSPEC 1hrBin ({np.sum(mask_1hr)})")
        
        # --- Add Diagonal (Slope=1) Lines to Main Axes ---
        line_x1 = np.linspace(*ax_group[0].get_xlim(), num=100)
        ax_group[0].plot(line_x1, line_x1, linestyle='-', color='gray', label='Slope = 1', linewidth=1)
#         line_x2 = np.linspace(*ax_group[-1].get_xlim(), num=100)
#         ax_group[-1].plot(line_x2, line_x2, linestyle='-', color='gray', label='Slope = 1', linewidth=1)
        
        # --- Add Zero Lines to Residual Axes ---
        line_rx1 = np.linspace(*ax_res_group[0].get_xlim(), num=100)
        ax_res_group[0].plot(line_rx1, 0*line_rx1, linestyle='-', color='gray', label='Zero Residual', linewidth=1)
#         line_rx2 = np.linspace(*ax_res_group[-1].get_xlim(), num=100)
#         ax_res_group[-1].plot(line_rx2, 0*line_rx2, linestyle='-', color='gray', label='Zero Residual', linewidth=1)
        
        # --- Synchronize Limits for Main Axes ---
        xlim = ax_group[0].get_xlim()
        ylim = ax_group[0].get_ylim()
        overall_min = min(xlim[0], ylim[0])
        overall_max = max(xlim[1], ylim[1])
        ax_group[0].set_xlim(overall_min, overall_max)
        ax_group[0].set_ylim(overall_min, overall_max)
        
        # --- Set Labels for Main Axes ---
        ax_group[0].set_ylabel(r'Incoherent log$_{10}$(PSPEC) FIT (mK)$^2$', fontsize=12)
        ax_res_group[0].set_xlabel(r'Incoherent log$_{10}$(PSPEC) DAT (mK)$^2$', fontsize=12)
        ax_res_group[0].set_ylabel(r'$\Delta$ (mK)$^2$', fontsize=12)
        
        ax_group[-1].set_ylabel(r'log$_{10}$(PSPEC) (mK)$^2$', fontsize=12)
        ax_group[-1].set_yscale('log')
        ax_group[-1].set_xlabel(r'LST [Hrs]', fontsize=12)
        ax_res_group[-1].set_xlabel(r'Coherent log$_{10}$(PSPEC) (mK)$^2$', fontsize=12)
        ax_res_group[-1].set_ylabel(r'PSPEC(Coh/Incoh)', fontsize=12)
        ax_res_group[-1].set_ylim(0.9, 1)
        
        # --- Set Titles and Legends (Using Baseline and Frequency Info) ---
        title_str = (f"Len {int(baseline_length)}m, Ang {forced_angle}°, "
                     f"SPW {idx}, {freq_range_str}, Pol: {pol}")
        ax_group[0].set_title(title_str, fontsize=14)
#         ax_group[-1].set_title(title_str, fontsize=14)
        ax_group[0].legend(fontsize=10)
        ax_group[-1].legend(fontsize=10)
        ax_res_group[0].legend(fontsize=10)
        ax_res_group[-1].legend(fontsize=10)
        
        ax_group[0].grid(False)
#         ax_group[-1].grid(False)
    
    # Hide unused subplots.
    # for idx in range(len(spw_info)*2, len(axes)):
    #     fig.delaxes(axes[idx])
    # for idx in range(len(spw_info)*2, len(residual_axes)):
    #     fig.delaxes(residual_axes[idx])
    
    fig.suptitle("Coh vs Incoh PSPEC Scatter Plots", fontsize=16)
    # plt.tight_layout()
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

# Example usage:
plot_percent_difference_with_1hrbin(
    lst_array = uvp.lst_avg_array * (24 / (2 * np.pi)),  # LST in hours
    pol = 'xx',
    uvp = combined_uvp_dict,
    uvpspec_averaged = combined_uvp_avg_dict
)


# Continuous visibility, redundancy, time averaging, and thermal-noise statistics

This cell rewrites the visibility model with cleaner notation.

We fix the polarization to `xx` throughout this cell. Therefore every visibility, beam, autocorrelation, and noise variance below is an `xx` quantity. If one later wants to restore polarization labels, replace each object by the corresponding object with explicit polarization labels.

## 1. Variables and data model

Let

- $i$ and $j$ label two antennas.
- $ij$ label the antenna pair, or baseline.
- $g$ label a redundant baseline group.
- $\mathcal{B}_g$ be the set of antenna pairs in redundant group $g$.
- $N_{b,g}$ be the number of baselines in group $g$.

Thus

$$
\mathcal{B}_g
=
\{ij:\;ij\;\text{belongs to redundant group}\;g\},
$$

and

$$
N_{b,g}=|\mathcal{B}_g|.
$$

Let

- $\nu$ be observing frequency.
- $t$ be local sidereal time, abbreviated LST.
- $\Delta \nu$ be the frequency-channel width.
- $\tau$ be the integration time for one visibility sample.

The measured complex visibility for antenna pair $ij$ is

$$
d_{ij}(\nu,t)
=
s_{ij}(\nu,t)
+
n_{ij}(\nu,t).
$$

Here

- $d_{ij}(\nu,t)$ is the measured calibrated visibility.
- $s_{ij}(\nu,t)$ is the sky signal visibility.
- $n_{ij}(\nu,t)$ is the thermal-noise visibility.

A slightly more realistic redundant-baseline model separates the common redundant signal from baseline-dependent errors:

$$
d_{ij}(\nu,t)
=
s_g(\nu,t)
+
e_{ij}(\nu,t)
+
n_{ij}(\nu,t),
\qquad ij\in\mathcal{B}_g.
$$

Here

- $s_g(\nu,t)$ is the sky visibility shared by all perfectly redundant baselines in group $g$.
- $e_{ij}(\nu,t)$ is the non-redundant residual for pair $ij$.
- $n_{ij}(\nu,t)$ is thermal noise.

Perfect redundancy means

$$
e_{ij}(\nu,t)=0
\qquad
\text{for every}\quad ij\in\mathcal{B}_g.
$$

## 2. Continuous RIME for the sky visibility

Let $\widehat{\boldsymbol{s}}$ be a direction on the celestial sphere. Let

$$
d\Omega
$$

be the solid-angle element.

Let

$$
I_{xx}(\widehat{\boldsymbol{s}},\nu)
$$

be the sky brightness entering the `xx` visibility. It can contain foregrounds and the 21 cm signal:

$$
I_{xx}(\widehat{\boldsymbol{s}},\nu)
=
I^{\mathrm{fg}}_{xx}(\widehat{\boldsymbol{s}},\nu)
+
I^{21\,\mathrm{cm}}_{xx}(\widehat{\boldsymbol{s}},\nu).
$$

Let

$$
A_i^x(\widehat{\boldsymbol{s}},\nu,t)
$$

be the electric-field beam response of antenna $i$ for the fixed `x` feed. Let

$$
A_j^x(\widehat{\boldsymbol{s}},\nu,t)
$$

be the corresponding beam response of antenna $j$.

Let

$$
\boldsymbol{b}_{ij}
=
\boldsymbol{r}_j-
\boldsymbol{r}_i
$$

be the baseline vector from antenna $i$ to antenna $j$, in meters.

The sky direction expressed in the local topocentric coordinate system at LST $t$ is

$$
\widehat{\boldsymbol{s}}_{\mathrm{top}}(\widehat{\boldsymbol{s}},t).
$$

Then the continuous visibility is

$$
s_{ij}(\nu,t)
=
\int_{4\pi}
A_i^x(\widehat{\boldsymbol{s}},\nu,t)
\left[A_j^x(\widehat{\boldsymbol{s}},\nu,t)\right]^\ast
I_{xx}(\widehat{\boldsymbol{s}},\nu)
\exp\left[
-2\pi i
\frac{\nu}{c}
\boldsymbol{b}_{ij}\cdot
\widehat{\boldsymbol{s}}_{\mathrm{top}}(\widehat{\boldsymbol{s}},t)
\right]
d\Omega.
$$

The phase for one sky direction is

$$
\phi_{ij}(\widehat{\boldsymbol{s}},\nu,t)
=
-2\pi
\frac{\nu}{c}
\boldsymbol{b}_{ij}\cdot
\widehat{\boldsymbol{s}}_{\mathrm{top}}(\widehat{\boldsymbol{s}},t).
$$

The fringe factor is

$$
\exp\left[i\phi_{ij}(\widehat{\boldsymbol{s}},\nu,t)\right]
=
\cos\left[\phi_{ij}(\widehat{\boldsymbol{s}},\nu,t)\right]
+i
\sin\left[\phi_{ij}(\widehat{\boldsymbol{s}},\nu,t)\right].
$$

The cosine is the real fringe pattern. The full complex exponential is the actual RIME phase factor.

## 3. Simplest redundant sky model

For an ideal redundant group $g$, all antenna pairs in $\mathcal{B}_g$ have the same baseline vector and the same effective beams. Then

$$
s_{ij}(\nu,t)=s_g(\nu,t)
\qquad
\text{for every}\quad ij\in\mathcal{B}_g.
$$

The data model becomes

$$
d_{ij}(\nu,t)=s_g(\nu,t)+n_{ij}(\nu,t).
$$

This is the most useful toy model for understanding coherent averaging.

## 4. Thermal-noise model

We model the thermal noise as a zero-mean complex Gaussian random variable:

$$
n_{ij}(\nu,t)
=
n^{\mathrm{R}}_{ij}(\nu,t)
+i n^{\mathrm{I}}_{ij}(\nu,t).
$$

The mean is

$$
\left\langle n_{ij}(\nu,t)\right\rangle=0.
$$

The complex variance is

$$
\left\langle
n_{ij}(\nu,t)n^\ast_{ij}(\nu,t)
\right\rangle
=
\sigma^2_{ij}(\nu,t).
$$

For circular complex Gaussian noise,

$$
\left\langle
\left[n^{\mathrm{R}}_{ij}(\nu,t)\right]^2
\right\rangle
=
\frac{1}{2}\sigma^2_{ij}(\nu,t),
$$

and

$$
\left\langle
\left[n^{\mathrm{I}}_{ij}(\nu,t)\right]^2
\right\rangle
=
\frac{1}{2}\sigma^2_{ij}(\nu,t).
$$

## 5. Autocorrelations and the visibility-noise variance

This section derives the usual autocorrelation-based visibility-noise estimate from a voltage and correlator model. The important conceptual point is this:

$$
\boxed{
\text{the autocorrelations estimate total voltage power, not receiver-only power.}
}
$$

That total voltage power includes sky power and receiver power. This is not a mistake. The variance of a finite-time cross-visibility estimate is set by the total stochastic power entering the two antennas.

### 5.1 Voltage entering one antenna

Let

$$
x_i(\nu,t,m)
$$

be the complex channelized voltage sample from antenna $i$ in a frequency channel centered at $\nu$ and an integration centered at LST $t$. The index $m$ labels a statistically independent voltage sample inside that channel and integration.

Write the voltage as

$$
\boxed{
x_i(\nu,t,m)
=
x_i^{\mathrm{sky}}(\nu,t,m)
+
x_i^{\mathrm{rx}}(\nu,t,m).
}
$$

Here

- $x_i^{\mathrm{sky}}$ is the voltage induced by the sky electric field after the antenna beam, analog chain, and channelization.
- $x_i^{\mathrm{rx}}$ is the receiver and electronics noise added locally at antenna $i$.

Both are stochastic voltage samples. The sky visibility $s_g(\nu,t)$ used above is fixed in the sense that it is the ensemble mean cross-correlation expected for a stable sky and instrument during the integration. The individual voltage samples still fluctuate.

A simple scalar `xx` voltage model is

$$
x_i^{\mathrm{sky}}(\nu,t,m)
=
\int d\Omega\;
J_i(\hat{s},\nu,t)
E_x(\hat{s},\nu,t,m)
\exp\left[-2\pi i\frac{\nu}{c}\mathbf{r}_i\cdot\hat{s}\right],
$$

where

- $\hat{s}$ is a sky direction.
- $J_i(\hat{s},\nu,t)$ is the direction-dependent antenna/electronics response for `xx`.
- $E_x(\hat{s},\nu,t,m)$ is the incident sky electric field in the `x` polarization basis.
- $\mathbf{r}_i$ is the antenna position vector.

For an incoherent thermal sky, different directions have uncorrelated electric fields:

$$
\left\langle
E_x(\hat{s},\nu,t,m)
E_x^\ast(\hat{s}',\nu,t,m)
\right\rangle
=
I_x(\hat{s},\nu,t)\delta(\hat{s}-\hat{s}'),
$$

where $I_x$ is the sky brightness seen by the `xx` response. The angle brackets mean an ensemble average over many microscopic electric-field realizations.


### 5.2 Integration time, bandwidth, and the residual voltage noise

Before forming a visibility, it is useful to ask what integration does to the voltage stream from one antenna.

For a short pedagogical toy model, write one channelized complex voltage sample as

$$
\boxed{
x_i(\nu,t,m)
=
\mu_i(\nu,t)
+
\epsilon_i(\nu,t,m).
}
$$

Here $\mu_i(\nu,t)$ is a fixed coherent voltage component during the integration, and

$$
\epsilon_i(\nu,t,m)
\sim
\mathcal{CN}\left(0,\sigma^2_{x,i}(\nu,t)\right)
$$

is circular complex Gaussian thermal noise. The notation $\mathcal{CN}$ means that the real and imaginary parts are Gaussian and that the complex noise has zero mean. Assume independent samples in $m$ for now.

If the instrument averaged the voltage samples directly, the finite-time voltage average would be

$$
\overline{x}_i(\nu,t)
=
\frac{1}{M_{\mathrm{eff}}(\nu,t)}
\sum_{m=1}^{M_{\mathrm{eff}}(\nu,t)}
x_i(\nu,t,m).
$$

Substituting the toy voltage model gives

$$
\overline{x}_i(\nu,t)
=
\mu_i(\nu,t)
+
\overline{\epsilon}_i(\nu,t),
$$

where

$$
\overline{\epsilon}_i(\nu,t)
=
\frac{1}{M_{\mathrm{eff}}(\nu,t)}
\sum_{m=1}^{M_{\mathrm{eff}}(\nu,t)}
\epsilon_i(\nu,t,m).
$$

Because the noise samples are independent and have zero mean,

$$
\left\langle \overline{\epsilon}_i(\nu,t)\right\rangle=0,
$$

and

$$
\begin{aligned}
\mathrm{Var}\left[\overline{\epsilon}_i(\nu,t)\right]
&=
\mathrm{Var}\left[
\frac{1}{M_{\mathrm{eff}}}
\sum_{m=1}^{M_{\mathrm{eff}}}\epsilon_i(\nu,t,m)
\right] \\
&=
\frac{1}{M_{\mathrm{eff}}^2}
\sum_{m=1}^{M_{\mathrm{eff}}}
\mathrm{Var}\left[\epsilon_i(\nu,t,m)\right] \\
&=
\frac{\sigma^2_{x,i}(\nu,t)}{M_{\mathrm{eff}}(\nu,t)}.
\end{aligned}
$$

Thus the residual voltage noise amplitude after averaging is

$$
\boxed{
\sigma_{\overline{x},i}(\nu,t)
=
\frac{\sigma_{x,i}(\nu,t)}
{\sqrt{M_{\mathrm{eff}}(\nu,t)}}.
}
$$

The effective number of independent samples is the time-bandwidth product,

$$
\boxed{
M_{\mathrm{eff}}(\nu,t)
\simeq
\eta(\nu,t)\Delta\nu\tau,
}
$$

so the voltage-average residual scales as

$$
\boxed{
\sigma_{\overline{x},i}(\nu,t)
\simeq
\frac{\sigma_{x,i}(\nu,t)}
{\sqrt{\eta(\nu,t)\Delta\nu\tau}}.
}
$$

This is the basic radiometer scaling. Doubling the integration time $\tau$ or doubling the channel bandwidth $\Delta\nu$ increases the number of statistically independent samples by a factor of two and reduces the residual amplitude by $1/\sqrt{2}$.

The central limit theorem is the reason this residual becomes Gaussian even when the detailed microscopic voltage distribution is not perfectly Gaussian: a normalized sum of many independent, finite-variance samples approaches a Gaussian random variable.

There is one important interferometric caveat. A correlator does not usually estimate the sky visibility by first averaging $x_i$ and $x_j$ separately and then multiplying those averages. It estimates the visibility by averaging instantaneous voltage products. Define

$$
z_{ij,m}(\nu,t)
=
x_i(\nu,t,m)x_j^\ast(\nu,t,m).
$$

The visibility estimator is

$$
\boxed{
d_{ij}(\nu,t)
=
\frac{1}{M_{\mathrm{eff}}(\nu,t)}
\sum_{m=1}^{M_{\mathrm{eff}}(\nu,t)}
z_{ij,m}(\nu,t).
}
$$

The same central-limit logic now applies to the product samples $z_{ij,m}$. Write

$$
d_{ij}(\nu,t)
=
V_{ij}(\nu,t)+n_{ij}(\nu,t),
$$

with

$$
V_{ij}(\nu,t)
=
\left\langle z_{ij,m}(\nu,t)\right\rangle.
$$

Then

$$
\boxed{
\mathrm{Var}\left[n_{ij}(\nu,t)\right]
=
\frac{\mathrm{Var}\left[z_{ij,m}(\nu,t)\right]}
{M_{\mathrm{eff}}(\nu,t)}.
}
$$

For a simple deterministic-signal plus receiver-noise toy model,

$$
x_i=s_i+\epsilon_i,
\qquad
x_j=s_j+\epsilon_j,
$$

with independent receiver noises $\epsilon_i$ and $\epsilon_j$, one product sample is

$$
\begin{aligned}
z_{ij,m}
&=(s_i+\epsilon_i)(s_j+\epsilon_j)^\ast \\
&=s_i s_j^\ast
+s_i\epsilon_j^\ast
+\epsilon_i s_j^\ast
+\epsilon_i\epsilon_j^\ast.
\end{aligned}
$$

The mean is the signal visibility in this toy model,

$$
\left\langle z_{ij,m}\right\rangle=s_i s_j^\ast,
$$

and the finite integration leaves a residual random part made of the three terms containing $\epsilon_i$ or $\epsilon_j$. For independent circular Gaussian receiver noises,

$$
\mathrm{Var}\left[z_{ij,m}\right]
=
|s_i|^2\sigma^2_{x,j}
+|s_j|^2\sigma^2_{x,i}
+\sigma^2_{x,i}\sigma^2_{x,j}.
$$

After averaging $M_{\mathrm{eff}}$ product samples,

$$
\boxed{
\mathrm{Var}\left[n_{ij}\right]
=
\frac{|s_i|^2\sigma^2_{x,j}
+|s_j|^2\sigma^2_{x,i}
+\sigma^2_{x,i}\sigma^2_{x,j}}
{M_{\mathrm{eff}}}.
}
$$

This toy equation shows how residual voltage noise propagates into residual visibility noise.

For real radio interferometry, the sky electric field itself is also a stochastic voltage source. The cleaner general expression is therefore not written in terms of deterministic antenna voltages $s_i$ and $s_j$, but in terms of total voltage powers

$$
A_i(\nu,t)=\left\langle |x_i(\nu,t,m)|^2\right\rangle,
\qquad
A_j(\nu,t)=\left\langle |x_j(\nu,t,m)|^2\right\rangle.
$$

For complex Gaussian voltages, the product-sample variance becomes

$$
\boxed{
\mathrm{Var}\left[z_{ij,m}(\nu,t)\right]
=
A_i(\nu,t)A_j(\nu,t),
}
$$

so the post-correlation visibility-noise variance is

$$
\boxed{
\mathrm{Var}\left[n_{ij}(\nu,t)\right]
\approx
\frac{A_i(\nu,t)A_j(\nu,t)}
{\eta(\nu,t)\Delta\nu\tau}.
}
$$

This is the bridge between the antenna-level integration picture and the autocorrelation-based visibility-noise expression derived below.

### 5.3 Why the cross-visibility estimator is an average of voltage products

A correlator estimates the cross-correlation between two antenna voltages. For antennas $i$ and $j$, the hardware operation is multiply and accumulate:

$$
\boxed{
d_{ij}(\nu,t)
=
\frac{1}{M(\nu,t)}
\sum_{m=1}^{M(\nu,t)}
x_i(\nu,t,m)x_j^\ast(\nu,t,m).
}
$$

This is the sample estimate of the ensemble average

$$
V_{ij}(\nu,t)
=
\left\langle
x_i(\nu,t,m)x_j^\ast(\nu,t,m)
\right\rangle.
$$

Substitute the sky-voltage model and assume the receiver noises in different antennas are independent:

$$
\left\langle
x_i^{\mathrm{rx}}(\nu,t,m)
\left[x_j^{\mathrm{rx}}(\nu,t,m)\right]^\ast
\right\rangle=0,
\qquad i\neq j.
$$

Also assume receiver noise is independent of the sky electric field. Then the mean cross-correlation is the sky term:

$$
\left\langle
x_i(\nu,t,m)x_j^\ast(\nu,t,m)
\right\rangle
=
\left\langle
x_i^{\mathrm{sky}}(\nu,t,m)
\left[x_j^{\mathrm{sky}}(\nu,t,m)\right]^\ast
\right\rangle.
$$

Using the uncorrelated-direction sky relation above gives

$$
V_{ij}(\nu,t)
=
\int d\Omega\;
J_i(\hat{s},\nu,t)J_j^\ast(\hat{s},\nu,t)
I_x(\hat{s},\nu,t)
\exp\left[-2\pi i\frac{\nu}{c}(\mathbf{r}_i-\mathbf{r}_j)\cdot\hat{s}\right].
$$

Let

$$
\mathbf{b}_{ij}=\mathbf{r}_i-\mathbf{r}_j.
$$

Then

$$
\boxed{
V_{ij}(\nu,t)
=
\int d\Omega\;
J_i(\hat{s},\nu,t)J_j^\ast(\hat{s},\nu,t)
I_x(\hat{s},\nu,t)
\exp\left[-2\pi i\frac{\nu}{c}\mathbf{b}_{ij}\cdot\hat{s}\right].
}
$$

This is the scalar `xx` form of the RIME used earlier. Therefore the correlator estimator $d_{ij}$ is an estimator of the RIME visibility $V_{ij}$.

In the redundant notation of this notebook, for $ij\in\mathcal{B}_g$,

$$
\left\langle d_{ij}(\nu,t)\right\rangle
=V_{ij}(\nu,t)
=s_g(\nu,t)
$$

for a perfectly redundant group.

### 5.4 Where $M\approx\Delta\nu\tau$ comes from

A real correlator first channelizes the voltage stream. After channelization, each frequency channel has an effective bandwidth

$$
\Delta\nu.
$$

A bandwidth-limited complex voltage stream has a correlation time of order

$$
\Delta t_{\mathrm{corr}}
\sim
\frac{1}{\Delta\nu}.
$$

A simple way to see this is Fourier duality: restricting a signal to a frequency width $\Delta\nu$ spreads its time-domain correlation function over a width of order $1/\Delta\nu$. Samples separated by much less than $1/\Delta\nu$ are strongly correlated; samples separated by much more than $1/\Delta\nu$ are approximately independent.

If the correlator integrates for time

$$
\tau,
$$

then the number of independent complex samples is approximately

$$
M
\sim
\frac{\tau}{\Delta t_{\mathrm{corr}}}
\sim
\Delta\nu\tau.
$$

This is the time-bandwidth product. In a digital FX correlator, this appears operationally as follows:

1. The antenna voltage is digitized.
2. A channelizer, such as an FFT or polyphase filter bank, produces complex samples in channels of width $\Delta\nu$.
3. Within each channel, neighboring independent complex samples are spaced by roughly $1/\Delta\nu$.
4. The correlator multiply-accumulates those products over an integration time $\tau$.

Thus

$$
\boxed{
M(\nu,t)
\approx
\Delta\nu\tau.
}
$$

More generally, channel windows, overlap in a polyphase filter bank, flagging, finite quantization, and nonuniform weights change the effective number of independent samples. We therefore write

$$
\boxed{
M(\nu,t)=\eta(\nu,t)\Delta\nu\tau,
}
$$

where $\eta(\nu,t)$ is an efficiency factor. If samples are lost to flagging or are correlated by windowing, then $\eta$ is smaller than the ideal value.

### 5.5 Autos contain sky plus receiver power

The autocorrelation estimator for antenna $i$ is

$$
a_i(\nu,t)
=
\frac{1}{M(\nu,t)}
\sum_{m=1}^{M(\nu,t)}
\left|x_i(\nu,t,m)\right|^2.
$$

Its expectation is the total voltage power entering antenna $i$:

$$
A_i(\nu,t)
=
\left\langle
\left|x_i(\nu,t,m)\right|^2
\right\rangle.
$$

Using

$$
x_i=x_i^{\mathrm{sky}}+x_i^{\mathrm{rx}},
$$

we get

$$
A_i
=
\left\langle
\left|x_i^{\mathrm{sky}}\right|^2
\right\rangle
+
\left\langle
\left|x_i^{\mathrm{rx}}\right|^2
\right\rangle
+2\mathrm{Re}\left\langle
x_i^{\mathrm{sky}}
\left[x_i^{\mathrm{rx}}\right]^\ast
\right\rangle.
$$

The sky voltage and receiver voltage are independent, so the cross term vanishes:

$$
\left\langle
x_i^{\mathrm{sky}}
\left[x_i^{\mathrm{rx}}\right]^\ast
\right\rangle=0.
$$

Therefore

$$
\boxed{
A_i(\nu,t)=A_i^{\mathrm{sky}}(\nu,t)+A_i^{\mathrm{rx}}(\nu,t).
}
$$

For the scalar sky model,

$$
A_i^{\mathrm{sky}}(\nu,t)
=
\int d\Omega\;
\left|J_i(\hat{s},\nu,t)\right|^2
I_x(\hat{s},\nu,t),
$$

and $A_i^{\mathrm{rx}}$ is the receiver contribution. In temperature language, this is the same idea as

$$
T_{\mathrm{sys},i}(\nu,t)
=
T_{\mathrm{sky},i}(\nu,t)+T_{\mathrm{rx},i}(\nu,t).
$$

So the measured autos do pick up the sky. That is expected. The auto-based noise estimate is estimating the variance from the total system power, not from receiver noise alone.

### 5.6 Why using autos does not double-count the mean visibility signal

Now derive the variance of the cross-visibility estimator around its mean. Define one instantaneous product sample

$$
z_m(\nu,t)=x_i(\nu,t,m)x_j^\ast(\nu,t,m).
$$

The visibility estimator is

$$
d_{ij}(\nu,t)=\frac{1}{M}\sum_{m=1}^{M}z_m(\nu,t).
$$

The mean is

$$
\left\langle z_m(\nu,t)\right\rangle
=V_{ij}(\nu,t).
$$

The variance of the estimator is

$$
\mathrm{Var}\left[d_{ij}(\nu,t)\right]
=
\left\langle
\left|d_{ij}(\nu,t)-V_{ij}(\nu,t)\right|^2
\right\rangle.
$$

If different samples $m$ are independent, then

$$
\mathrm{Var}\left[d_{ij}(\nu,t)\right]
=
\frac{1}{M}\mathrm{Var}\left[z_m(\nu,t)\right].
$$

For complex Gaussian voltages, Isserlis' theorem gives

$$
\left\langle
|x_i|^2|x_j|^2
\right\rangle
=
A_iA_j+|V_{ij}|^2.
$$

Since

$$
\left|\left\langle z_m\right\rangle\right|^2
=|V_{ij}|^2,
$$

we get

$$
\mathrm{Var}\left[z_m\right]
=
\left\langle |z_m|^2\right\rangle
-\left|\left\langle z_m\right\rangle\right|^2
=
A_iA_j.
$$

Therefore

$$
\boxed{
\mathrm{Var}\left[d_{ij}(\nu,t)\right]
=
\frac{A_i(\nu,t)A_j(\nu,t)}{M(\nu,t)}.
}
$$

Using

$$
M(\nu,t)=\eta(\nu,t)\Delta\nu\tau,
$$

gives

$$
\boxed{
\sigma^2_{ij}(\nu,t)
\equiv
\mathrm{Var}\left[d_{ij}(\nu,t)\right]
\approx
\frac{A_i(\nu,t)A_j(\nu,t)}
{\eta(\nu,t)\Delta\nu\tau}.
}
$$

The measured autocorrelations $a_i(\nu,t)$ and $a_j(\nu,t)$ estimate $A_i(\nu,t)$ and $A_j(\nu,t)$. Therefore the practical estimator is

$$
\boxed{
\widehat{\sigma}^2_{ij}(\nu,t)
\approx
\frac{a_i(\nu,t)a_j(\nu,t)}
{\eta(\nu,t)\Delta\nu\tau}.
}
$$

If $\eta(\nu,t)=1$, this becomes

$$
\widehat{\sigma}^2_{ij}(\nu,t)
\approx
\frac{a_i(\nu,t)a_j(\nu,t)}
{\Delta\nu\tau}.
$$

The noise standard deviation is therefore

$$
\boxed{
\widehat{\sigma}_{ij}(\nu,t)
\approx
\frac{\sqrt{a_i(\nu,t)a_j(\nu,t)}}
{\sqrt{\eta(\nu,t)\Delta\nu\tau}}.
}
$$

This answers the conceptual worry directly. The autos contain sky plus receiver power, but the finite-sample variance of the cross visibility is also controlled by sky plus receiver power. The coherent sky visibility $V_{ij}$ is the mean of the estimator. The autos determine the scatter of the estimator around that mean.

### 5.7 Toy example: identical antennas

If

$$
a_i(\nu,t)=a_j(\nu,t)=a(\nu,t),
$$

then

$$
\widehat{\sigma}^2_{ij}(\nu,t)
\approx
\frac{a^2(\nu,t)}{\eta(\nu,t)\Delta\nu\tau}.
$$

Doubling the channel width gives

$$
\Delta\nu\rightarrow 2\Delta\nu,
$$

so

$$
\widehat{\sigma}_{ij}(\nu,t)
\rightarrow
\frac{1}{\sqrt{2}}
\widehat{\sigma}_{ij}(\nu,t).
$$

Doubling the integration time gives

$$
\tau\rightarrow 2\tau,
$$

so again

$$
\widehat{\sigma}_{ij}(\nu,t)
\rightarrow
\frac{1}{\sqrt{2}}
\widehat{\sigma}_{ij}(\nu,t).
$$

## 6. Coherent redundant averaging

The coherent redundant average is the weighted average of complex visibilities before forming any power-like quantity:

$$
\overline{d}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)d_{ij}(\nu,t).
$$

The weights obey

$$
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)=1.
$$

Substitute the redundant toy model

$$
d_{ij}(\nu,t)=s_g(\nu,t)+n_{ij}(\nu,t).
$$

Then

$$
\overline{d}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)
\left[s_g(\nu,t)+n_{ij}(\nu,t)\right].
$$

Distribute the sum:

$$
\overline{d}_g(\nu,t)
=
s_g(\nu,t)
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)
+
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)n_{ij}(\nu,t).
$$

Using the normalization of the weights,

$$
\boxed{
\overline{d}_g(\nu,t)
=
s_g(\nu,t)
+
\overline{n}_g(\nu,t)
}
$$

where

$$
\overline{n}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)n_{ij}(\nu,t).
$$

Thus coherent redundant averaging preserves the common redundant sky visibility and averages down the noise.

## 7. Noise reduction from coherent redundant averaging

The variance of the coherently averaged noise is

$$
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\left\langle
\overline{n}_g(\nu,t)
\overline{n}_g^\ast(\nu,t)
\right\rangle.
$$

Substitute the weighted sum:

$$
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\left\langle
\left[
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)n_{ij}(\nu,t)
\right]
\left[
\sum_{i'j'\in\mathcal{B}_g}
W_{i'j',g}(\nu,t)n_{i'j'}(\nu,t)
\right]^\ast
\right\rangle.
$$

Therefore

$$
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\sum_{ij\in\mathcal{B}_g}
\sum_{i'j'\in\mathcal{B}_g}
W_{ij,g}(\nu,t)W^\ast_{i'j',g}(\nu,t)
\left\langle
n_{ij}(\nu,t)n^\ast_{i'j'}(\nu,t)
\right\rangle.
$$

If the noise in different baselines is independent, then

$$
\left\langle
n_{ij}(\nu,t)n^\ast_{i'j'}(\nu,t)
\right\rangle
=
\sigma^2_{ij}(\nu,t)
\delta_{ij,i'j'}.
$$

Then

$$
\boxed{
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\sum_{ij\in\mathcal{B}_g}
\left|W_{ij,g}(\nu,t)\right|^2
\sigma^2_{ij}(\nu,t)
}
$$

For equal weights,

$$
W_{ij,g}(\nu,t)=\frac{1}{N_{b,g}},
$$

and equal baseline noise variances,

$$
\sigma^2_{ij}(\nu,t)=\sigma^2_g(\nu,t),
$$

we get

$$
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\sum_{ij\in\mathcal{B}_g}
\left(\frac{1}{N_{b,g}}\right)^2
\sigma^2_g(\nu,t).
$$

There are $N_{b,g}$ terms, so

$$
\boxed{
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\frac{\sigma^2_g(\nu,t)}{N_{b,g}}
}
$$

and

$$
\boxed{
\sigma^{\mathrm{coh}}_g(\nu,t)
=
\frac{\sigma_g(\nu,t)}{\sqrt{N_{b,g}}}
}
$$

### Inverse-variance weights

If different baselines have different noise variances, a common choice is

$$
W_{ij,g}(\nu,t)
=
\frac{\sigma^{-2}_{ij}(\nu,t)}
{\sum_{i'j'\in\mathcal{B}_g}
\sigma^{-2}_{i'j'}(\nu,t)}.
$$

With independent baseline noise, the resulting variance is

$$
\boxed{
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\left[
\sum_{ij\in\mathcal{B}_g}
\sigma^{-2}_{ij}(\nu,t)
\right]^{-1}
}
$$

## 8. Toy model: coherent average of four redundant baselines

Let

$$
N_{b,g}=4.
$$

Label the four antenna pairs as

$$
ij_1,\quad ij_2,\quad ij_3,\quad ij_4.
$$

Assume

$$
d_{ij_a}(\nu,t)=s_g(\nu,t)+n_{ij_a}(\nu,t)
\qquad
\text{for}\quad a=1,2,3,4.
$$

With equal weights,

$$
\overline{d}_g(\nu,t)
=
\frac{1}{4}
\left[
 d_{ij_1}(\nu,t)
+d_{ij_2}(\nu,t)
+d_{ij_3}(\nu,t)
+d_{ij_4}(\nu,t)
\right].
$$

Substitute the signal plus noise model:

$$
\overline{d}_g(\nu,t)
=
\frac{1}{4}
\left[
 s_g(\nu,t)+n_{ij_1}(\nu,t)
+s_g(\nu,t)+n_{ij_2}(\nu,t)
+s_g(\nu,t)+n_{ij_3}(\nu,t)
+s_g(\nu,t)+n_{ij_4}(\nu,t)
\right].
$$

Therefore

$$
\overline{d}_g(\nu,t)
=
s_g(\nu,t)
+
\frac{1}{4}
\left[
 n_{ij_1}(\nu,t)
+n_{ij_2}(\nu,t)
+n_{ij_3}(\nu,t)
+n_{ij_4}(\nu,t)
\right].
$$

The signal is unchanged. If each noise term has variance $\sigma^2_g(\nu,t)$ and the four noise terms are independent, then

$$
\mathrm{Var}\left[
\frac{1}{4}
\left(
 n_{ij_1}(\nu,t)
+n_{ij_2}(\nu,t)
+n_{ij_3}(\nu,t)
+n_{ij_4}(\nu,t)
\right)
\right]
=
\frac{1}{16}
\left[
\sigma^2_g(\nu,t)
+\sigma^2_g(\nu,t)
+\sigma^2_g(\nu,t)
+\sigma^2_g(\nu,t)
\right].
$$

Thus

$$
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\frac{\sigma^2_g(\nu,t)}{4}.
$$

The noise standard deviation is reduced by two.

## 9. Coherent signal loss from imperfect redundancy

Suppose the baselines have small baseline-dependent phase errors. A simple model is

$$
d_{ij}(\nu,t)
=
s_g(\nu,t)e^{i\epsilon_{ij}(\nu,t)}
+
n_{ij}(\nu,t).
$$

The coherently averaged signal part is

$$
\overline{s}^{\mathrm{coh}}_g(\nu,t)
=
s_g(\nu,t)
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)e^{i\epsilon_{ij}(\nu,t)}.
$$

The signal amplitude is multiplied by

$$
\Gamma_g(\nu,t)
=
\left|
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)e^{i\epsilon_{ij}(\nu,t)}
\right|.
$$

If every phase error is zero, then

$$
\Gamma_g(\nu,t)=1.
$$

If the phase errors scatter, then

$$
0\leq \Gamma_g(\nu,t)<1.
$$

This is a simple analytic model for coherent signal loss.

## 10. Time averaging as an operation in LST

Let

$$
\overline{d}_g(\nu,t)
$$

be the coherently averaged visibility for redundant group $g$.

Define a time-averaging kernel centered at LST $t_0$:

$$
K_T(t-t_0).
$$

Normalize it so that

$$
\int_{-\infty}^{+\infty}K_T(t-t_0)dt=1.
$$

The time-averaged visibility is

$$
\boxed{
\overline{d}_{g,T}(\nu,t_0)
=
\int_{-\infty}^{+\infty}
K_T(t-t_0)
\overline{d}_g(\nu,t)dt
}
$$

The same operation acts on the signal and the noise:

$$
\overline{d}_{g,T}(\nu,t_0)
=
\overline{s}_{g,T}(\nu,t_0)
+
\overline{n}_{g,T}(\nu,t_0).
$$

For a top-hat average of width $T$, measured in LST hours,

$$
K_T(t-t_0)=
\begin{cases}
1/T, & |t-t_0|\leq T/2,\\
0, & |t-t_0|>T/2.
\end{cases}
$$

## 11. Noise reduction from time averaging

Suppose the top-hat time average contains $M_T$ independent equal-noise visibility samples. Then the average is

$$
\overline{d}_{g,T}(\nu,t_0)
=
\frac{1}{M_T}
\sum_{m=1}^{M_T}
\overline{d}_g(\nu,t_m).
$$

The time-averaged noise is

$$
\overline{n}_{g,T}(\nu,t_0)
=
\frac{1}{M_T}
\sum_{m=1}^{M_T}
\overline{n}_g(\nu,t_m).
$$

If each sample has variance

$$
\mathrm{Var}\left[\overline{n}_g(\nu,t_m)\right]
=
\sigma^2_{g,\mathrm{coh}}(\nu,t_m),
$$

and if all samples have the same variance

$$
\sigma^2_{g,\mathrm{coh}}(\nu,t_m)=\sigma^2_{g,\mathrm{coh}}(\nu,t_0),
$$

then

$$
\mathrm{Var}\left[\overline{n}_{g,T}(\nu,t_0)\right]
=
\frac{\sigma^2_{g,\mathrm{coh}}(\nu,t_0)}{M_T}.
$$

Thus the noise standard deviation is reduced by

$$
\sqrt{M_T}.
$$

Combining equal-noise redundant averaging and equal-noise time averaging gives

$$
\boxed{
\sigma_{g,\mathrm{coh+time}}(\nu,t_0)
=
\frac{\sigma_g(\nu,t_0)}
{\sqrt{N_{b,g}M_T}}
}
$$

This equation is only about thermal-noise reduction. It does not say that the sky signal is unchanged by time averaging.

## 12. Time averaging as a low-pass filter along LST

Time averaging smooths the visibility along the LST axis. Therefore it suppresses high fringe-rate or high LST-frequency structure.

Consider a single temporal Fourier mode in the sky visibility:

$$
s_g(\nu,t)
=
S_g(\nu,f)
\exp\left(2\pi i f t\right),
$$

where

- $f$ is the LST Fourier frequency in cycles per LST hour.
- $S_g(\nu,f)$ is the complex amplitude of that mode.

Apply a continuous top-hat average of width $T$:

$$
s_{g,T}(\nu,t_0)
=
\frac{1}{T}
\int_{t_0-T/2}^{t_0+T/2}
S_g(\nu,f)
\exp\left(2\pi i f t\right)dt.
$$

Let

$$
u=t-t_0.
$$

Then

$$
s_{g,T}(\nu,t_0)
=
S_g(\nu,f)
\exp\left(2\pi i f t_0\right)
\frac{1}{T}
\int_{-T/2}^{+T/2}
\exp\left(2\pi i f u\right)du.
$$

The integral is

$$
\int_{-T/2}^{+T/2}
\exp\left(2\pi i f u\right)du
=
\frac{\sin(\pi fT)}{\pi f}.
$$

Therefore

$$
s_{g,T}(\nu,t_0)
=
S_g(\nu,f)
\exp\left(2\pi i f t_0\right)
\frac{\sin(\pi fT)}{\pi fT}.
$$

Define

$$
\mathrm{sinc}(x)=\frac{\sin(x)}{x}.
$$

Then the top-hat time average multiplies the mode by

$$
\boxed{
H_T(f)=\mathrm{sinc}(\pi fT)
}
$$

The fractional amplitude retained is

$$
R_T(f)=\left|H_T(f)\right|
=
\left|\mathrm{sinc}(\pi fT)\right|.
$$

The fractional amplitude loss is

$$
L_T(f)=1-R_T(f).
$$

For small $\pi fT$,

$$
\mathrm{sinc}(\pi fT)
\approx
1-
\frac{(\pi fT)^2}{6}.
$$

If one wants the fractional amplitude loss to be less than $\epsilon$, require

$$
\frac{(\pi f_{\max}T)^2}{6}\leq\epsilon.
$$

Solving for $T$ gives

$$
\boxed{
T\leq
\frac{\sqrt{6\epsilon}}{\pi f_{\max}}
}
$$

where $T$ is in LST hours if $f_{\max}$ is in cycles per LST hour.

### Example time-average limits

For 1 percent amplitude loss,

$$
\epsilon=0.01,
$$

so

$$
T\leq\frac{0.078}{f_{\max}}.
$$

For 5 percent amplitude loss,

$$
\epsilon=0.05,
$$

so

$$
T\leq\frac{0.174}{f_{\max}}.
$$

## 13. Relating 21 cm angular structure to an LST averaging scale

Let a 21 cm sky structure have an angular scale along the drift direction

$$
\theta_{21,\mathrm{deg}}.
$$

At declination $\delta$, the drift rate along the constant-declination circle is approximately

$$
\omega_{\mathrm{drift}}(\delta)
=
15|\cos\delta|
\quad
\mathrm{degrees\;per\;LST\;hour}.
$$

The time for the sky to drift across one such angular scale is

$$
T_{21}
\approx
\frac{\theta_{21,\mathrm{deg}}}
{15|\cos\delta|}.
$$

A corresponding LST Fourier frequency is

$$
f_{21}
\approx
\frac{1}{T_{21}}
=
\frac{15|\cos\delta|}
{\theta_{21,\mathrm{deg}}}.
$$

To keep this structure from being significantly smoothed, choose a time-average width

$$
T\ll T_{21}.
$$

Using the sinc-loss estimate, choose

$$
\boxed{
T\leq
\frac{\sqrt{6\epsilon}}{\pi}
\frac{\theta_{21,\mathrm{deg}}}
{15|\cos\delta|}
}
$$

For a zenith-dominated drift-scan feature, one often uses

$$
\delta\approx\phi_{\mathrm{site}},
$$

where $\phi_{\mathrm{site}}$ is the observatory latitude.

### Toy example: 10 degree structure at HERA latitude

Take

$$
\phi_{\mathrm{site}}=-30.7215^{\circ}.
$$

Then

$$
|\cos\phi_{\mathrm{site}}|\approx0.8597.
$$

For

$$
\theta_{21,\mathrm{deg}}=10^{\circ},
$$

the frequency scale is

$$
f_{21}
\approx
\frac{15\times0.8597}{10}
\approx
1.29\;\mathrm{cycles\;per\;LST\;hour}.
$$

For 5 percent amplitude loss,

$$
T\leq
\frac{0.174}{1.29}
\approx
0.135\;\mathrm{LST\;hour}.
$$

This is about 8.1 minutes.

## 14. Baseline fringe spacing as an LST-frequency scale

For a projected baseline length

$$
b_g,
$$

and wavelength

$$
\lambda=\frac{c}{\nu},
$$

the small-angle fringe spacing is

$$
\Delta\theta_{g}(\nu)
\approx
\frac{\lambda}{b_g}
=
\frac{c}{\nu b_g}
\quad
\mathrm{radians}.
$$

In degrees,

$$
\Delta\theta_{g,\mathrm{deg}}(\nu)
\approx
\frac{180}{\pi}
\frac{c}{\nu b_g}.
$$

At latitude $\phi_{\mathrm{site}}$, the effective LST width for a full drift around the smaller latitude circle is

$$
\Delta t^{\phi}_{g}(\nu)
\approx
\frac{
\Delta\theta_{g,\mathrm{deg}}(\nu)
}
{15|\cos\phi_{\mathrm{site}}|}
\quad
\mathrm{LST\;hours}.
$$

The corresponding LST Fourier frequency is

$$
\boxed{
f^{\phi}_{g}(\nu)
\approx
\frac{1}{\Delta t^{\phi}_{g}(\nu)}
=
\frac{15|\cos\phi_{\mathrm{site}}|}
{\Delta\theta_{g,\mathrm{deg}}(\nu)}
}
$$

This is the kind of frequency scale used when a bandstop is centered using the fringe-spacing prediction.

## 15. Coherent power, incoherent power, and noise bias

A coherent visibility average forms

$$
\overline{d}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)d_{ij}(\nu,t)
$$

and then one may form a power-like quantity

$$
P^{\mathrm{coh}}_g(\nu,t)
=
\left|\overline{d}_g(\nu,t)\right|^2.
$$

This contains a noise bias:

$$
\left\langle
\left|\overline{d}_g(\nu,t)\right|^2
\right\rangle
=
\left|s_g(\nu,t)\right|^2
+
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right].
$$

A bias-subtracted coherent auto-power toy estimator is

$$
\widehat{P}^{\mathrm{coh}}_g(\nu,t)
=
\left|\overline{d}_g(\nu,t)\right|^2
-
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right].
$$

An incoherent power average squares each baseline first and then averages:

$$
P^{\mathrm{inc}}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)
\left|d_{ij}(\nu,t)\right|^2.
$$

The corresponding bias-subtracted toy estimator is

$$
\widehat{P}^{\mathrm{inc}}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)
\left[
\left|d_{ij}(\nu,t)\right|^2
-
\sigma^2_{ij}(\nu,t)
\right].
$$

For perfect redundancy, both coherent and incoherent estimators target

$$
\left|s_g(\nu,t)\right|^2.
$$

However, coherent averaging can lose signal if the baselines are not truly redundant in phase or amplitude. Incoherent averaging is less sensitive to phase cancellation, but it does not reduce visibility-domain noise in the same way before squaring.

## 16. Split cross-power to avoid noise bias

A common way to avoid direct noise bias is to split the data into two statistically independent data sets, labeled $A$ and $B$:

$$
\overline{d}^{A}_g(\nu,t)
=
s_g(\nu,t)+\overline{n}^{A}_g(\nu,t),
$$

and

$$
\overline{d}^{B}_g(\nu,t)
=
s_g(\nu,t)+\overline{n}^{B}_g(\nu,t).
$$

Assume

$$
\left\langle
\overline{n}^{A}_g(\nu,t)
\left[\overline{n}^{B}_g(\nu,t)\right]^\ast
\right\rangle=0.
$$

Then

$$
\left\langle
\overline{d}^{A}_g(\nu,t)
\left[\overline{d}^{B}_g(\nu,t)\right]^\ast
\right\rangle
=
\left|s_g(\nu,t)\right|^2.
$$

The cross-power has no additive thermal-noise bias if the two noise realizations are independent.

## 17. Operational summary

The basic visibility model is

$$
d_{ij}(\nu,t)=s_g(\nu,t)+e_{ij}(\nu,t)+n_{ij}(\nu,t),
\qquad ij\in\mathcal{B}_g.
$$

If the baselines are perfectly redundant, then

$$
e_{ij}(\nu,t)=0.
$$

The coherent redundant average is

$$
\overline{d}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)d_{ij}(\nu,t).
$$

For independent equal-variance baseline noise, coherent averaging gives

$$
\sigma_g(\nu,t)
\rightarrow
\frac{\sigma_g(\nu,t)}{\sqrt{N_{b,g}}}.
$$

The autocorrelation-based visibility-noise estimate is

$$
\widehat{\sigma}^2_{ij}(\nu,t)
\approx
\frac{a_i(\nu,t)a_j(\nu,t)}
{\eta(\nu,t)\Delta\nu\tau}.
$$

The time average is

$$
\overline{d}_{g,T}(\nu,t_0)
=
\int K_T(t-t_0)\overline{d}_g(\nu,t)dt.
$$

A top-hat time average over $M_T$ independent equal-noise samples gives

$$
\sigma_g(\nu,t)
\rightarrow
\frac{\sigma_g(\nu,t)}{\sqrt{M_T}}.
$$

But the same top-hat average multiplies an LST Fourier mode by

$$
H_T(f)=\mathrm{sinc}(\pi fT).
$$

Thus time averaging should satisfy

$$
T\leq
\frac{\sqrt{6\epsilon}}{\pi f_{\max}}
$$

if the goal is to keep amplitude loss below $\epsilon$ for all LST Fourier modes up to $f_{\max}$.

## 18. Coherent versus incoherent redundant PSPEC averages for signal plus Gaussian noise

This section uses the same notation as above and fixes the polarization to `xx`. The goal is to compare two operations that look similar in code but are mathematically different:

1. Average complex redundant visibilities first, then delay-transform and square.
2. Delay-transform and square each redundant visibility first, then average the powers.

The key point is that these two operations do not commute because squaring is nonlinear.

### 18.1 First-principles signal plus noise model

Let $g$ label a redundant baseline group and let

$$
\mathcal{B}_g
=
\{ij:\;ij\;\text{belongs to redundant group}\;g\}
$$

contain

$$
N_{b,g}=|\mathcal{B}_g|
$$

baseline samples. In this section we assume perfect redundancy. That means the sky signal is the same for every baseline pair in the group:

$$
\boxed{
d_{ij}(\nu,t)=s_g(\nu,t)+n_{ij}(\nu,t),
\qquad ij\in\mathcal{B}_g.
}
$$

Here

- $d_{ij}(\nu,t)$ is the measured complex visibility for antenna pair $ij$.
- $s_g(\nu,t)$ is the fixed redundant sky signal shared by every baseline in group $g$.
- $n_{ij}(\nu,t)$ is the thermal-noise visibility for pair $ij$.

We take the noise to be complex Gaussian with known mean and covariance:

$$
\left\langle n_{ij}(\nu,t)\right\rangle
=
\mu_{ij}(\nu,t),
$$

and

$$
\left\langle
\left[n_{ij}(\nu,t)-\mu_{ij}(\nu,t)\right]
\left[n_{i'j'}(\nu',t)-\mu_{i'j'}(\nu',t)\right]^\ast
\right\rangle
=
C_{ij,i'j'}(\nu,\nu';t).
$$

For thermal noise after calibration, the working model is usually zero mean:

$$
\mu_{ij}(\nu,t)=0.
$$

If the noise has a nonzero mean, that mean behaves like an additive systematic visibility and should be subtracted or modeled separately. The rest of this derivation assumes

$$
\boxed{\left\langle n_{ij}(\nu,t)\right\rangle=0.}
$$

For independent baseline noise, the covariance simplifies to

$$
C_{ij,i'j'}(\nu,\nu';t)
=
C_{ij}(\nu,\nu';t)\delta_{ij,i'j'}.
$$

For independent frequency channels as a toy model,

$$
C_{ij}(\nu,\nu';t)
=
\sigma^2_{ij}(\nu,t)\delta(\nu-\nu').
$$

### 18.2 Delay transform as a linear operation

Let $\psi(\nu)$ be the frequency taper or bandpass weight used in the delay transform. Define the delay transform of any visibility-like quantity $x(\nu,t)$ as

$$
\widetilde{x}(\tau,t)
=
\int d\nu\;\psi(\nu)x(\nu,t)
\exp\left(2\pi i\nu\tau\right).
$$

Because this operation is linear,

$$
\widetilde{d}_{ij}(\tau,t)
=
\widetilde{s}_g(\tau,t)
+
\widetilde{n}_{ij}(\tau,t).
$$

The delay-transformed signal is

$$
\widetilde{s}_g(\tau,t)
=
\int d\nu\;\psi(\nu)s_g(\nu,t)
\exp\left(2\pi i\nu\tau\right),
$$

and the delay-transformed noise is

$$
\widetilde{n}_{ij}(\tau,t)
=
\int d\nu\;\psi(\nu)n_{ij}(\nu,t)
\exp\left(2\pi i\nu\tau\right).
$$

Since the noise has zero mean before the transform, it also has zero mean after the transform:

$$
\left\langle \widetilde{n}_{ij}(\tau,t)\right\rangle=0.
$$

Its variance is

$$
\left\langle
\left|\widetilde{n}_{ij}(\tau,t)\right|^2
\right\rangle
=
\int d\nu\int d\nu'\;
\psi(\nu)\psi^\ast(\nu')
\exp\left[2\pi i(\nu-\nu')\tau\right]
C_{ij}(\nu,\nu';t).
$$

For independent frequency channels, this reduces to

$$
\boxed{
\sigma^2_{ij,\tau}(\tau,t)
\equiv
\left\langle
\left|\widetilde{n}_{ij}(\tau,t)\right|^2
\right\rangle
=
\int d\nu\;|\psi(\nu)|^2\sigma^2_{ij}(\nu,t).
}
$$

A discrete-channel version, with channels $\nu_a$ and channel width $\Delta\nu$, is

$$
\boxed{
\sigma^2_{ij,\tau}(\tau,t)
\approx
\sum_a
(\Delta\nu)^2
|\psi(\nu_a)|^2
\sigma^2_{ij}(\nu_a,t).
}
$$

The detailed cosmological normalization of a power spectrum is not important for this comparison, so write it as a positive factor $\mathcal{C}_P(\tau)$. Define

$$
\boxed{
P_{S,g}(\tau,t)
=
\mathcal{C}_P(\tau)
\left|\widetilde{s}_g(\tau,t)\right|^2.
}
$$

### 18.3 Coherent redundant average first, then delay transform and square

Define a normalized coherent redundant average in the visibility domain:

$$
\overline{d}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)d_{ij}(\nu,t),
$$

with

$$
\sum_{ij\in\mathcal{B}_g}W_{ij,g}(\nu,t)=1.
$$

Substitute $d_{ij}=s_g+n_{ij}$:

$$
\overline{d}_g(\nu,t)
=
s_g(\nu,t)
+
\overline{n}_g(\nu,t),
$$

where

$$
\overline{n}_g(\nu,t)
=
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)n_{ij}(\nu,t).
$$

The signal survives unchanged because the weights sum to one. The noise variance becomes

$$
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\sum_{ij\in\mathcal{B}_g}
\sum_{i'j'\in\mathcal{B}_g}
W_{ij,g}(\nu,t)W^\ast_{i'j',g}(\nu,t)
\left\langle
n_{ij}(\nu,t)n^\ast_{i'j'}(\nu,t)
\right\rangle.
$$

For independent baseline noise,

$$
\boxed{
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\sum_{ij\in\mathcal{B}_g}
\left|W_{ij,g}(\nu,t)\right|^2
\sigma^2_{ij}(\nu,t).
}
$$

For equal weights and equal variances,

$$
W_{ij,g}(\nu,t)=\frac{1}{N_{b,g}},
\qquad
\sigma^2_{ij}(\nu,t)=\sigma^2_g(\nu,t),
$$

so

$$
\boxed{
\mathrm{Var}\left[\overline{n}_g(\nu,t)\right]
=
\frac{\sigma^2_g(\nu,t)}{N_{b,g}}.
}
$$

Thus the visibility-domain noise standard deviation goes down as

$$
\boxed{
\sigma_g(\nu,t)
\rightarrow
\frac{\sigma_g(\nu,t)}{\sqrt{N_{b,g}}}.
}
$$

Now delay-transform the coherently averaged visibility:

$$
\widetilde{\overline{d}}_g(\tau,t)
=
\widetilde{s}_g(\tau,t)
+
\widetilde{\overline{n}}_g(\tau,t).
$$

The corresponding auto-power-like estimator is

$$
\widehat{P}^{\mathrm{coh}}_g(\tau,t)
=
\mathcal{C}_P(\tau)
\left|
\widetilde{\overline{d}}_g(\tau,t)
\right|^2.
$$

Expanding from first principles,

$$
\widehat{P}^{\mathrm{coh}}_g(\tau,t)
=
\mathcal{C}_P(\tau)
\left|
\widetilde{s}_g(\tau,t)
+
\widetilde{\overline{n}}_g(\tau,t)
\right|^2.
$$

Therefore

$$
\widehat{P}^{\mathrm{coh}}_g(\tau,t)
=
P_{S,g}(\tau,t)
+
\mathcal{C}_P(\tau)
\left|\widetilde{\overline{n}}_g(\tau,t)\right|^2
+2\mathcal{C}_P(\tau)
\mathrm{Re}
\left[
\widetilde{s}_g(\tau,t)
\widetilde{\overline{n}}^\ast_g(\tau,t)
\right].
$$

The cross term has zero ensemble mean because the noise has zero mean:

$$
\left\langle
\widetilde{s}_g(\tau,t)
\widetilde{\overline{n}}^\ast_g(\tau,t)
\right\rangle=0.
$$

Thus

$$
\boxed{
\left\langle
\widehat{P}^{\mathrm{coh}}_g(\tau,t)
\right\rangle
=
P_{S,g}(\tau,t)
+
P_{N,g}^{\mathrm{redavg}}(\tau,t)
}
$$

where

$$
\boxed{
P_{N,g}^{\mathrm{redavg}}(\tau,t)
=
\mathcal{C}_P(\tau)
\left\langle
\left|
\widetilde{\overline{n}}_g(\tau,t)
\right|^2
\right\rangle.
}
$$

For independent baselines and independent frequency channels,

$$
\boxed{
P_{N,g}^{\mathrm{redavg}}(\tau,t)
=
\mathcal{C}_P(\tau)
\int d\nu\;|\psi(\nu)|^2
\sum_{ij\in\mathcal{B}_g}
\left|W_{ij,g}(\nu,t)\right|^2
\sigma^2_{ij}(\nu,t).
}
$$

For equal weights and equal variances,

$$
\boxed{
P_{N,g}^{\mathrm{redavg}}(\tau,t)
=
\frac{1}{N_{b,g}}
P_{N,g}^{\mathrm{single}}(\tau,t).
}
$$

This is the clean mathematical statement of why coherent redundant averaging lowers the thermal-noise contribution before forming power.

### 18.4 Power each redundant visibility first, then average powers incoherently

Now reverse the order. For each baseline pair $ij$, first delay-transform:

$$
\widetilde{d}_{ij}(\tau,t)
=
\widetilde{s}_g(\tau,t)
+
\widetilde{n}_{ij}(\tau,t).
$$

Then form the power for that one baseline:

$$
\widehat{P}_{ij}(\tau,t)
=
\mathcal{C}_P(\tau)
\left|
\widetilde{d}_{ij}(\tau,t)
\right|^2.
$$

Expanding,

$$
\widehat{P}_{ij}(\tau,t)
=
P_{S,g}(\tau,t)
+
\mathcal{C}_P(\tau)
\left|
\widetilde{n}_{ij}(\tau,t)
\right|^2
+2\mathcal{C}_P(\tau)
\mathrm{Re}
\left[
\widetilde{s}_g(\tau,t)
\widetilde{n}^\ast_{ij}(\tau,t)
\right].
$$

Now average these powers with normalized incoherent weights $V_{ij,g}(\tau,t)$:

$$
\sum_{ij\in\mathcal{B}_g}V_{ij,g}(\tau,t)=1,
$$

and

$$
\widehat{P}^{\mathrm{inc}}_g(\tau,t)
=
\sum_{ij\in\mathcal{B}_g}
V_{ij,g}(\tau,t)\widehat{P}_{ij}(\tau,t).
$$

Substitute the single-baseline power expression:

$$
\widehat{P}^{\mathrm{inc}}_g(\tau,t)
=
P_{S,g}(\tau,t)
+
\mathcal{C}_P(\tau)
\sum_{ij\in\mathcal{B}_g}
V_{ij,g}(\tau,t)
\left|
\widetilde{n}_{ij}(\tau,t)
\right|^2
+
2\mathcal{C}_P(\tau)
\mathrm{Re}
\left[
\widetilde{s}_g(\tau,t)
\sum_{ij\in\mathcal{B}_g}
V_{ij,g}(\tau,t)
\widetilde{n}^\ast_{ij}(\tau,t)
\right].
$$

Again, the final cross term has zero ensemble mean. Therefore

$$
\boxed{
\left\langle
\widehat{P}^{\mathrm{inc}}_g(\tau,t)
\right\rangle
=
P_{S,g}(\tau,t)
+
P_{N,g}^{\mathrm{incavg}}(\tau,t)
}
$$

where

$$
\boxed{
P_{N,g}^{\mathrm{incavg}}(\tau,t)
=
\sum_{ij\in\mathcal{B}_g}
V_{ij,g}(\tau,t)
P_{N,ij}^{\mathrm{single}}(\tau,t)
}
$$

and

$$
P_{N,ij}^{\mathrm{single}}(\tau,t)
=
\mathcal{C}_P(\tau)
\left\langle
\left|
\widetilde{n}_{ij}(\tau,t)
\right|^2
\right\rangle.
$$

For equal incoherent weights and equal baseline noise powers,

$$
V_{ij,g}(\tau,t)=\frac{1}{N_{b,g}},
\qquad
P_{N,ij}^{\mathrm{single}}(\tau,t)=P_{N,g}^{\mathrm{single}}(\tau,t),
$$

so

$$
\boxed{
P_{N,g}^{\mathrm{incavg}}(\tau,t)
=
P_{N,g}^{\mathrm{single}}(\tau,t).
}
$$

This is the crucial distinction: incoherent averaging reduces the scatter of the averaged power estimate, but it does not reduce the mean additive noise-power bias in the same way that coherent visibility averaging does.

### 18.5 Direct comparison

Under perfect redundancy, independent equal-variance baseline noise, and equal weights,

$$
\boxed{
\left\langle
\widehat{P}^{\mathrm{coh}}_g(\tau,t)
\right\rangle
=
P_{S,g}(\tau,t)
+
\frac{1}{N_{b,g}}P_{N,g}^{\mathrm{single}}(\tau,t)
}
$$

but

$$
\boxed{
\left\langle
\widehat{P}^{\mathrm{inc}}_g(\tau,t)
\right\rangle
=
P_{S,g}(\tau,t)
+
P_{N,g}^{\mathrm{single}}(\tau,t).
}
$$

Therefore, before noise-bias subtraction,

$$
\boxed{
\left\langle
\widehat{P}^{\mathrm{inc}}_g(\tau,t)
\right\rangle
-
\left\langle
\widehat{P}^{\mathrm{coh}}_g(\tau,t)
\right\rangle
=
\left(1-\frac{1}{N_{b,g}}\right)
P_{N,g}^{\mathrm{single}}(\tau,t).
}
$$

Equivalently,

$$
\boxed{
P_{N,g}^{\mathrm{redavg}}(\tau,t)
=
\frac{1}{N_{b,g}}P_{N,g}^{\mathrm{incavg}}(\tau,t)
}
$$

for the equal-noise toy model.

### 18.6 Toy model: four identical redundant baselines

Take four perfectly redundant baselines and suppress the explicit $(\tau,t)$ labels for readability. Let

$$
d_{ij_1}=s+n_{ij_1},
\qquad
d_{ij_2}=s+n_{ij_2},
\qquad
d_{ij_3}=s+n_{ij_3},
\qquad
d_{ij_4}=s+n_{ij_4},
$$

with

$$
n_{ij_k}\sim\mathcal{CN}(0,\sigma^2),
\qquad k=1,2,3,4.
$$

Here the delay transform has already been applied, so $s$ and $n_{ij_k}$ should be read as delay-domain quantities.

The coherent average is

$$
\overline{d}
=
\frac{1}{4}
\left(d_{ij_1}+d_{ij_2}+d_{ij_3}+d_{ij_4}\right)
=
s+
\frac{1}{4}
\left(n_{ij_1}+n_{ij_2}+n_{ij_3}+n_{ij_4}\right).
$$

The averaged noise has variance

$$
\mathrm{Var}\left[
\frac{1}{4}
\left(n_{ij_1}+n_{ij_2}+n_{ij_3}+n_{ij_4}\right)
\right]
=
\frac{\sigma^2}{4}.
$$

Thus

$$
\left\langle |\overline{d}|^2\right\rangle
=
|s|^2+\frac{\sigma^2}{4}.
$$

The incoherent average is

$$
P^{\mathrm{inc}}
=
\frac{1}{4}
\left(
|d_{ij_1}|^2+|d_{ij_2}|^2+|d_{ij_3}|^2+|d_{ij_4}|^2
\right).
$$

For each term,

$$
\left\langle |s+n_{ij_k}|^2\right\rangle
=
|s|^2+\sigma^2.
$$

Therefore

$$
\left\langle P^{\mathrm{inc}}\right\rangle
=
|s|^2+\sigma^2.
$$

So for four identical redundant baselines,

$$
\boxed{
\text{coherent noise-power bias}=\frac{\sigma^2}{4},
\qquad
\text{incoherent noise-power bias}=\sigma^2.
}
$$

The coherent visibility average lowers the noise amplitude by $2$ and the noise power by $4$. The incoherent power average does not lower the mean noise-power bias; it only averages fluctuations in the power estimate.

### 18.7 Bias versus uncertainty

It is useful to separate two different concepts:

- The additive noise bias in the mean power.
- The statistical scatter of the estimated power around its mean.

For one complex Gaussian noise realization $n\sim\mathcal{CN}(0,\sigma^2)$,

$$
\left\langle |s+n|^2\right\rangle=|s|^2+\sigma^2,
$$

and

$$
\mathrm{Var}\left[|s+n|^2\right]
=
\sigma^4+2|s|^2\sigma^2.
$$

For the coherent average of $N_{b,g}$ equal-noise baselines, replace

$$
\sigma^2
\rightarrow
\frac{\sigma^2}{N_{b,g}}.
$$

Therefore

$$
\mathrm{Var}\left[\widehat{P}^{\mathrm{coh}}_g\right]
\propto
\frac{\sigma^4}{N^2_{b,g}}
+
\frac{2|s|^2\sigma^2}{N_{b,g}}.
$$

For the incoherent average of $N_{b,g}$ independent powers,

$$
\mathrm{Var}\left[\widehat{P}^{\mathrm{inc}}_g\right]
\propto
\frac{1}{N_{b,g}}
\left(
\sigma^4+2|s|^2\sigma^2
\right).
$$

Thus:

- In the noise-dominated regime, coherent averaging is much better because the noise-power scatter scales like $1/N^2_{b,g}$, while incoherent power averaging scales like $1/N_{b,g}$.
- In the signal-dominated regime, the leading signal-noise cross scatter scales like $1/N_{b,g}$ for both.
- The mean additive noise bias is still different: coherent gives $P_N/N_{b,g}$, while incoherent gives $P_N$ for equal-noise baselines.

### 18.8 Relation to section 9

Section 9 described coherent signal loss when the redundant signals are not exactly aligned:

$$
s_g(\nu,t)
\rightarrow
s_g(\nu,t)e^{i\epsilon_{ij}(\nu,t)}.
$$

In that case, coherent averaging multiplies the signal amplitude by

$$
\Gamma_g(\nu,t)
=
\left|
\sum_{ij\in\mathcal{B}_g}
W_{ij,g}(\nu,t)e^{i\epsilon_{ij}(\nu,t)}
\right|.
$$

This section assumed perfect redundancy, so

$$
\Gamma_g(\nu,t)=1.
$$

If imperfect redundancy is restored, the coherent expectation becomes schematically

$$
\left\langle
\widehat{P}^{\mathrm{coh}}_g
\right\rangle
\approx
\Gamma^2_g P_{S,g}
+
P_{N,g}^{\mathrm{redavg}}.
$$

The incoherent estimator is less sensitive to phase cancellation because each baseline is squared before averaging, but its additive noise-power bias remains closer to the single-baseline noise power unless it is explicitly estimated and subtracted.

The compact takeaway is

$$
\boxed{
\text{coherent average: average }d\text{ first}\Rightarrow
P_S+P_N/N_{b,g}
}
$$

for perfect equal-noise redundancy, while

$$
\boxed{
\text{incoherent average: average }|d|^2\text{ first}\Rightarrow
P_S+P_N.
}
$$

Coherent averaging therefore buys thermal-noise suppression before power formation, but it requires the redundant signals to be phase-aligned. Incoherent averaging is more forgiving of phase scatter, but it retains the mean noise-power bias unless that bias is removed separately.




## Appendix — Convolution vs. Beats: a toy model

**Question:** if we take two periodic functions — e.g. two cosines of *different* wavelengths — and **convolve** them, what comes out? Is that how beats form?

**Short answer: no.** *Beats* come from **adding** (superposing) two tones, and are equivalently revealed by **multiplying** them. **Convolution** is a different beast: by the convolution theorem it **multiplies the two spectra**, so for two pure tones at *different* wavenumbers it gives (ideally) **zero**. Convolution is a frequency-domain *selector / filter*, not a beat generator.

### Setup
Work in position $x$ with wavenumber $k = 2\pi/\lambda$:

$$
y_1(x)=\cos(k_1 x),\qquad y_2(x)=\cos(k_2 x),\qquad k_i=\frac{2\pi}{\lambda_i}.
$$

Let $\Delta k = k_1-k_2$ (difference) and $\Sigma k = k_1+k_2$ (sum).

### 1. Addition → **beats**
$$
y_1+y_2 \;=\; 2\,\underbrace{\cos\!\Big(\tfrac{\Delta k}{2}\,x\Big)}_{\text{slow envelope}}\;\underbrace{\cos\!\Big(\tfrac{\Sigma k}{2}\,x\Big)}_{\text{fast carrier}}.
$$

A fast carrier at the mean wavenumber $\bar k=\Sigma k/2$ sits inside a slow envelope at $\Delta k/2$. The *amplitude* (loudness) maxima repeat with the **beat wavelength**

$$
\lambda_{\text{beat}}=\frac{2\pi}{|\Delta k|}=\frac{\lambda_1\lambda_2}{|\lambda_1-\lambda_2|}.
$$

**This is how beats form** — pure superposition, no nonlinearity required.

### 2. Multiplication → amplitude modulation

**Goal:** simplify the product $y_1 y_2 = \cos(k_1 x)\cos(k_2 x)$ into a sum of pure tones.

**Step 1 — write the two angle-addition identities.** For any angles $A,B$,

$$
\cos(A-B)=\cos A\cos B+\sin A\sin B,
$$
$$
\cos(A+B)=\cos A\cos B-\sin A\sin B.
$$

**Step 2 — add them.** The $\sin A\sin B$ terms have opposite signs and cancel, leaving only the product we want:

$$
\cos(A-B)+\cos(A+B)=2\cos A\cos B.
$$

**Step 3 — solve for the product** (this is the *product-to-sum* identity):

$$
\cos A\cos B=\tfrac{1}{2}\big[\cos(A-B)+\cos(A+B)\big].
$$

**Step 4 — substitute the physical phases** $A=k_1 x$ and $B=k_2 x$, with $\Delta k=k_1-k_2$ and $\Sigma k=k_1+k_2$:

$$
\boxed{\;y_1\,y_2=\cos(k_1 x)\cos(k_2 x)=\tfrac{1}{2}\big[\cos(\Delta k\,x)+\cos(\Sigma k\,x)\big]\;}
$$

**Interpretation.** Multiplying the two cosines puts *no* power back at the original $k_1,k_2$; instead it creates exactly two new tones — a slow *difference* tone at $\Delta k$ and a fast *sum* tone at $\Sigma k$. That is amplitude modulation: the fast $\cos(\Sigma k\,x)$ carrier with a slowly varying amplitude set by $\cos(\Delta k\,x)$.

This ties back to the beats of §1. There the *displacement* envelope is $\cos(\tfrac{\Delta k}{2}x)$, but the *intensity / power* $\propto\cos^2(\tfrac{\Delta k}{2}x)=\tfrac{1}{2}\big[1+\cos(\Delta k\,x)\big]$ recurs at the full $\Delta k$ — the same difference tone produced here. So a square-law detector applied to the sum $y_1+y_2$ (i.e. forming $(y_1+y_2)^2$) is exactly what turns beats into a measurable/audible signal at $\Delta k$.

### 3. Convolution → spectral multiplication ($\approx 0$, not beats)
By the **convolution theorem**,

$$
(y_1 * y_2)(x)\;=\;\int_{-\infty}^{\infty} y_1(u)\,y_2(x-u)\,du
\;\;\Longleftrightarrow\;\;
\tilde y_1(k)\,\tilde y_2(k).
$$

Each cosine has a *line spectrum*, $\tilde y_i(k)=\pi\big[\delta(k-k_i)+\delta(k+k_i)\big]$. The product of two line spectra vanishes unless the lines coincide, so

$$
(y_1 * y_2)(x)=
\begin{cases}
0, & k_1\neq k_2 \quad(\text{orthogonality of distinct sinusoids}),\\[4pt]
\text{resonant build-up} \;\propto\; (\text{record length}), & k_1=k_2.
\end{cases}
$$

So convolving two *different*-wavelength cosines yields essentially nothing (any residual seen numerically is finite-record spectral leakage), while convolving *equal*-wavelength cosines accumulates coherently. **Convolution selects matching frequencies; it does not create beats.** (This is exactly the matched-filter / orthogonality fact underlying Fourier and delay-transform power spectra.)

| operation | result | makes beats? |
|---|---|---|
| $y_1+y_2$ &nbsp;(add) | $2\cos(\tfrac{\Delta k}{2}x)\,\cos(\tfrac{\Sigma k}{2}x)$ | **yes** |
| $y_1\cdot y_2$ &nbsp;(multiply) | $\tfrac{1}{2}[\cos(\Delta k\,x)+\cos(\Sigma k\,x)]$ | AM (beat-related) |
| $y_1 * y_2$ &nbsp;(convolve) | $\tilde y_1(k)\,\tilde y_2(k)\approx 0$ for $k_1\neq k_2$ | **no** (it is a filter) |

The cell below implements and plots all three.


In [ ]:
# Convolution vs. beats: add / multiply / convolve two cosines of different wavelength
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Two cosines with DIFFERENT wavelengths   y_i(x) = cos(k_i x),  k = 2*pi/lambda
# ----------------------------------------------------------------------
lam1, lam2 = 10.0, 12.0                 # the two (different) wavelengths
k1, k2 = 2*np.pi/lam1, 2*np.pi/lam2
dk, ksum = k1 - k2, k1 + k2

L, N = 480.0, 9600                      # long record so the resonant build-up is visible
x = np.linspace(0, L, N); dx = x[1] - x[0]
y1, y2 = np.cos(k1*x), np.cos(k2*x)

# --- Three ways to combine the two signals ---
y_add    = y1 + y2                                          # ADDITION    -> beats
y_mul    = y1 * y2                                          # PRODUCT     -> amplitude modulation
y_cnv    = dx*np.convolve(y1, y2,            mode='same')   # CONVOLUTION, different lambda
y_cnv_eq = dx*np.convolve(y1, np.cos(k1*x),  mode='same')   # CONVOLUTION, equal lambda -> resonance

env = 2*np.cos(dk*x/2)                                      # analytic beat envelope of the sum
lam_beat = 2*np.pi/abs(dk)

# --- One-sided spectra (to show WHY convolution != beats) ---
kf = 2*np.pi*np.fft.rfftfreq(N, d=dx)
def amp(sig):
    A = np.abs(np.fft.rfft(sig)); return A/A.max()
A1, A2 = amp(y1), amp(y2)
Aprod = A1*A2; Aprod = Aprod/(Aprod.max() if Aprod.max() > 0 else 1)   # conv spectrum (diff lambda)
Aself = A1*A1; Aself = Aself/Aself.max()                                # conv spectrum (equal lambda)

fig, ax = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle("Two cosines of different wavelength: addition (beats) vs. product vs. convolution",
             fontsize=16, y=0.995)

# (0,0) inputs
m = x <= 60
ax[0,0].plot(x[m], y1[m], label=fr"$y_1=\cos(k_1 x),\ \lambda_1={lam1:g}$")
ax[0,0].plot(x[m], y2[m], label=fr"$y_2=\cos(k_2 x),\ \lambda_2={lam2:g}$")
ax[0,0].set_title("Inputs: two cosines (different $\\lambda$)")
ax[0,0].set_xlabel("x"); ax[0,0].legend(loc="upper right", fontsize=10)

# (0,1) input spectra
ax[0,1].plot(kf, A1, label=r"$|\tilde y_1(k)|$")
ax[0,1].plot(kf, A2, label=r"$|\tilde y_2(k)|$")
ax[0,1].axvline(k1, color='C0', ls=':', alpha=.6); ax[0,1].axvline(k2, color='C1', ls=':', alpha=.6)
ax[0,1].set_xlim(0, 1.6)
ax[0,1].set_title("Their spectra are single lines at $k_1, k_2$")
ax[0,1].set_xlabel("wavenumber k"); ax[0,1].legend(fontsize=10)

# (1,0) addition -> beats
ax[1,0].plot(x, y_add, lw=.8, color='C2')
ax[1,0].plot(x,  env, 'k--', lw=1, alpha=.8, label=r"$\pm 2\cos(\Delta k\, x/2)$ envelope")
ax[1,0].plot(x, -env, 'k--', lw=1, alpha=.8)
ax[1,0].set_title(fr"ADDITION  $y_1+y_2 \to$ BEATS  ($\lambda_{{beat}}=2\pi/|\Delta k|={lam_beat:.0f}$)")
ax[1,0].set_xlabel("x"); ax[1,0].legend(loc="upper right", fontsize=10)

# (1,1) product -> AM
mm = x <= 120
ax[1,1].plot(x[mm], y_mul[mm], color='C3', lw=.9)
ax[1,1].set_title(r"PRODUCT  $y_1 y_2=\frac{1}{2}[\cos(\Delta k\,x)+\cos(\Sigma k\,x)]$  (AM)")
ax[1,1].set_xlabel("x")

# (2,0) convolution: equal-lambda resonance towers over different-lambda
ax[2,0].plot(x, y_cnv_eq, color='C4', lw=.8,
             label=fr"equal $\lambda$:  $\cos(k_1x)\ast\cos(k_1x)$,  max={np.abs(y_cnv_eq).max():.0f}")
ax[2,0].plot(x, y_cnv, color='C5', lw=1.2,
             label=fr"diff $\lambda$:  $\cos(k_1x)\ast\cos(k_2x)$,  max={np.abs(y_cnv).max():.1f}")
ax[2,0].set_title("CONVOLUTION builds up ONLY when wavelengths match (resonance) — not beats")
ax[2,0].set_xlabel("x"); ax[2,0].legend(loc="upper right", fontsize=9)

# (2,1) spectral explanation: convolution multiplies spectra
ax[2,1].plot(kf, Aself, color='C4', label=r"$|\tilde y_1|\,|\tilde y_1|$ (equal $\lambda$) $\to$ peak")
ax[2,1].plot(kf, Aprod, color='C5', lw=2, label=r"$|\tilde y_1|\,|\tilde y_2|$ (diff $\lambda$) $\approx 0$")
ax[2,1].axvline(k1, color='C0', ls=':', alpha=.6); ax[2,1].axvline(k2, color='C1', ls=':', alpha=.6)
ax[2,1].set_xlim(0, 1.6)
ax[2,1].set_title(r"Convolution theorem:  $y_1\ast y_2 \;\leftrightarrow\; \tilde y_1(k)\,\tilde y_2(k)$")
ax[2,1].set_xlabel("wavenumber k"); ax[2,1].legend(fontsize=9)

for a in ax.ravel():
    a.grid(alpha=.25)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

# --- numeric summary ---
print(f"beat wavelength  lambda_beat = 2*pi/|k1-k2| = {lam_beat:.1f}")
print(f"max|y1 + y2|              = {np.abs(y_add).max():.2f}   (beats: envelope up to 2)")
print(f"max|y1 * y2|              = {np.abs(y_mul).max():.2f}")
print(f"max|conv|, diff lambda    = {np.abs(y_cnv).max():.2f}   (bounded; residual = finite-record leakage)")
print(f"max|conv|, equal lambda   = {np.abs(y_cnv_eq).max():.2f}   (grows ~ record length => resonance)")
print(f"resonant / non-resonant ratio = {np.abs(y_cnv_eq).max()/np.abs(y_cnv).max():.1f}")
